# Comprehensive GeoHEIF Implementation for HEIF/GeoHEIF Encoder Notebook with C++23 (xeus-cpp)

Based on the OGC Testbed 20 GEOHEIF Specification, I'll provide a complete implementation to add GeoHEIF support to your encoder notebook. This will enable creating geospatially-referenced HEIF files.

This notebook demonstrates encoding HEIF files from GeoTIFF sources using libheif with various compression and tiling options.

## Features Demonstrated:
- Non-tiled images (uncompressed and compressed)
- Tiled images using grid/unci/tili schemes
- Pyramid/multi-resolution images
- Various compression methods (HEVC, AV1, uncompressed, DEFLATE, Brotli, etc.)
- GeoTIFF input handling with GDAL
- Detailed HEIF file structure analysis

## Environment:
- Runs in isolated mamba/conda environment
- All libraries installed in $CONDA_PREFIX
- System-independent library loading using dlopen
- Cross-platform path handling

## Cell Re-execution Safe:
All definitions are protected with include guards to prevent redefinition errors.

# SETUP

## Setup: Environment Detection and Global Configuration

In [ ]:
#ifndef HEIF_NOTEBOOK_GLOBALS_H
#define HEIF_NOTEBOOK_GLOBALS_H

#include <iostream>
#include <filesystem>
#include <cstdlib>
#include <string>
#include <vector>

namespace fs = std::filesystem;

// Detect operating system
#if defined(_WIN32) || defined(_WIN64)
    #define HEIF_OS_WINDOWS 1
    #define HEIF_LIB_PREFIX ""
    #define HEIF_LIB_EXT ".dll"
#elif defined(__APPLE__) || defined(__MACH__)
    #define HEIF_OS_MACOS 1
    #define HEIF_LIB_PREFIX "lib"
    #define HEIF_LIB_EXT ".dylib"
#elif defined(__linux__)
    #define HEIF_OS_LINUX 1
    #define HEIF_LIB_PREFIX "lib"
    #define HEIF_LIB_EXT ".so"
#else
    #define HEIF_OS_UNKNOWN 1
    #define HEIF_LIB_PREFIX "lib"
    #define HEIF_LIB_EXT ".so"
#endif

// Global namespace for persistent environment configuration
namespace heif_notebook {

// Environment configuration structure
struct EnvironmentConfig {
    fs::path conda_prefix;
    fs::path include_path;
    fs::path lib_path;
    std::string os_name;
    std::string lib_extension;
    std::string lib_prefix;
    bool is_conda_env;
    
    EnvironmentConfig() : is_conda_env(false) {
        // Detect OS
#ifdef HEIF_OS_WINDOWS
        os_name = "Windows";
#elif defined(HEIF_OS_MACOS)
        os_name = "macOS";
#elif defined(HEIF_OS_LINUX)
        os_name = "Linux";
#else
        os_name = "Unknown";
#endif
        
        lib_extension = HEIF_LIB_EXT;
        lib_prefix = HEIF_LIB_PREFIX;
        
        // Get CONDA_PREFIX
        const char* conda_prefix_env = std::getenv("CONDA_PREFIX");
        if (conda_prefix_env) {
            conda_prefix = fs::path(conda_prefix_env);
            is_conda_env = true;
            include_path = conda_prefix / "include";
            lib_path = conda_prefix / "lib";
        } else {
            // Fallback to system paths
            is_conda_env = false;
#ifdef HEIF_OS_WINDOWS
            include_path = "C:/Program Files/include";
            lib_path = "C:/Program Files/lib";
#else
            include_path = "/usr/local/include";
            lib_path = "/usr/local/lib";
#endif
        }
    }
    
    void print_info() const {
        std::cout << "\n=== Environment Information ===" << std::endl;
        std::cout << "Operating System: " << os_name << std::endl;
        std::cout << "Library Prefix: " << lib_prefix << std::endl;
        std::cout << "Library Extension: " << lib_extension << std::endl;
        std::cout << "Conda Environment: " << (is_conda_env ? "Yes" : "No") << std::endl;
        
        if (is_conda_env) {
            std::cout << "CONDA_PREFIX: " << conda_prefix << std::endl;
            std::cout << "Include Path: " << include_path << std::endl;
            std::cout << "Library Path: " << lib_path << std::endl;
        }
        std::cout << "\n✓ Environment detected" << std::endl;
    }
};

// Global configuration instance - singleton pattern
inline EnvironmentConfig& get_env_config() {
    static EnvironmentConfig instance;
    return instance;
}

} // namespace heif_notebook

// Initialize and print environment info
heif_notebook::get_env_config().print_info();

#endif // HEIF_NOTEBOOK_GLOBALS_H

## Setup: Dynamic Library Loading with dlopen

In [ ]:
#ifndef HEIF_NOTEBOOK_DLOPEN_H
#define HEIF_NOTEBOOK_DLOPEN_H

#include <iostream>
#include <filesystem>
#include <string>
#include <vector>
#include <map>

// Platform-specific includes for dlopen
#ifdef HEIF_OS_WINDOWS
    #include <windows.h>
    #define RTLD_NOW 0
    #define RTLD_GLOBAL 0
    typedef HMODULE dl_handle_t;
    
    void* dlopen(const char* filename, int flags) {
        return (void*)LoadLibraryA(filename);
    }
    
    const char* dlerror() {
        static char error_msg[256];
        FormatMessageA(FORMAT_MESSAGE_FROM_SYSTEM, NULL, GetLastError(),
                      MAKELANGID(LANG_NEUTRAL, SUBLANG_DEFAULT),
                      error_msg, sizeof(error_msg), NULL);
        return error_msg;
    }
    
    int dlclose(void* handle) {
        return FreeLibrary((HMODULE)handle) ? 0 : -1;
    }
#else
    #include <dlfcn.h>
    typedef void* dl_handle_t;
#endif

namespace heif_notebook {

// Library handle manager
class LibraryLoader {
private:
    std::map<std::string, dl_handle_t> loaded_libraries;
    
public:
    // Construct full library path and find existing library
    std::string get_library_path(const std::string& lib_name) {
        namespace fs = std::filesystem;
        
        // Get environment config
        auto& env = get_env_config();
        
        // Try with different version suffix patterns
        std::vector<std::string> try_patterns = {
            env.lib_prefix + lib_name + env.lib_extension,
        };
        
        // Add versioned patterns for Unix-like systems
#ifndef HEIF_OS_WINDOWS
        // Try .dylib.X or .so.X patterns
        for (int ver = 1; ver <= 10; ver++) {
            try_patterns.push_back(env.lib_prefix + lib_name + env.lib_extension + "." + std::to_string(ver));
        }
        // Try .X.Y.Z patterns
        for (int major = 1; major <= 5; major++) {
            for (int minor = 0; minor <= 9; minor++) {
                try_patterns.push_back(env.lib_prefix + lib_name + env.lib_extension + "." + 
                                      std::to_string(major) + "." + std::to_string(minor));
            }
        }
        // Try .X.dylib patterns (alternate macOS versioning)
        for (int ver = 1; ver <= 10; ver++) {
            try_patterns.push_back(env.lib_prefix + lib_name + "." + std::to_string(ver) + env.lib_extension);
        }
#endif
        
        // Search for existing library file
        for (const auto& pattern : try_patterns) {
            fs::path lib_path = env.lib_path / pattern;
            if (fs::exists(lib_path)) {
                return lib_path.string();
            }
        }
        
        // Return default path (for error reporting)
        fs::path default_path = env.lib_path / (env.lib_prefix + lib_name + env.lib_extension);
        return default_path.string();
    }
    
    // Check if library exists
    bool library_exists(const std::string& lib_name) {
        namespace fs = std::filesystem;
        std::string lib_path = get_library_path(lib_name);
        return fs::exists(lib_path);
    }
    
    // Load a library
    bool load_library(const std::string& lib_name, bool required = true) {
        // Check if already loaded
        if (loaded_libraries.find(lib_name) != loaded_libraries.end()) {
            std::cout << "  ℹ " << lib_name << " already loaded" << std::endl;
            return true;
        }
        
        std::cout << "  Loading: " << lib_name << std::endl;
        
        std::string lib_path = get_library_path(lib_name);
        std::cout << "    Path: " << lib_path << std::endl;
        
        namespace fs = std::filesystem;
        if (!fs::exists(lib_path)) {
            if (required) {
                std::cout << "    ✗ Not found (Required)" << std::endl;
            } else {
                std::cout << "    ⚠ Not found (Optional)" << std::endl;
            }
            return false;
        }
        
        dl_handle_t handle = (dl_handle_t)dlopen(lib_path.c_str(), RTLD_NOW | RTLD_GLOBAL);
        
        if (!handle) {
            const char* error = dlerror();
            std::cout << "    ✗ Failed to load: " << (error ? error : "Unknown error") << std::endl;
            return false;
        }
        
        loaded_libraries[lib_name] = handle;
        std::cout << "    ✓ Loaded successfully" << std::endl;
        return true;
    }
    
    // Load multiple libraries in order
    void load_libraries(const std::vector<std::pair<std::string, bool>>& libraries) {
        std::cout << "\n=== Loading Libraries ===" << std::endl;
        for (const auto& [lib_name, required] : libraries) {
            load_library(lib_name, required);
        }
        std::cout << "\n✓ Library loading complete" << std::endl;
    }
    
    // Get count of loaded libraries
    size_t loaded_count() const {
        return loaded_libraries.size();
    }
    
    // Unload all libraries
    ~LibraryLoader() {
        for (auto& [name, handle] : loaded_libraries) {
            if (handle) {
                dlclose(handle);
            }
        }
    }
};

// Global library loader instance - singleton pattern
inline LibraryLoader& get_library_loader() {
    static LibraryLoader instance;
    return instance;
}

} // namespace heif_notebook

std::cout << "\n✓ Library loader initialized" << std::endl;

#endif // HEIF_NOTEBOOK_DLOPEN_H

## Load All Required Libraries

In [ ]:
#ifndef HEIF_NOTEBOOK_LOAD_ALL_H
#define HEIF_NOTEBOOK_LOAD_ALL_H

// Define library loading order (dependencies first)
std::vector<std::pair<std::string, bool>> libraries_to_load = {
    // Core compression libraries (no dependencies)
    {"z", true},              // zlib - required by many libraries
    {"bz2", false},           // bzip2 - optional
    {"lzma", false},          // LZMA - optional
    {"zstd", false},          // Zstandard - optional
    {"brotlicommon", false},  // Brotli common - optional
    {"brotlidec", false},     // Brotli decoder - optional
    {"brotlienc", false},     // Brotli encoder - optional
    
    // Image format libraries
    {"jpeg", false},          // JPEG - used by TIFF/GDAL
    {"png", false},           // PNG - used by GDAL
    {"png16", false},         // PNG 1.6 - alternative name
    {"webp", false},          // WebP - optional
    
    // TIFF library (depends on zlib, jpeg, etc.)
    {"tiff", true},           // libtiff - required
    
    // NOTE: We skip loading libgeotiff separately since GDAL includes GeoTIFF support
    // Loading both causes header conflicts
    
    // Geospatial libraries
    {"proj", false},          // PROJ - used by GDAL
    {"sqlite3", false},       // SQLite - used by GDAL
    {"curl", false},          // cURL - used by GDAL
    {"expat", false},         // Expat XML parser - used by GDAL
    {"xml2", false},          // libxml2 - used by GDAL
    
    // GDAL (includes GeoTIFF support, depends on many of the above)
    {"gdal", true},           // GDAL - required
    
    // Video codec libraries (for HEIF)
    {"x265", false},          // x265 HEVC encoder - optional
    {"de265", false},         // de265 HEVC decoder - optional
    {"aom", false},           // AOM AV1 codec - optional
    {"dav1d", false},         // dav1d AV1 decoder - optional
    {"rav1e", false},         // rav1e AV1 encoder - optional
    {"SvtAv1Enc", false},     // SVT-AV1 encoder - optional
    {"SvtAv1Dec", false},     // SVT-AV1 decoder - optional
    
    // HEIF library (depends on codecs)
    {"heif", true},           // libheif - required
};

// Load all libraries
heif_notebook::get_library_loader().load_libraries(libraries_to_load);

// Report summary
std::cout << "\nSuccessfully loaded " << heif_notebook::get_library_loader().loaded_count() 
          << " libraries" << std::endl;

#endif // HEIF_NOTEBOOK_LOAD_ALL_H

## Standard C++23 Libraries

In [ ]:
#ifndef HEIF_NOTEBOOK_STD_HEADERS_H
#define HEIF_NOTEBOOK_STD_HEADERS_H

// C++23 Standard Library
#include <iostream>
#include <fstream>
#include <sstream>
#include <iomanip>
#include <format>      // C++20/23 formatting
#include <print>       // C++23 print functions
#include <expected>    // C++23 error handling
#include <optional>
#include <variant>
#include <memory>
#include <vector>
#include <array>
#include <string>
#include <string_view>
#include <span>        // C++20 span
#include <map>
#include <ranges>      // C++20/23 ranges
#include <algorithm>
#include <functional>
#include <concepts>    // C++20 concepts
#include <cstring>
#include <cstdint>
#include <stdexcept>
#include <source_location> // C++20 source location
#include <filesystem>  // C++17 filesystem

// Check C++ version
static_assert(__cplusplus >= 202002L, "C++20 or later required");

std::cout << "\nC++ Version: " << __cplusplus << std::endl;
std::cout << "✓ Standard C++ libraries loaded" << std::endl;

#endif // HEIF_NOTEBOOK_STD_HEADERS_H

## Load Library Headers - Single Combined Cell



**IMPORTANT**: All library headers must be loaded in a single cell to avoid redefinition conflicts.

**NOTE**: We do NOT load libgeotiff headers separately. GDAL includes GeoTIFF support internally,
and loading both libgeotiff and GDAL headers causes `CE_None` enum redefinition conflicts.

In [ ]:
#ifndef HEIF_NOTEBOOK_ALL_HEADERS_H
#define HEIF_NOTEBOOK_ALL_HEADERS_H

// ============================================================================
// Load all library headers in correct order (dependencies first)
// NOTE: Do NOT include libgeotiff headers - GDAL includes GeoTIFF support
// ============================================================================

// 1. zlib (no dependencies)
#ifndef HEIF_ZLIB_LOADED
#define HEIF_ZLIB_LOADED
#include <zlib.h>
#endif

// 2. libtiff (depends on zlib)
#ifndef HEIF_TIFF_LOADED
#define HEIF_TIFF_LOADED
#include <tiffio.h>
#include <tiffio.hxx>
#endif

// 3. GDAL (depends on TIFF, includes GeoTIFF support internally)
#ifndef HEIF_GDAL_LOADED
#define HEIF_GDAL_LOADED
#ifdef __has_include
#if __has_include(<gdal.h>)
    #include <gdal.h>
    #include <gdal_priv.h>
    #include <cpl_conv.h>
    #include <cpl_string.h>
    #include <ogr_spatialref.h>
    #define HEIF_GDAL_INCLUDE_PREFIX ""
#elif __has_include(<gdal/gdal.h>)
    #include <gdal/gdal.h>
    #include <gdal/gdal_priv.h>
    #include <gdal/cpl_conv.h>
    #include <gdal/cpl_string.h>
    #include <gdal/ogr_spatialref.h>
    #define HEIF_GDAL_INCLUDE_PREFIX "gdal/"
#else
    #error "GDAL headers not found. Check installation."
#endif
#else
    #include <gdal.h>
    #include <gdal_priv.h>
    #include <cpl_conv.h>
    #include <cpl_string.h>
    #include <ogr_spatialref.h>
    #define HEIF_GDAL_INCLUDE_PREFIX ""
#endif
#endif // HEIF_GDAL_LOADED

// 4. libheif (may depend on codecs)
#ifndef HEIF_LIBHEIF_LOADED
#define HEIF_LIBHEIF_LOADED

// Enable experimental features
#ifndef HEIF_ENABLE_EXPERIMENTAL_FEATURES
#define HEIF_ENABLE_EXPERIMENTAL_FEATURES 1
#endif

#ifndef WITH_UNCOMPRESSED_CODEC
#define WITH_UNCOMPRESSED_CODEC 1
#endif

#ifdef __has_include
#if __has_include(<libheif/heif.h>)
    #include <libheif/heif.h>
    #include <libheif/heif_properties.h>
    #include <libheif/heif_items.h>
    #include <libheif/heif_experimental.h>
    #include <libheif/heif_uncompressed.h>
    #define HEIF_INCLUDE_PREFIX "libheif/"
#elif __has_include(<heif.h>)
    #include <heif.h>
    #include <heif_properties.h>
    #include <heif_items.h>
    #include <heif_experimental.h>
    #include <heif_uncompressed.h>
    #define HEIF_INCLUDE_PREFIX ""
#else
    #error "libheif headers not found. Check installation."
#endif
#else
    #include <libheif/heif.h>
    #include <libheif/heif_properties.h>
    #include <libheif/heif_items.h>
    #include <libheif/heif_experimental.h>
    #include <libheif/heif_uncompressed.h>
    #define HEIF_INCLUDE_PREFIX "libheif/"
#endif
#endif // HEIF_LIBHEIF_LOADED

// 5. Optional compression libraries
#ifndef HEIF_OPTIONAL_LIBS_LOADED
#define HEIF_OPTIONAL_LIBS_LOADED

#ifdef __has_include
#if __has_include(<brotli/encode.h>)
    #ifndef HEIF_HAVE_BROTLI
    #define HEIF_HAVE_BROTLI 1
    #include <brotli/encode.h>
    #include <brotli/decode.h>
    #endif
#endif

#if __has_include(<zstd.h>)
    #ifndef HEIF_HAVE_ZSTD
    #define HEIF_HAVE_ZSTD 1
    #include <zstd.h>
    #endif
#endif
#endif

#endif // HEIF_OPTIONAL_LIBS_LOADED

// ============================================================================
// Check and report loaded headers
// ============================================================================

std::cout << "\n=== Library Headers Loaded ===" << std::endl;
std::cout << "✓ zlib" << std::endl;
std::cout << "✓ libtiff" << std::endl;
std::cout << "✓ GDAL (include prefix: " << HEIF_GDAL_INCLUDE_PREFIX << ")" << std::endl;
std::cout << "  (GeoTIFF support included via GDAL)" << std::endl;
std::cout << "✓ libheif (include prefix: " << HEIF_INCLUDE_PREFIX << ")" << std::endl;

#ifdef HEIF_HAVE_BROTLI
std::cout << "✓ Brotli" << std::endl;
#else
std::cout << "⚠ Brotli (not available)" << std::endl;
#endif

#ifdef HEIF_HAVE_ZSTD
std::cout << "✓ ZSTD" << std::endl;
#else
std::cout << "⚠ ZSTD (not available)" << std::endl;
#endif

std::cout << "\n✓ All library headers loaded successfully" << std::endl;

#endif // HEIF_NOTEBOOK_ALL_HEADERS_H

### GDAL Headers

In [ ]:
#ifndef HEIF_NOTEBOOK_GDAL_H
#define HEIF_NOTEBOOK_GDAL_H

// Try different include paths for GDAL
#ifdef __has_include
#if __has_include(<gdal.h>)
    // Conda/system installation without prefix
    #include <gdal.h>
    #include <gdal_priv.h>
    #include <cpl_conv.h>
    #include <cpl_string.h>
    #include <ogr_spatialref.h>
    #define GDAL_INCLUDE_PREFIX ""
#elif __has_include(<gdal/gdal.h>)
    // System installation with prefix
    #include <gdal/gdal.h>
    #include <gdal/gdal_priv.h>
    #include <gdal/cpl_conv.h>
    #include <gdal/cpl_string.h>
    #include <gdal/ogr_spatialref.h>
    #define GDAL_INCLUDE_PREFIX "gdal/"
#else
    #error "GDAL headers not found. Check installation."
#endif
#else
    // Fallback: try without prefix
    #include <gdal.h>
    #include <gdal_priv.h>
    #include <cpl_conv.h>
    #include <cpl_string.h>
    #include <ogr_spatialref.h>
    #define GDAL_INCLUDE_PREFIX ""
#endif

void check_gdal_capabilities() {
    using namespace std;
    
    println("\n=== GDAL Information ===");
    println("Include prefix: {}", GDAL_INCLUDE_PREFIX);
    println("Version: {}", GDALVersionInfo("RELEASE_NAME"));
    println("Version Number: {}", GDALVersionInfo("VERSION_NUM"));
    
    GDALAllRegister();
    
    int driver_count = GDALGetDriverCount();
    println("\nRegistered Drivers: {}", driver_count);
    
    println("\nImportant Raster Format Support:");
    
    constexpr std::array<std::string_view, 11> important_formats = {
        "GTiff", "COG", "JPEG", "PNG", "HFA", "netCDF",
        "JP2OpenJPEG", "WEBP", "ZMAP", "NITF", "HDF5"
    };
    
    for (const auto& format : important_formats) {
        GDALDriverH driver = GDALGetDriverByName(format.data());
        if (driver) {
            const char* long_name = GDALGetMetadataItem(driver, "DMD_LONGNAME", nullptr);
            println("  ✓ {:15s} - {}", format, long_name ? long_name : "");
        }
    }
    
    println("\nGeoTIFF Compression Methods:");
    GDALDriverH gtiff_driver = GDALGetDriverByName("GTiff");
    if (gtiff_driver) {
        const char* compress_list = GDALGetMetadataItem(gtiff_driver, "DMD_CREATIONOPTIONLIST", nullptr);
        if (compress_list) {
            std::string_view compress_str(compress_list);
            
            constexpr std::array<std::string_view, 10> compression_methods = {
                "NONE", "JPEG", "LZW", "PACKBITS", "DEFLATE",
                "LZMA", "ZSTD", "LERC", "WEBP", "JXL"
            };
            
            for (const auto& method : compression_methods) {
                auto search_str = format("<Value>{}</Value>", method);
                if (compress_str.find(search_str) != std::string_view::npos) {
                    println("  ✓ {}", method);
                }
            }
        }
    }
}

check_gdal_capabilities();

std::println("✓ GDAL headers loaded");

#endif // HEIF_NOTEBOOK_GDAL_H

### Verify Library Capabilities

In [ ]:
#ifndef HEIF_NOTEBOOK_VERIFY_H
#define HEIF_NOTEBOOK_VERIFY_H

void verify_all_libraries() {
    using namespace std;
    
    // libheif
    println("\n=== libheif ===");
    println("Version: {}", heif_get_version());
    
    // GDAL
    println("\n=== GDAL ===");
    println("Version: {}", GDALVersionInfo("RELEASE_NAME"));
    GDALAllRegister();
    println("Registered Drivers: {}", GDALGetDriverCount());
    
    // Check for GeoTIFF driver in GDAL
    GDALDriverH gtiff_driver = GDALGetDriverByName("GTiff");
    if (gtiff_driver) {
        println("✓ GeoTIFF support available via GDAL");
    } else {
        println("⚠ GeoTIFF driver not found in GDAL");
    }
    
    // TIFF
    println("\n=== libtiff ===");
    const char* tiff_version = TIFFGetVersion();
    if (tiff_version) {
        string_view version_str(tiff_version);
        auto pos = version_str.find("Version");
        if (pos != string_view::npos) {
            auto start = version_str.find_first_of("0123456789", pos);
            auto end = version_str.find("\n", start);
            if (start != string_view::npos) {
                println("Version: {}", version_str.substr(start, end - start));
            }
        }
    }
    
    // zlib
    println("\n=== zlib ===");
    println("Version: {}", zlibVersion());
    
#ifdef HEIF_HAVE_BROTLI
    println("\n=== Brotli ===");
    uint32_t brotli_ver = BrotliEncoderVersion();
    println("Version: {}.{}.{}", brotli_ver >> 24, (brotli_ver >> 12) & 0xFFF, brotli_ver & 0xFFF);
#endif
    
#ifdef HEIF_HAVE_ZSTD
    println("\n=== ZSTD ===");
    println("Version: {}", ZSTD_versionString());
#endif
    
    println("\n✓ All libraries verified");
}

verify_all_libraries();

#endif // HEIF_NOTEBOOK_VERIFY_H

## Important Notes:

### Why No Separate libgeotiff?

1. **GDAL includes GeoTIFF**: Modern GDAL includes comprehensive GeoTIFF support via the GTiff driver
2. **Header conflicts**: libgeotiff and GDAL both define CPL error enums (CE_None, etc.)
3. **Redundant**: Loading both provides no additional functionality

### GeoTIFF Support via GDAL:

GDAL provides full GeoTIFF support including:
- Reading/writing GeoTIFF tags
- Coordinate system handling
- Projection information
- All standard GeoTIFF metadata

### Header Loading Order:

1. **zlib** - Foundation compression library
2. **libtiff** - TIFF file I/O
3. **GDAL** - Includes GeoTIFF support, depends on TIFF
4. **libheif** - HEIF container format
5. **Optional libs** - Brotli, ZSTD, etc.

## Compression Algorithm Enumerations with C++23 Features

In [ ]:
#ifndef HEIF_COMPRESSION_TYPES_H
#define HEIF_COMPRESSION_TYPES_H

// C++23: Use enum class with underlying type specification
enum class ExtendedCompressionType : uint8_t {
    None = 0,
    HEVC,
    AV1,
    VVC,
    AVC,
    JPEG,
    JPEG2000,
    HTJ2K,
    Uncompressed,
    // Additional GDAL-compatible compressions
    DEFLATE,
    ZLIB,
    Brotli,
    ZSTD,
    LZW,
    WebP,
    LERC,
    PackBits
};

// C++23: Use designated initializers and constexpr
struct CompressionInfo {
    ExtendedCompressionType type;
    std::string_view name;
    std::string_view description;
    bool lossless;
    bool supported_by_libheif;
    bool supported_by_gdal;
};

// C++23: constexpr vector-like container
constexpr inline std::array<CompressionInfo, 16> COMPRESSION_ALGORITHMS = {{
    {ExtendedCompressionType::None, "NONE", "No compression", true, true, true},
    {ExtendedCompressionType::HEVC, "HEVC", "H.265/HEVC video codec", false, true, false},
    {ExtendedCompressionType::AV1, "AV1", "AV1 video codec", false, true, false},
    {ExtendedCompressionType::VVC, "VVC", "H.266/VVC video codec", false, true, false},
    {ExtendedCompressionType::AVC, "AVC", "H.264/AVC video codec", false, true, false},
    {ExtendedCompressionType::JPEG, "JPEG", "JPEG compression", false, true, true},
    {ExtendedCompressionType::JPEG2000, "JPEG2000", "JPEG 2000", false, true, true},
    {ExtendedCompressionType::HTJ2K, "HTJ2K", "High Throughput JPEG 2000", false, true, false},
    {ExtendedCompressionType::DEFLATE, "DEFLATE", "DEFLATE (zlib) compression", true, true, true},
    {ExtendedCompressionType::ZLIB, "ZLIB", "ZLIB compression", true, true, true},
    {ExtendedCompressionType::Brotli, "BROTLI", "Brotli compression", true, true, false},
    {ExtendedCompressionType::ZSTD, "ZSTD", "Zstandard compression", true, false, true},
    {ExtendedCompressionType::LZW, "LZW", "LZW compression", true, false, true},
    {ExtendedCompressionType::WebP, "WEBP", "WebP compression", false, false, true},
    {ExtendedCompressionType::LERC, "LERC", "Limited Error Raster Compression", true, false, true},
    {ExtendedCompressionType::PackBits, "PACKBITS", "PackBits compression", true, false, true}
}};

void list_all_compression_algorithms() {
    using namespace std;
    
    println("\n=== Available Compression Algorithms ===");
    println("{:15s} {:40s} {:10s} {:10s} {:10s}",
            "Name", "Description", "Lossless", "libheif", "GDAL");
    println("{}", string(85, '-'));
    
    // C++23: Use ranges with views
    for (const auto& comp : COMPRESSION_ALGORITHMS) {
        println("{:15s} {:40s} {:10s} {:10s} {:10s}",
                comp.name,
                comp.description,
                comp.lossless ? "Yes" : "No",
                comp.supported_by_libheif ? "Yes" : "No",
                comp.supported_by_gdal ? "Yes" : "No");
    }
}

std::println("Compression types defined.");

#endif // HEIF_COMPRESSION_TYPES_H

## Helper Functions with C++23 Error Handling (Updated)

In [ ]:
#ifndef HEIF_HELPER_FUNCTIONS_H
#define HEIF_HELPER_FUNCTIONS_H

// C++23: Use std::expected for error handling
using HeifResult = std::expected<void, std::string>;

template<typename T>
using HeifExpected = std::expected<T, std::string>;

// Helper function to check errors with source location
HeifResult check_error(heif_error error, 
                       std::string_view context,
                       std::source_location loc = std::source_location::current()) {
    if (error.code != heif_error_Ok) {
        auto err_msg = std::format("Error in {} at {}:{}: {}",
                                   context,
                                   loc.file_name(),
                                   loc.line(),
                                   error.message);
        return std::unexpected(err_msg);
    }
    return {};
}

// Simpler version without std::expected for immediate error reporting
void check_error_simple(heif_error error, std::string_view context) {
    if (error.code != heif_error_Ok) {
        std::cerr << "Error in " << context << ": " << error.message << std::endl;
        throw std::runtime_error(error.message);
    }
}

// Helper to list available encoders with modern formatting
void list_encoders(heif_compression_format format) {
    using namespace std;
    
    const heif_encoder_descriptor* descriptors[10];
    int count = heif_get_encoder_descriptors(format, nullptr, descriptors, 10);
    
    println("Available encoders for format {}:", static_cast<int>(format));
    
    // C++23: Use ranges to transform and print
    for (int i : std::views::iota(0, count)) {
        println("  - {}: {}",
                heif_encoder_descriptor_get_id_name(descriptors[i]),
                heif_encoder_descriptor_get_name(descriptors[i]));
    }
}

// C++23: Use constexpr mapping function
constexpr heif_compression_format to_heif_compression_format(ExtendedCompressionType type) {
    using enum ExtendedCompressionType;
    
    switch(type) {
        case HEVC: return heif_compression_HEVC;
        case AV1: return heif_compression_AV1;
        case VVC: return heif_compression_VVC;
        case AVC: return heif_compression_AVC;
        case JPEG: return heif_compression_JPEG;
        case JPEG2000: return heif_compression_JPEG2000;
        case HTJ2K: return heif_compression_HTJ2K;
        case Uncompressed:
        case DEFLATE:
        case ZLIB:
        case Brotli:
            return heif_compression_uncompressed;
        default:
            return heif_compression_undefined;
    }
}

// C++23: Use constexpr for compile-time evaluation
constexpr heif_unci_compression to_unci_compression(ExtendedCompressionType type) {
    using enum ExtendedCompressionType;
    
    switch(type) {
        case DEFLATE: return heif_unci_compression_deflate;
        case ZLIB: return heif_unci_compression_zlib;
        case Brotli: return heif_unci_compression_brotli;
        default: return heif_unci_compression_off;
    }
}

// Get compression info by type - IMPORTANT: Make this inline to avoid multiple definition errors
inline const CompressionInfo* get_compression_info(ExtendedCompressionType type) {
    for (const auto& info : COMPRESSION_ALGORITHMS) {
        if (info.type == type) {
            return &info;
        }
    }
    return nullptr;
}

// Get compression name as string
inline std::string_view get_compression_name(ExtendedCompressionType type) {
    const CompressionInfo* info = get_compression_info(type);
    return info ? info->name : "Unknown";
}

// Check if compression is lossless
inline bool is_lossless_compression(ExtendedCompressionType type) {
    const CompressionInfo* info = get_compression_info(type);
    return info ? info->lossless : false;
}

std::cout << "Helper functions defined." << std::endl;

#endif // HEIF_HELPER_FUNCTIONS_H

## GDAL-based GeoTIFF Loading

In [ ]:
#ifndef HEIF_GDAL_LOADER_H
#define HEIF_GDAL_LOADER_H

struct GDALImageData {
    int width;
    int height;
    int bands;
    GDALDataType data_type;
    std::vector<std::vector<uint8_t>> band_data;
    std::string projection;
    double geo_transform[6];
    std::map<std::string, std::string> metadata;
};

class GDALImageLoader {
private:
    static bool gdal_initialized;
    
public:
    static void initialize() {
        if (!gdal_initialized) {
            GDALAllRegister();
            gdal_initialized = true;
            std::cout << "GDAL initialized with " << GDALGetDriverCount() << " drivers." << std::endl;
        }
    }
    
    static GDALImageData load(const char* filename) {
        initialize();
        
        GDALDataset* dataset = (GDALDataset*)GDALOpen(filename, GA_ReadOnly);
        if (!dataset) {
            throw std::runtime_error("Cannot open file with GDAL: " + std::string(filename));
        }
        
        GDALImageData data;
        data.width = dataset->GetRasterXSize();
        data.height = dataset->GetRasterYSize();
        data.bands = dataset->GetRasterCount();
        
        std::cout << "Loaded: " << data.width << "x" << data.height 
                  << ", " << data.bands << " bands" << std::endl;
        
        // Get projection and geotransform
        const char* proj = dataset->GetProjectionRef();
        if (proj) {
            data.projection = proj;
        }
        
        dataset->GetGeoTransform(data.geo_transform);
        
        // Get metadata
        char** metadata_list = dataset->GetMetadata();
        if (metadata_list) {
            for (int i = 0; metadata_list[i] != nullptr; i++) {
                std::string item(metadata_list[i]);
                size_t eq_pos = item.find('=');
                if (eq_pos != std::string::npos) {
                    std::string key = item.substr(0, eq_pos);
                    std::string value = item.substr(eq_pos + 1);
                    data.metadata[key] = value;
                }
            }
        }
        
        // Read band data
        for (int band_idx = 1; band_idx <= data.bands; band_idx++) {
            GDALRasterBand* band = dataset->GetRasterBand(band_idx);
            data.data_type = band->GetRasterDataType();
            
            int pixel_size = GDALGetDataTypeSizeBytes(data.data_type);
            std::vector<uint8_t> band_buffer(data.width * data.height * pixel_size);
            
            CPLErr err = band->RasterIO(GF_Read, 0, 0, data.width, data.height,
                                       band_buffer.data(), data.width, data.height,
                                       data.data_type, 0, 0);
            
            if (err != CE_None) {
                GDALClose(dataset);
                throw std::runtime_error("Failed to read band " + std::to_string(band_idx));
            }
            
            data.band_data.push_back(std::move(band_buffer));
        }
        
        GDALClose(dataset);
        return data;
    }
    
    static heif_image* to_heif_image(const GDALImageData& data, int output_bit_depth = 8) {
        heif_image* image = nullptr;
        heif_colorspace colorspace = (data.bands >= 3) ? heif_colorspace_RGB : heif_colorspace_monochrome;
        heif_chroma chroma = (data.bands >= 3) ? heif_chroma_444 : heif_chroma_monochrome;
        
        heif_error error = heif_image_create(data.width, data.height, colorspace, chroma, &image);
        check_error(error, "heif_image_create");
        
        // Add planes and copy data
        if (data.bands >= 3) {
            heif_channel channels[] = {heif_channel_R, heif_channel_G, heif_channel_B};
            for (int i = 0; i < 3; i++) {
                heif_image_add_plane(image, channels[i], data.width, data.height, output_bit_depth);
                
                int stride;
                uint8_t* plane_data = heif_image_get_plane(image, channels[i], &stride);
                
                // Simple copy for 8-bit data
                if (data.data_type == GDT_Byte) {
                    for (int y = 0; y < data.height; y++) {
                        memcpy(plane_data + y * stride,
                               data.band_data[i].data() + y * data.width,
                               data.width);
                    }
                }
            }
            
            // Add alpha if 4 bands
            if (data.bands >= 4) {
                heif_image_add_plane(image, heif_channel_Alpha, data.width, data.height, output_bit_depth);
                int stride;
                uint8_t* alpha_data = heif_image_get_plane(image, heif_channel_Alpha, &stride);
                
                if (data.data_type == GDT_Byte) {
                    for (int y = 0; y < data.height; y++) {
                        memcpy(alpha_data + y * stride,
                               data.band_data[3].data() + y * data.width,
                               data.width);
                    }
                }
            }
        } else {
            heif_image_add_plane(image, heif_channel_Y, data.width, data.height, output_bit_depth);
            
            int stride;
            uint8_t* plane_data = heif_image_get_plane(image, heif_channel_Y, &stride);
            
            if (data.data_type == GDT_Byte) {
                for (int y = 0; y < data.height; y++) {
                    memcpy(plane_data + y * stride,
                           data.band_data[0].data() + y * data.width,
                           data.width);
                }
            }
        }
        
        return image;
    }
};

bool GDALImageLoader::gdal_initialized = false;

std::cout << "GDAL image loader defined." << std::endl;

#endif // HEIF_GDAL_LOADER_H

## HEIF File Structure Analyzer

In [ ]:
#ifndef HEIF_FILE_ANALYZER_H
#define HEIF_FILE_ANALYZER_H

class HEIFFileAnalyzer {
private:
    struct BoxInfo {
        uint64_t offset;
        uint64_t size;
        std::string type;
        std::string description;
        int level;
        std::vector<BoxInfo> children;
    };
    
    static std::string fourcc_to_string(uint32_t fourcc) {
        char str[5];
        str[0] = (fourcc >> 24) & 0xFF;
        str[1] = (fourcc >> 16) & 0xFF;
        str[2] = (fourcc >> 8) & 0xFF;
        str[3] = fourcc & 0xFF;
        str[4] = '\0';
        return std::string(str);
    }
    
    static uint32_t read_uint32(std::ifstream& file) {
        uint32_t value;
        file.read(reinterpret_cast<char*>(&value), 4);
        // Convert from big-endian
        return ((value & 0xFF) << 24) | ((value & 0xFF00) << 8) |
               ((value & 0xFF0000) >> 8) | ((value & 0xFF000000) >> 24);
    }
    
    static uint64_t read_uint64(std::ifstream& file) {
        uint64_t value;
        file.read(reinterpret_cast<char*>(&value), 8);
        // Convert from big-endian
        return ((value & 0xFF) << 56) | ((value & 0xFF00) << 40) |
               ((value & 0xFF0000) << 24) | ((value & 0xFF000000) << 8) |
               ((value & 0xFF00000000) >> 8) | ((value & 0xFF0000000000) >> 24) |
               ((value & 0xFF000000000000) >> 40) | ((value & 0xFF00000000000000) >> 56);
    }
    
    static std::string get_box_description(const std::string& type) {
        static const std::map<std::string, std::string> descriptions = {
            {"ftyp", "File Type Box - Brand identification"},
            {"meta", "Meta Box - Container for metadata"},
            {"hdlr", "Handler Box - Media handler type"},
            {"pitm", "Primary Item Box - Primary item reference"},
            {"iloc", "Item Location Box - Item data locations"},
            {"iinf", "Item Information Box - Item information entries"},
            {"iprp", "Item Properties Box - Item properties container"},
            {"ipco", "Item Property Container Box - Property definitions"},
            {"ipma", "Item Property Association Box - Property associations"},
            {"iref", "Item Reference Box - Item relationships"},
            {"idat", "Item Data Box - Embedded item data"},
            {"mdat", "Media Data Box - Media/image data"},
            {"grpl", "Group List Box - Entity groups"},
            {"dinf", "Data Information Box - Data reference info"},
            {"dref", "Data Reference Box - Data entry references"},
            {"ispe", "Image Spatial Extents Property - Image dimensions"},
            {"colr", "Color Information Box - Color profile"},
            {"pixi", "Pixel Information Property - Bits per channel"},
            {"auxC", "Auxiliary Type Property - Auxiliary image type"},
            {"hvcC", "HEVC Configuration - HEVC decoder config"},
            {"av1C", "AV1 Configuration - AV1 decoder config"},
            {"vvcC", "VVC Configuration - VVC decoder config"},
            {"avcC", "AVC Configuration - AVC decoder config"},
            {"tols", "Tiled Output Layer Set - Tiling info"},
            {"unci", "Uncompressed Image - ISO 23001-17"},
            {"tili", "Tiled Image - Tiling structure"},
            {"grid", "Image Grid - Grid layout"},
            {"iovl", "Image Overlay - Overlay composition"},
            {"Exif", "Exif Metadata - Camera/image metadata"},
            {"mime", "MIME Type Data - MIME content"},
            {"clli", "Content Light Level Info - HDR light levels"},
            {"pasp", "Pixel Aspect Ratio - Aspect ratio info"}
        };
        
        auto it = descriptions.find(type);
        if (it != descriptions.end()) {
            return it->second;
        }
        return "Unknown box type";
    }
    
    static BoxInfo read_box(std::ifstream& file, uint64_t max_offset, int level = 0) {
        BoxInfo box;
        box.level = level;
        box.offset = file.tellg();
        
        if (box.offset >= max_offset) {
            box.size = 0;
            return box;
        }
        
        uint32_t size32 = read_uint32(file);
        uint32_t type = read_uint32(file);
        box.type = fourcc_to_string(type);
        
        if (size32 == 1) {
            box.size = read_uint64(file);
        } else if (size32 == 0) {
            box.size = max_offset - box.offset;
        } else {
            box.size = size32;
        }
        
        box.description = get_box_description(box.type);
        
        // Check if this is a container box
        static const std::vector<std::string> container_boxes = {
            "meta", "iprp", "ipco", "iinf", "dinf", "grpl"
        };
        
        bool is_container = std::find(container_boxes.begin(), 
                                     container_boxes.end(), 
                                     box.type) != container_boxes.end();
        
        if (is_container) {
            uint64_t content_start = file.tellg();
            uint64_t content_end = box.offset + box.size;
            
            // Skip version/flags for full boxes
            if (box.type == "meta") {
                file.seekg(4, std::ios::cur);
                content_start = file.tellg();
            }
            
            while (file.tellg() < content_end && file.good()) {
                BoxInfo child = read_box(file, content_end, level + 1);
                if (child.size == 0) break;
                box.children.push_back(child);
                file.seekg(child.offset + child.size);
            }
        }
        
        return box;
    }
    
    static void print_box(const BoxInfo& box, uint64_t total_size) {
        std::string indent(box.level * 2, ' ');
        double percentage = (box.size * 100.0) / total_size;
        
        std::cout << indent
                  << "[" << box.type << "] "
                  << "Offset: 0x" << std::hex << box.offset << std::dec
                  << ", Size: " << box.size << " bytes"
                  << " (" << std::fixed << std::setprecision(2) << percentage << "%)";
        
        if (!box.description.empty()) {
            std::cout << "\n" << indent << "  ↳ " << box.description;
        }
        std::cout << std::endl;
        
        for (const auto& child : box.children) {
            print_box(child, total_size);
        }
    }
    
public:
    static void analyze(const char* filename) {
        std::cout << "\n" << std::string(80, '=') << std::endl;
        std::cout << "HEIF File Structure Analysis: " << filename << std::endl;
        std::cout << std::string(80, '=') << "\n" << std::endl;
        
        std::ifstream file(filename, std::ios::binary);
        if (!file) {
            std::cerr << "Cannot open file: " << filename << std::endl;
            return;
        }
        
        // Get file size
        file.seekg(0, std::ios::end);
        uint64_t file_size = file.tellg();
        file.seekg(0, std::ios::beg);
        
        std::cout << "File size: " << file_size << " bytes (" 
                  << (file_size / 1024.0) << " KB)\n" << std::endl;
        
        std::vector<BoxInfo> boxes;
        
        while (file.tellg() < file_size && file.good()) {
            BoxInfo box = read_box(file, file_size, 0);
            if (box.size == 0) break;
            boxes.push_back(box);
            file.seekg(box.offset + box.size);
        }
        
        std::cout << "Box Structure:\n" << std::endl;
        for (const auto& box : boxes) {
            print_box(box, file_size);
        }
        
        // Summary statistics
        std::cout << "\n" << std::string(80, '-') << std::endl;
        std::cout << "Summary Statistics\n" << std::endl;
        
        std::map<std::string, uint64_t> box_sizes;
        std::function<void(const BoxInfo&)> collect_sizes = [&](const BoxInfo& box) {
            box_sizes[box.type] += box.size;
            for (const auto& child : box.children) {
                collect_sizes(child);
            }
        };
        
        for (const auto& box : boxes) {
            collect_sizes(box);
        }
        
        std::vector<std::pair<std::string, uint64_t>> sorted_sizes(box_sizes.begin(), box_sizes.end());
        std::sort(sorted_sizes.begin(), sorted_sizes.end(),
                 [](const auto& a, const auto& b) { return a.second > b.second; });
        
        std::cout << std::left << std::setw(20) << "Box Type" 
                  << std::setw(15) << "Total Size"
                  << std::setw(15) << "Percentage" << std::endl;
        std::cout << std::string(50, '-') << std::endl;
        
        for (const auto& [type, size] : sorted_sizes) {
            double percentage = (size * 100.0) / file_size;
            std::cout << std::left << std::setw(20) << type
                      << std::setw(15) << size
                      << std::fixed << std::setprecision(2) << percentage << "%"
                      << std::endl;
        }
        
        std::cout << "\n" << std::string(80, '=') << "\n" << std::endl;
    }
};

std::cout << "HEIF file analyzer defined." << std::endl;

#endif // HEIF_FILE_ANALYZER_H

## Encoding Functions: Non-Tiled Images

In [ ]:
#ifndef HEIF_ENCODE_NONTILED_H
#define HEIF_ENCODE_NONTILED_H

void encode_nontiled(const char* input_file, const char* output_file,
                    ExtendedCompressionType compression,
                    int quality = 50,
                    int output_bit_depth = 8) {
    
    const CompressionInfo* comp_info = nullptr;
    for (const auto& info : COMPRESSION_ALGORITHMS) {
        if (info.type == compression) {
            comp_info = &info;
            break;
        }
    }
    
    std::cout << "\n=== Encoding Non-Tiled Image ===" << std::endl;
    std::cout << "Compression: " << (comp_info ? comp_info->name : "UNKNOWN") << std::endl;
    if (!comp_info->lossless) {
        std::cout << "Quality: " << quality << std::endl;
    }
    
    // Load input with GDAL
    GDALImageData gdal_data = GDALImageLoader::load(input_file);
    heif_image* image = GDALImageLoader::to_heif_image(gdal_data, output_bit_depth);
    
    // Create context
    heif_context* ctx = heif_context_alloc();
    
    // Get encoder
    heif_compression_format heif_format = to_heif_compression_format(compression);
    heif_encoder* encoder = nullptr;
    heif_error error = heif_context_get_encoder_for_format(ctx, heif_format, &encoder);
    check_error(error, "get encoder");
    
    // Set compression parameters
    if (!comp_info->lossless && heif_format != heif_compression_uncompressed) {
        heif_encoder_set_lossy_quality(encoder, quality);
    } else if (comp_info->lossless) {
        heif_encoder_set_lossless(encoder, true);
    }
    
    // Set encoding options
    heif_encoding_options* options = heif_encoding_options_alloc();
    
    // Handle UNCI compression methods
    if (heif_format == heif_compression_uncompressed) {
        options->unci_compression = to_unci_compression(compression);
    }
    
    // Encode
    heif_image_handle* handle = nullptr;
    error = heif_context_encode_image(ctx, image, encoder, options, &handle);
    check_error(error, "encode image");
    
    // Set as primary
    heif_context_set_primary_image(ctx, handle);
    
    // Write to file
    error = heif_context_write_to_file(ctx, output_file);
    check_error(error, "write to file");
    
    std::cout << "Successfully encoded: " << output_file << std::endl;
    
    // Analyze the output file
    HEIFFileAnalyzer::analyze(output_file);
    
    // Cleanup
    heif_image_handle_release(handle);
    heif_encoding_options_free(options);
    heif_encoder_release(encoder);
    heif_context_free(ctx);
    heif_image_release(image);
}

std::cout << "Non-tiled encoding function defined." << std::endl;

#endif // HEIF_ENCODE_NONTILED_H

# Work Point 0 - low-level i/o

## Lower-Level ISOBMFF Implementation for GeoHEIF

### ISOBMFF Box Writing Function

In [ ]:
#ifndef HEIF_ISOBMFF_BOX_WRITER_H
#define HEIF_ISOBMFF_BOX_WRITER_H

#include <vector>
#include <cstdint>
#include <cstring>
#include <fstream>
#include <memory>
#include <stdexcept>

namespace isobmff {

// ============================================================================
// Low-Level ISOBMFF Box Writing Utilities
// ============================================================================

class ByteWriter {
private:
    std::vector<uint8_t> buffer;
    
public:
    ByteWriter() { buffer.reserve(1024); }
    
    // Write unsigned integers (big-endian)
    void write_uint8(uint8_t value) {
        buffer.push_back(value);
    }
    
    void write_uint16(uint16_t value) {
        buffer.push_back((value >> 8) & 0xFF);
        buffer.push_back(value & 0xFF);
    }
    
    void write_uint32(uint32_t value) {
        buffer.push_back((value >> 24) & 0xFF);
        buffer.push_back((value >> 16) & 0xFF);
        buffer.push_back((value >> 8) & 0xFF);
        buffer.push_back(value & 0xFF);
    }
    
    void write_uint64(uint64_t value) {
        buffer.push_back((value >> 56) & 0xFF);
        buffer.push_back((value >> 48) & 0xFF);
        buffer.push_back((value >> 40) & 0xFF);
        buffer.push_back((value >> 32) & 0xFF);
        buffer.push_back((value >> 24) & 0xFF);
        buffer.push_back((value >> 16) & 0xFF);
        buffer.push_back((value >> 8) & 0xFF);
        buffer.push_back(value & 0xFF);
    }
    
    // Write signed integers (big-endian)
    void write_int32(int32_t value) {
        write_uint32(static_cast<uint32_t>(value));
    }
    
    // Write floating point (big-endian)
    void write_float32(float value) {
        uint32_t bits;
        std::memcpy(&bits, &value, sizeof(float));
        write_uint32(bits);
    }
    
    void write_float64(double value) {
        uint64_t bits;
        std::memcpy(&bits, &value, sizeof(double));
        write_uint64(bits);
    }
    
    // Write strings (UTF-8 with null terminator)
    void write_utf8string(const std::string& str) {
        for (char c : str) {
            buffer.push_back(static_cast<uint8_t>(c));
        }
        buffer.push_back(0);  // Null terminator
    }
    
    // Write four-character code (4CC)
    void write_fourcc(const char* fourcc) {
        size_t len = std::strlen(fourcc);
        if (len != 4) {
            throw std::runtime_error("FourCC must be exactly 4 characters");
        }
        for (int i = 0; i < 4; i++) {
            buffer.push_back(static_cast<uint8_t>(fourcc[i]));
        }
    }
    
    // Write raw bytes
    void write_bytes(const uint8_t* data, size_t length) {
        buffer.insert(buffer.end(), data, data + length);
    }
    
    void write_bytes(const std::vector<uint8_t>& data) {
        buffer.insert(buffer.end(), data.begin(), data.end());
    }
    
    // Get buffer
    const std::vector<uint8_t>& get_buffer() const { return buffer; }
    std::vector<uint8_t>& get_buffer() { return buffer; }
    size_t size() const { return buffer.size(); }
    
    void clear() { buffer.clear(); }
    void reserve(size_t size) { buffer.reserve(size); }
};

// ============================================================================
// ISOBMFF Box Structure
// ============================================================================

class Box {
protected:
    char fourcc[5];  // 4CC + null terminator
    ByteWriter content;
    
public:
    Box(const char* box_type) {
        size_t len = std::strlen(box_type);
        if (len != 4) {
            throw std::runtime_error("Box type must be exactly 4 characters");
        }
        std::memcpy(fourcc, box_type, 4);
        fourcc[4] = '\0';
    }
    
    virtual ~Box() = default;
    
    // Get box type
    const char* get_type() const { return fourcc; }
    
    // Write box header and content
    virtual std::vector<uint8_t> serialize() const {
        ByteWriter writer;
        
        size_t content_size = content.size();
        size_t total_size = 8 + content_size;  // 4 bytes size + 4 bytes type + content
        
        if (total_size < 0xFFFFFFFF) {
            // Standard 32-bit size
            writer.write_uint32(static_cast<uint32_t>(total_size));
            writer.write_fourcc(fourcc);
        } else {
            // Extended 64-bit size
            writer.write_uint32(1);  // Size = 1 indicates extended size
            writer.write_fourcc(fourcc);
            writer.write_uint64(total_size + 8);  // Include extended header size
        }
        
        writer.write_bytes(content.get_buffer());
        
        return writer.get_buffer();
    }
    
    size_t get_size() const {
        return 8 + content.size();
    }
    
protected:
    ByteWriter& get_writer() { return content; }
    const ByteWriter& get_writer() const { return content; }
};

// ============================================================================
// FullBox (Box with version and flags)
// ============================================================================

class FullBox : public Box {
protected:
    uint8_t version;
    uint32_t flags;  // Only 24 bits used
    
public:
    FullBox(const char* box_type, uint8_t ver = 0, uint32_t flgs = 0)
        : Box(box_type), version(ver), flags(flgs & 0x00FFFFFF) {}
    
    std::vector<uint8_t> serialize() const override {
        ByteWriter writer;
        
        // Calculate size (box header + version/flags + content)
        size_t total_size = 8 + 4 + content.size();
        
        if (total_size >= 0xFFFFFFFF) {
            throw std::runtime_error("Box too large for current implementation");
        }
        
        // Write box size and type
        writer.write_uint32(static_cast<uint32_t>(total_size));
        writer.write_fourcc(fourcc);
        
        // Write version (1 byte) and flags (3 bytes)
        writer.write_uint8(version);
        writer.write_uint8((flags >> 16) & 0xFF);
        writer.write_uint8((flags >> 8) & 0xFF);
        writer.write_uint8(flags & 0xFF);
        
        // Write content
        writer.write_bytes(content.get_buffer());
        
        return writer.get_buffer();
    }
    
    uint8_t get_version() const { return version; }
    uint32_t get_flags() const { return flags; }
    
    void set_version(uint8_t ver) { version = ver; }
    void set_flags(uint32_t flgs) { flags = flgs & 0x00FFFFFF; }
};

// ============================================================================
// Container Box (contains other boxes)
// ============================================================================

class ContainerBox : public Box {
protected:
    std::vector<std::shared_ptr<Box>> children;
    
public:
    ContainerBox(const char* box_type) : Box(box_type) {}
    
    void add_child(std::shared_ptr<Box> child) {
        if (child) {
            children.push_back(child);
        }
    }
    
    std::vector<uint8_t> serialize() const override {
        ByteWriter writer;
        
        // Serialize all children first
        ByteWriter children_data;
        for (const auto& child : children) {
            if (child) {
                auto child_data = child->serialize();
                children_data.write_bytes(child_data);
            }
        }
        
        // Calculate total size
        size_t total_size = 8 + children_data.size();
        
        if (total_size >= 0xFFFFFFFF) {
            throw std::runtime_error("Container box too large");
        }
        
        // Write box header
        writer.write_uint32(static_cast<uint32_t>(total_size));
        writer.write_fourcc(fourcc);
        
        // Write children data
        writer.write_bytes(children_data.get_buffer());
        
        return writer.get_buffer();
    }
    
    const std::vector<std::shared_ptr<Box>>& get_children() const { return children; }
    size_t child_count() const { return children.size(); }
};

} // namespace isobmff

// Note: Removed the std::cout statement that was causing the crash

#endif // HEIF_ISOBMFF_BOX_WRITER_H

###  GeoHEIF Properties Header

In [ ]:
#ifndef HEIF_GEOHEIF_PROPERTIES_H
#define HEIF_GEOHEIF_PROPERTIES_H

#include <string>
#include <vector>
#include <cstdint>
#include <cstring>
#include <optional>

// ============================================================================
// GeoHEIF Property Structures (Based on OGC 24-038r1)
// ============================================================================

namespace geoheif {

// --- CRS Encoding Types (Table 2 in spec) ---
enum class CRSEncoding : uint32_t {
    CRS_JSON = 0x6372736A,  // 'crsj' - Reserved, not used
    CRS_URI  = 0x63727375,  // 'crsu' - URI for CRS
    CURIE    = 0x63757269,  // 'curi' - Compact URI (CURIE)
    WKT2     = 0x776B7432   // 'wkt2' - Well-Known Text v2
};

// --- Coordinate Reference System Property (mcrs) ---
struct CoordinateReferenceSystemProperty {
    uint32_t crsEncoding;           // One of CRSEncoding values
    std::string crs;                // CRS definition in specified encoding
    std::optional<float> epoch;     // Coordinate epoch (for dynamic CRS)
    
    CoordinateReferenceSystemProperty() 
        : crsEncoding(static_cast<uint32_t>(CRSEncoding::CURIE))
        , crs("[EPSG:4326]") {}  // Default: WGS84
    
    // Helper: Create from EPSG code using CURIE
    static CoordinateReferenceSystemProperty from_epsg(int epsg_code) {
        CoordinateReferenceSystemProperty prop;
        prop.crsEncoding = static_cast<uint32_t>(CRSEncoding::CURIE);
        prop.crs = "[EPSG:" + std::to_string(epsg_code) + "]";
        return prop;
    }
    
    // Helper: Create from URI
    static CoordinateReferenceSystemProperty from_uri(const std::string& uri) {
        CoordinateReferenceSystemProperty prop;
        prop.crsEncoding = static_cast<uint32_t>(CRSEncoding::CRS_URI);
        prop.crs = uri;
        return prop;
    }
    
    // Helper: Create from WKT2
    static CoordinateReferenceSystemProperty from_wkt2(const std::string& wkt) {
        CoordinateReferenceSystemProperty prop;
        prop.crsEncoding = static_cast<uint32_t>(CRSEncoding::WKT2);
        prop.crs = wkt;
        return prop;
    }
};

// --- Model Transformation Property (mtxf) - Affine Transform ---
struct ModelTransformationProperty {
    bool is_2d;  // true = 2D, false = 3D
    
    // 2D case: 6 values (2x3 matrix)
    double m00, m01, m03;  // First row
    double m10, m11, m13;  // Second row
    
    // 3D case: 12 values (3x4 matrix)
    double m02, m12;       // Additional for 3D
    double m20, m21, m22, m23;  // Third row for 3D
    
    ModelTransformationProperty() 
        : is_2d(true)
        , m00(1.0), m01(0.0), m03(0.0)
        , m10(0.0), m11(1.0), m13(0.0)
        , m02(0.0), m12(0.0)
        , m20(0.0), m21(0.0), m22(1.0), m23(0.0) {}
    
    // Helper: Create from GeoTIFF-style parameters
    static ModelTransformationProperty from_geotiff_style(
        double pixel_scale_x,
        double pixel_scale_y,
        double tie_point_x,
        double tie_point_y,
        double tie_point_i = 0.0,
        double tie_point_j = 0.0
    ) {
        ModelTransformationProperty prop;
        prop.is_2d = true;
        
        // Standard georectified transformation
        prop.m00 = pixel_scale_x;
        prop.m01 = 0.0;
        prop.m03 = tie_point_x - (tie_point_i * pixel_scale_x);
        
        prop.m10 = 0.0;
        prop.m11 = -pixel_scale_y;  // Negative for standard image orientation
        prop.m13 = tie_point_y + (tie_point_j * pixel_scale_y);
        
        return prop;
    }
};

// --- Model Tiepoint Property (tiep) - Georeferenceable ---
struct TiePoint {
    uint32_t i, j;      // Pixel coordinates (integer)
    double x, y;        // Model coordinates
    std::optional<double> z;  // Optional elevation
    
    TiePoint(uint32_t i_, uint32_t j_, double x_, double y_)
        : i(i_), j(j_), x(x_), y(y_) {}
    
    TiePoint(uint32_t i_, uint32_t j_, double x_, double y_, double z_)
        : i(i_), j(j_), x(x_), y(y_), z(z_) {}
};

struct ModelTiepointProperty {
    bool is_2d;  // true = 2D, false = 3D
    std::vector<TiePoint> tie_points;
    
    ModelTiepointProperty() : is_2d(true) {}
    
    void add_tie_point(const TiePoint& tp) {
        tie_points.push_back(tp);
        if (tp.z.has_value()) {
            is_2d = false;
        }
    }
};

// --- Extra Dimension Property (edim) ---
enum class ExtraDimensionType : uint8_t {
    RegularSampled = 0,     // Uniformly sampled
    IrregularValues = 1,    // Non-uniform samples
    Categorical = 2         // Categories
};

struct ExtraDimension {
    std::string name;           // Variable name
    std::string definition;     // URI to definition
    ExtraDimensionType type;
    
    // For RegularSampled (type=0)
    double minimum, maximum, resolution;
    
    // For IrregularValues (type=1) and Categorical (type=2)
    std::vector<double> values;              // For IrregularValues
    std::vector<std::string> categories;     // For Categorical
    std::vector<std::string> value_definitions;  // URIs
    
    // Units (for type 0 and 1)
    std::string unit;
    std::string unit_lang;  // Default: "UCUM"
    
    ExtraDimension()
        : type(ExtraDimensionType::RegularSampled)
        , minimum(0.0), maximum(1.0), resolution(1.0)
        , unit_lang("UCUM") {}
};

struct ExtraDimensionProperty {
    std::vector<ExtraDimension> dimensions;
};

// --- Extra Dimension Value Property (edvl) ---
struct ExtraDimensionValueProperty {
    std::vector<uint32_t> value_indices;  // Index into dimension values
};

// --- Cell Property Type Property (pcel) ---
enum class ComponentFormat : uint8_t {
    UnsignedInt = 0,
    Float = 1,
    Complex = 2,
    SignedInt = 3
};

struct CellPropertyComponent {
    std::string name;
    std::string definition;
    std::string unit;
    std::string unit_lang;
    
    bool measure_is_point;
    
    uint8_t component_bit_depth_minus_one;
    ComponentFormat component_format;
    
    double minimum, maximum;
    
    struct NilValue {
        double value;
        std::string reason;
    };
    std::vector<NilValue> nil_values;
    
    CellPropertyComponent()
        : unit_lang("UCUM")
        , measure_is_point(false)
        , component_bit_depth_minus_one(7)
        , component_format(ComponentFormat::UnsignedInt)
        , minimum(0.0), maximum(255.0) {}
};

struct CellPropertyTypeProperty {
    std::vector<CellPropertyComponent> components;
};

// --- Cell Property Categories Property (pcat) ---
struct PropertyCategory {
    uint32_t value;
    std::string name;
    std::string description;
    std::string definition;
};

struct CellPropertyCategoriesProperty {
    std::vector<std::vector<PropertyCategory>> component_categories;
    uint8_t component_bit_depth_minus_one;
    ComponentFormat component_format;
    
    CellPropertyCategoriesProperty()
        : component_bit_depth_minus_one(7)
        , component_format(ComponentFormat::UnsignedInt) {}
};

// ============================================================================
// Complete GeoHEIF Metadata Container
// ============================================================================

struct GeoHEIFMetadata {
    std::optional<CoordinateReferenceSystemProperty> crs;
    std::optional<ModelTransformationProperty> affine_transform;
    std::optional<ModelTiepointProperty> tie_points;
    std::optional<ExtraDimensionProperty> extra_dimensions;
    std::optional<ExtraDimensionValueProperty> extra_dim_values;
    std::optional<CellPropertyTypeProperty> cell_properties;
    std::optional<CellPropertyCategoriesProperty> cell_categories;
    
    // Validation
    bool is_valid() const {
        if (affine_transform.has_value() || tie_points.has_value()) {
            if (!crs.has_value()) return false;
        }
        
        if (crs.has_value()) {
            if (!affine_transform.has_value() && !tie_points.has_value()) {
                return false;
            }
        }
        
        if (extra_dimensions.has_value() != extra_dim_values.has_value()) {
            return false;
        }
        
        return true;
    }
    
    std::string validation_error() const {
        if (affine_transform.has_value() || tie_points.has_value()) {
            if (!crs.has_value()) return "CRS required when georeferencing is specified";
        }
        
        if (crs.has_value()) {
            if (!affine_transform.has_value() && !tie_points.has_value()) {
                return "Affine transform or tie points required when CRS is specified";
            }
        }
        
        if (extra_dimensions.has_value() != extra_dim_values.has_value()) {
            return "Extra dimensions and values must both be present or absent";
        }
        
        return "Valid";
    }
};

} // namespace geoheif

#endif // HEIF_GEOHEIF_PROPERTIES_H

### GeoHEIF Property Box Implementation

In [ ]:
#ifndef HEIF_GEOHEIF_BOXES_H
#define HEIF_GEOHEIF_BOXES_H

namespace geoheif {

using namespace isobmff;

// ============================================================================
// GeoHEIF Property Boxes (ISO Base Media File Format Extensions)
// ============================================================================

// --- CoordinateReferenceSystemProperty (mcrs) ---
class MCRSBox : public FullBox {
public:
    MCRSBox(const CoordinateReferenceSystemProperty& crs_prop)
        : FullBox("mcrs", 0, crs_prop.epoch.has_value() ? 1 : 0)
    {
        try {
            auto& writer = get_writer();
            
            // Write CRS encoding
            writer.write_uint32(crs_prop.crsEncoding);
            
            // Write CRS string (ensure not empty)
            if (crs_prop.crs.empty()) {
                writer.write_utf8string("[EPSG:4326]");  // Default
            } else {
                writer.write_utf8string(crs_prop.crs);
            }
            
            // Write epoch if present
            if (crs_prop.epoch.has_value()) {
                writer.write_float32(*crs_prop.epoch);
            }
        } catch (const std::exception& e) {
            std::cerr << "Error creating MCRSBox: " << e.what() << std::endl;
            throw;
        }
    }
};

// --- ModelTransformationProperty (mtxf) ---
class MTXFBox : public FullBox {
public:
    MTXFBox(const ModelTransformationProperty& transform)
        : FullBox("mtxf", 0, transform.is_2d ? 0x01 : 0x00)
    {
        try {
            auto& writer = get_writer();
            
            if (transform.is_2d) {
                // 2D case: 6 values (2x3 matrix)
                writer.write_float64(transform.m00);
                writer.write_float64(transform.m01);
                writer.write_float64(transform.m03);
                writer.write_float64(transform.m10);
                writer.write_float64(transform.m11);
                writer.write_float64(transform.m13);
            } else {
                // 3D case: 12 values (3x4 matrix)
                writer.write_float64(transform.m00);
                writer.write_float64(transform.m01);
                writer.write_float64(transform.m02);
                writer.write_float64(transform.m03);
                writer.write_float64(transform.m10);
                writer.write_float64(transform.m11);
                writer.write_float64(transform.m12);
                writer.write_float64(transform.m13);
                writer.write_float64(transform.m20);
                writer.write_float64(transform.m21);
                writer.write_float64(transform.m22);
                writer.write_float64(transform.m23);
            }
        } catch (const std::exception& e) {
            std::cerr << "Error creating MTXFBox: " << e.what() << std::endl;
            throw;
        }
    }
};

// --- ModelTiepointProperty (tiep) ---
class TIEPBox : public FullBox {
public:
    TIEPBox(const ModelTiepointProperty& tiepoint_prop)
        : FullBox("tiep", 0, tiepoint_prop.is_2d ? 0x01 : 0x00)
    {
        try {
            auto& writer = get_writer();
            
            // Validate tie point count
            size_t tp_count = tiepoint_prop.tie_points.size();
            if (tp_count == 0 || tp_count > 65535) {
                throw std::runtime_error("Invalid tie point count");
            }
            
            // Write number of tie points
            writer.write_uint16(static_cast<uint16_t>(tp_count));
            
            // Write each tie point
            for (const auto& tp : tiepoint_prop.tie_points) {
                writer.write_uint32(tp.i);
                writer.write_uint32(tp.j);
                writer.write_float64(tp.x);
                writer.write_float64(tp.y);
                
                if (!tiepoint_prop.is_2d && tp.z.has_value()) {
                    writer.write_float64(*tp.z);
                }
            }
        } catch (const std::exception& e) {
            std::cerr << "Error creating TIEPBox: " << e.what() << std::endl;
            throw;
        }
    }
};

// --- ExtraDimensionProperty (edim) - SIMPLIFIED ---
class EDIMBox : public FullBox {
public:
    EDIMBox(const ExtraDimensionProperty& edim_prop)
        : FullBox("edim", 0, 0)
    {
        try {
            auto& writer = get_writer();
            
            // Validate dimension count
            size_t dim_count = edim_prop.dimensions.size();
            if (dim_count > 255) {  // Reasonable limit
                throw std::runtime_error("Too many dimensions");
            }
            
            // Write number of extra dimensions
            writer.write_uint32(static_cast<uint32_t>(dim_count));
            
            // Write each dimension
            for (const auto& dim : edim_prop.dimensions) {
                // Write name and definition (with validation)
                writer.write_utf8string(dim.name.empty() ? "dimension" : dim.name);
                writer.write_utf8string(dim.definition.empty() ? "" : dim.definition);
                writer.write_uint8(static_cast<uint8_t>(dim.type));
                
                if (dim.type == ExtraDimensionType::RegularSampled) {
                    // Regular sampled range
                    writer.write_float64(dim.minimum);
                    writer.write_float64(dim.maximum);
                    writer.write_float64(dim.resolution);
                    writer.write_utf8string(dim.unit.empty() ? "" : dim.unit);
                    writer.write_utf8string(dim.unit_lang.empty() ? "UCUM" : dim.unit_lang);
                }
                else if (dim.type == ExtraDimensionType::IrregularValues) {
                    // Irregular parameterized values
                    size_t val_count = dim.values.size();
                    if (val_count > 10000) {  // Reasonable limit
                        throw std::runtime_error("Too many irregular values");
                    }
                    
                    writer.write_uint32(static_cast<uint32_t>(val_count));
                    
                    for (size_t j = 0; j < val_count; j++) {
                        writer.write_float64(dim.values[j]);
                        
                        // Write definition if available
                        if (j < dim.value_definitions.size()) {
                            writer.write_utf8string(dim.value_definitions[j]);
                        } else {
                            writer.write_utf8string("");
                        }
                    }
                    
                    writer.write_utf8string(dim.unit.empty() ? "" : dim.unit);
                    writer.write_utf8string(dim.unit_lang.empty() ? "UCUM" : dim.unit_lang);
                }
                else if (dim.type == ExtraDimensionType::Categorical) {
                    // Categorical values
                    size_t cat_count = dim.categories.size();
                    if (cat_count > 10000) {  // Reasonable limit
                        throw std::runtime_error("Too many categories");
                    }
                    
                    writer.write_uint32(static_cast<uint32_t>(cat_count));
                    
                    for (size_t j = 0; j < cat_count; j++) {
                        writer.write_utf8string(dim.categories[j]);
                        
                        // Write definition if available
                        if (j < dim.value_definitions.size()) {
                            writer.write_utf8string(dim.value_definitions[j]);
                        } else {
                            writer.write_utf8string("");
                        }
                    }
                }
            }
        } catch (const std::exception& e) {
            std::cerr << "Error creating EDIMBox: " << e.what() << std::endl;
            throw;
        }
    }
};

// --- ExtraDimensionValueProperty (edvl) ---
class EDVLBox : public FullBox {
public:
    EDVLBox(const ExtraDimensionValueProperty& edvl_prop)
        : FullBox("edvl", 0, 0)
    {
        try {
            auto& writer = get_writer();
            
            // Validate count
            size_t count = edvl_prop.value_indices.size();
            if (count > 255) {
                throw std::runtime_error("Too many dimension values");
            }
            
            // Write number of dimensions
            writer.write_uint32(static_cast<uint32_t>(count));
            
            // Write each value index
            for (uint32_t idx : edvl_prop.value_indices) {
                writer.write_uint32(idx);
            }
        } catch (const std::exception& e) {
            std::cerr << "Error creating EDVLBox: " << e.what() << std::endl;
            throw;
        }
    }
};

// --- CellPropertyTypeProperty (pcel) - SIMPLIFIED ---
class PCELBox : public FullBox {
public:
    PCELBox(const CellPropertyTypeProperty& pcel_prop)
        : FullBox("pcel", 0, 0)
    {
        try {
            auto& writer = get_writer();
            
            // Validate component count
            size_t comp_count = pcel_prop.components.size();
            if (comp_count == 0 || comp_count > 255) {
                throw std::runtime_error("Invalid component count");
            }
            
            // Write number of components
            writer.write_uint32(static_cast<uint32_t>(comp_count));
            
            // Write each component
            for (const auto& comp : pcel_prop.components) {
                writer.write_utf8string(comp.name.empty() ? "component" : comp.name);
                writer.write_utf8string(comp.definition.empty() ? "" : comp.definition);
                writer.write_utf8string(comp.unit.empty() ? "" : comp.unit);
                writer.write_utf8string(comp.unit_lang.empty() ? "UCUM" : comp.unit_lang);
                
                // Write measure_is_point as boolean
                writer.write_uint8(comp.measure_is_point ? 1 : 0);
                
                writer.write_uint8(comp.component_bit_depth_minus_one);
                writer.write_uint8(static_cast<uint8_t>(comp.component_format));
                
                // Write min/max based on format (simplified - use doubles)
                writer.write_float64(comp.minimum);
                writer.write_float64(comp.maximum);
                
                // Write nil values (limit to reasonable count)
                size_t nil_count = std::min(comp.nil_values.size(), size_t(10));
                writer.write_uint8(static_cast<uint8_t>(nil_count));
                
                for (size_t i = 0; i < nil_count; i++) {
                    const auto& nil = comp.nil_values[i];
                    writer.write_float64(nil.value);
                    writer.write_utf8string(nil.reason.empty() ? "" : nil.reason);
                }
            }
        } catch (const std::exception& e) {
            std::cerr << "Error creating PCELBox: " << e.what() << std::endl;
            throw;
        }
    }
};

// --- CellPropertyCategoriesProperty (pcat) - SIMPLIFIED ---
class PCATBox : public FullBox {
public:
    PCATBox(const CellPropertyCategoriesProperty& pcat_prop)
        : FullBox("pcat", 0, 0)
    {
        try {
            auto& writer = get_writer();
            
            // Validate component count
            size_t comp_count = pcat_prop.component_categories.size();
            if (comp_count == 0 || comp_count > 255) {
                throw std::runtime_error("Invalid component category count");
            }
            
            // Write number of components
            writer.write_uint32(static_cast<uint32_t>(comp_count));
            
            // Write each component's categories
            for (const auto& categories : pcat_prop.component_categories) {
                size_t cat_count = std::min(categories.size(), size_t(10000));
                
                writer.write_uint32(static_cast<uint32_t>(cat_count));
                writer.write_uint8(pcat_prop.component_bit_depth_minus_one);
                writer.write_uint8(static_cast<uint8_t>(pcat_prop.component_format));
                
                for (size_t i = 0; i < cat_count; i++) {
                    const auto& cat = categories[i];
                    
                    // Write value (simplified - always as uint32)
                    writer.write_uint32(cat.value);
                    
                    writer.write_utf8string(cat.name.empty() ? "" : cat.name);
                    writer.write_utf8string(cat.description.empty() ? "" : cat.description);
                    writer.write_utf8string(cat.definition.empty() ? "" : cat.definition);
                }
            }
        } catch (const std::exception& e) {
            std::cerr << "Error creating PCATBox: " << e.what() << std::endl;
            throw;
        }
    }
};

} // namespace geoheif

#endif // HEIF_GEOHEIF_BOXES_H

### HEIF File modifier (Add Properties to Existing HEIF)

In [ ]:
#ifndef HEIF_GEOHEIF_FILE_MODIFIER_H
#define HEIF_GEOHEIF_FILE_MODIFIER_H

#include <fstream>
#include <map>
#include <iostream>
#include <algorithm>

namespace geoheif {

// ============================================================================
// HEIF File Modifier - Add GeoHEIF Properties to Existing HEIF
// ============================================================================

class HEIFFileModifier {
private:
    struct BoxLocation {
        size_t offset;
        size_t size;
        std::string type;
        size_t header_size;
        
        BoxLocation() : offset(0), size(0), header_size(8) {}
    };
    
    std::vector<BoxLocation> box_locations;
    std::vector<uint8_t> file_data;
    
    // Safe read operations with bounds checking
    bool safe_read_uint32(size_t offset, uint32_t& result) const {
        if (offset + 4 > file_data.size()) {
            return false;
        }
        result = (static_cast<uint32_t>(file_data[offset]) << 24) |
                (static_cast<uint32_t>(file_data[offset + 1]) << 16) |
                (static_cast<uint32_t>(file_data[offset + 2]) << 8) |
                static_cast<uint32_t>(file_data[offset + 3]);
        return true;
    }
    
    bool safe_read_uint64(size_t offset, uint64_t& result) const {
        if (offset + 8 > file_data.size()) {
            return false;
        }
        uint32_t high, low;
        if (!safe_read_uint32(offset, high)) return false;
        if (!safe_read_uint32(offset + 4, low)) return false;
        result = (static_cast<uint64_t>(high) << 32) | static_cast<uint64_t>(low);
        return true;
    }
    
    bool safe_write_uint32(size_t offset, uint32_t value) {
        if (offset + 4 > file_data.size()) {
            return false;
        }
        file_data[offset] = (value >> 24) & 0xFF;
        file_data[offset + 1] = (value >> 16) & 0xFF;
        file_data[offset + 2] = (value >> 8) & 0xFF;
        file_data[offset + 3] = value & 0xFF;
        return true;
    }
    
    // Safe box parsing with recursion limit
    bool parse_boxes_recursive(size_t offset, size_t end_offset, int depth = 0) {
        const int MAX_DEPTH = 20;  // Prevent infinite recursion
        
        if (depth > MAX_DEPTH) {
            std::cerr << "Warning: Maximum parsing depth exceeded" << std::endl;
            return false;
        }
        
        while (offset < end_offset && offset < file_data.size()) {
            // Need at least 8 bytes for box header
            if (offset + 8 > file_data.size() || offset + 8 > end_offset) {
                break;
            }
            
            uint32_t size;
            if (!safe_read_uint32(offset, size)) {
                break;
            }
            
            // Read box type (4CC)
            if (offset + 8 > file_data.size()) {
                break;
            }
            
            char type[5] = {0};
            std::memcpy(type, &file_data[offset + 4], 4);
            
            // Validate type characters
            bool valid_type = true;
            for (int i = 0; i < 4; i++) {
                if (!std::isprint(static_cast<unsigned char>(type[i]))) {
                    valid_type = false;
                    break;
                }
            }
            
            if (!valid_type) {
                std::cerr << "Warning: Invalid box type at offset " << offset << std::endl;
                break;
            }
            
            size_t box_size = size;
            size_t header_size = 8;
            
            if (size == 1) {
                // Extended size (64-bit)
                if (offset + 16 > file_data.size() || offset + 16 > end_offset) {
                    break;
                }
                uint64_t size64;
                if (!safe_read_uint64(offset + 8, size64)) {
                    break;
                }
                box_size = static_cast<size_t>(size64);
                header_size = 16;
            } else if (size == 0) {
                // To end of file
                box_size = file_data.size() - offset;
            }
            
            // Validate box size
            if (box_size < header_size) {
                std::cerr << "Warning: Invalid box size " << box_size 
                          << " at offset " << offset << std::endl;
                break;
            }
            
            if (offset + box_size > file_data.size()) {
                std::cerr << "Warning: Box extends beyond file at offset " << offset << std::endl;
                break;
            }
            
            // Store box location
            BoxLocation loc;
            loc.offset = offset;
            loc.size = box_size;
            loc.type = type;
            loc.header_size = header_size;
            box_locations.push_back(loc);
            
            // Recursively parse container boxes
            std::string type_str(type);
            if (type_str == "meta" || type_str == "iprp" || 
                type_str == "ipco" || type_str == "moov") {
                
                size_t content_start = offset + header_size;
                
                // For 'meta' box, skip version/flags (4 bytes)
                if (type_str == "meta") {
                    content_start += 4;
                }
                
                if (content_start < offset + box_size) {
                    parse_boxes_recursive(content_start, offset + box_size, depth + 1);
                }
            }
            
            offset += box_size;
        }
        
        return true;
    }
    
public:
    HEIFFileModifier() {}
    
    // Load HEIF file safely
    bool load(const std::string& filename) {
        try {
            std::ifstream file(filename, std::ios::binary | std::ios::ate);
            if (!file) {
                std::cerr << "Cannot open file: " << filename << std::endl;
                return false;
            }
            
            std::streamsize file_size = file.tellg();
            if (file_size <= 0 || file_size > 1024 * 1024 * 1024) {  // Max 1GB
                std::cerr << "Invalid file size: " << file_size << std::endl;
                return false;
            }
            
            file.seekg(0, std::ios::beg);
            
            file_data.clear();
            file_data.resize(static_cast<size_t>(file_size));
            
            if (!file.read(reinterpret_cast<char*>(file_data.data()), file_size)) {
                std::cerr << "Failed to read file" << std::endl;
                return false;
            }
            
            // Verify this looks like an ISOBMFF file
            if (file_data.size() < 8) {
                std::cerr << "File too small to be valid HEIF" << std::endl;
                return false;
            }
            
            // Check for ftyp box at start
            char first_type[5] = {0};
            if (file_data.size() >= 8) {
                std::memcpy(first_type, &file_data[4], 4);
                if (std::string(first_type) != "ftyp") {
                    std::cerr << "Warning: File does not start with ftyp box" << std::endl;
                }
            }
            
            // Parse box structure
            box_locations.clear();
            if (!parse_boxes_recursive(0, file_data.size())) {
                std::cerr << "Warning: Box parsing encountered errors" << std::endl;
            }
            
            std::cout << "Loaded HEIF file: " << filename 
                      << " (" << file_size << " bytes)" << std::endl;
            std::cout << "Found " << box_locations.size() << " boxes" << std::endl;
            
            return true;
            
        } catch (const std::exception& e) {
            std::cerr << "Exception loading file: " << e.what() << std::endl;
            return false;
        }
    }
    
    // Find box by type
    BoxLocation* find_box(const std::string& box_type) {
        for (auto& loc : box_locations) {
            if (loc.type == box_type) {
                return &loc;
            }
        }
        return nullptr;
    }
    
    // Find all boxes of a type
    std::vector<BoxLocation*> find_all_boxes(const std::string& box_type) {
        std::vector<BoxLocation*> results;
        for (auto& loc : box_locations) {
            if (loc.type == box_type) {
                results.push_back(&loc);
            }
        }
        return results;
    }
    
    // Add 'ogeo' brand to ftyp box
    bool add_ogeo_brand() {
        auto* ftyp = find_box("ftyp");
        if (!ftyp) {
            std::cerr << "ftyp box not found" << std::endl;
            return false;
        }
        
        std::cout << "Adding 'ogeo' brand to ftyp..." << std::endl;
        
        try {
            // Create new brand
            std::vector<uint8_t> ogeo_brand = {'o', 'g', 'e', 'o'};
            
            // Insert at end of ftyp box
            size_t insert_pos = ftyp->offset + ftyp->size;
            if (insert_pos > file_data.size()) {
                std::cerr << "Invalid insert position" << std::endl;
                return false;
            }
            
            file_data.insert(file_data.begin() + insert_pos, 
                            ogeo_brand.begin(), ogeo_brand.end());
            
            // Update ftyp size
            if (!safe_write_uint32(ftyp->offset, static_cast<uint32_t>(ftyp->size + 4))) {
                std::cerr << "Failed to update ftyp size" << std::endl;
                return false;
            }
            
            // Update all subsequent box offsets
            for (auto& loc : box_locations) {
                if (loc.offset > ftyp->offset) {
                    loc.offset += 4;
                }
            }
            
            ftyp->size += 4;
            
            std::cout << "  ✓ 'ogeo' brand added" << std::endl;
            return true;
            
        } catch (const std::exception& e) {
            std::cerr << "Exception adding brand: " << e.what() << std::endl;
            return false;
        }
    }
    
    // Add GeoHEIF properties to ipco
    bool add_geoheif_properties(const GeoHEIFMetadata& metadata) {
        auto* ipco = find_box("ipco");
        if (!ipco) {
            std::cerr << "ipco (ItemPropertyContainerBox) not found" << std::endl;
            std::cerr << "This may not be a valid HEIF file or uses different structure" << std::endl;
            
            // List what boxes we found for debugging
            std::cout << "\nAvailable boxes:" << std::endl;
            std::map<std::string, int> box_counts;
            for (const auto& loc : box_locations) {
                box_counts[loc.type]++;
            }
            for (const auto& [type, count] : box_counts) {
                std::cout << "  " << type << ": " << count << std::endl;
            }
            
            return false;
        }
        
        std::cout << "\nAdding GeoHEIF properties to ipco..." << std::endl;
        std::cout << "  ipco box at offset " << ipco->offset 
                  << ", size " << ipco->size << std::endl;
        
        try {
            std::vector<std::shared_ptr<isobmff::Box>> property_boxes;
            
            // Create property boxes
            if (metadata.crs.has_value()) {
                std::cout << "  Creating mcrs box..." << std::endl;
                property_boxes.push_back(std::make_shared<MCRSBox>(*metadata.crs));
            }
            
            if (metadata.affine_transform.has_value()) {
                std::cout << "  Creating mtxf box..." << std::endl;
                property_boxes.push_back(std::make_shared<MTXFBox>(*metadata.affine_transform));
            }
            
            if (metadata.tie_points.has_value()) {
                std::cout << "  Creating tiep box..." << std::endl;
                property_boxes.push_back(std::make_shared<TIEPBox>(*metadata.tie_points));
            }
            
            if (metadata.extra_dimensions.has_value()) {
                std::cout << "  Creating edim box..." << std::endl;
                property_boxes.push_back(std::make_shared<EDIMBox>(*metadata.extra_dimensions));
            }
            
            if (metadata.extra_dim_values.has_value()) {
                std::cout << "  Creating edvl box..." << std::endl;
                property_boxes.push_back(std::make_shared<EDVLBox>(*metadata.extra_dim_values));
            }
            
            if (metadata.cell_properties.has_value()) {
                std::cout << "  Creating pcel box..." << std::endl;
                property_boxes.push_back(std::make_shared<PCELBox>(*metadata.cell_properties));
            }
            
            if (metadata.cell_categories.has_value()) {
                std::cout << "  Creating pcat box..." << std::endl;
                property_boxes.push_back(std::make_shared<PCATBox>(*metadata.cell_categories));
            }
            
            // Serialize all properties
            isobmff::ByteWriter properties_data;
            for (const auto& box : property_boxes) {
                auto box_data = box->serialize();
                properties_data.write_bytes(box_data);
                std::cout << "    Added " << box->get_type() 
                          << " (" << box_data.size() << " bytes)" << std::endl;
            }
            
            size_t added_size = properties_data.size();
            if (added_size == 0) {
                std::cout << "  No properties to add" << std::endl;
                return true;
            }
            
            std::cout << "  Total properties size: " << added_size << " bytes" << std::endl;
            
            // Insert at end of ipco (before closing)
            size_t insert_position = ipco->offset + ipco->size;
            
            if (insert_position > file_data.size()) {
                std::cerr << "Invalid insert position: " << insert_position 
                          << " (file size: " << file_data.size() << ")" << std::endl;
                return false;
            }
            
            file_data.insert(file_data.begin() + insert_position,
                            properties_data.get_buffer().begin(),
                            properties_data.get_buffer().end());
            
            // Update ipco size
            if (!safe_write_uint32(ipco->offset, static_cast<uint32_t>(ipco->size + added_size))) {
                std::cerr << "Failed to update ipco size" << std::endl;
                return false;
            }
            
            // Update parent boxes (iprp, meta) sizes
            update_parent_box_sizes(ipco->offset, added_size);
            
            // Update all subsequent box offsets
            for (auto& loc : box_locations) {
                if (loc.offset >= insert_position && loc.offset != ipco->offset) {
                    loc.offset += added_size;
                }
            }
            
            ipco->size += added_size;
            
            std::cout << "  ✓ Added " << added_size << " bytes of GeoHEIF properties" << std::endl;
            return true;
            
        } catch (const std::exception& e) {
            std::cerr << "Exception adding properties: " << e.what() << std::endl;
            return false;
        }
    }
    
    // Update sizes of parent boxes
    void update_parent_box_sizes(size_t box_offset, size_t size_increase) {
        try {
            // Find and update parent boxes
            for (auto& loc : box_locations) {
                if (loc.type == "iprp" || loc.type == "meta" || loc.type == "moov") {
                    // Check if this box contains the modified box
                    if (loc.offset < box_offset && 
                        (loc.offset + loc.size) > box_offset) {
                        
                        uint32_t new_size = static_cast<uint32_t>(loc.size + size_increase);
                        if (safe_write_uint32(loc.offset, new_size)) {
                            std::cout << "    Updated " << loc.type << " size to " 
                                      << new_size << std::endl;
                            loc.size += size_increase;
                        }
                    }
                }
            }
        } catch (const std::exception& e) {
            std::cerr << "Exception updating parent sizes: " << e.what() << std::endl;
        }
    }
    
    // Save modified file
    bool save(const std::string& filename) {
        try {
            std::ofstream file(filename, std::ios::binary);
            if (!file) {
                std::cerr << "Cannot write file: " << filename << std::endl;
                return false;
            }
            
            file.write(reinterpret_cast<const char*>(file_data.data()), file_data.size());
            
            if (!file) {
                std::cerr << "Error writing file" << std::endl;
                return false;
            }
            
            std::cout << "\n✓ Saved GeoHEIF file: " << filename 
                      << " (" << file_data.size() << " bytes)" << std::endl;
            return true;
            
        } catch (const std::exception& e) {
            std::cerr << "Exception saving file: " << e.what() << std::endl;
            return false;
        }
    }
    
    // Print box structure (safely)
    void print_structure() const {
        std::cout << "\n=== HEIF File Structure ===" << std::endl;
        
        std::map<std::string, std::vector<BoxLocation>> boxes_by_type;
        for (const auto& loc : box_locations) {
            boxes_by_type[loc.type].push_back(loc);
        }
        
        for (const auto& [type, locs] : boxes_by_type) {
            std::cout << "\n[" << type << "] (" << locs.size() << " instance(s))" << std::endl;
            for (const auto& loc : locs) {
                std::cout << "    offset=" << loc.offset 
                          << " size=" << loc.size 
                          << " header=" << loc.header_size << std::endl;
            }
        }
        
        std::cout << "\nTotal boxes: " << box_locations.size() << std::endl;
        std::cout << "File size: " << file_data.size() << " bytes" << std::endl;
    }
};

} // namespace geoheif

#endif // HEIF_GEOHEIF_FILE_MODIFIER_H

### Complete Workign Example

In [ ]:
#ifndef HEIF_GEOHEIF_COMPLETE_EXAMPLE_H
#define HEIF_GEOHEIF_COMPLETE_EXAMPLE_H

namespace geoheif {

// ============================================================================
// Safe GeoHEIF Creation - Step by Step
// ============================================================================

// Step 1: Create basic HEIF (NO GeoHEIF yet)
bool step1_create_basic_heif(
    const char* input_image,
    const char* output_heif,
    ExtendedCompressionType compression = ExtendedCompressionType::HEVC,
    int quality = 70
) {
    std::cout << "\n[Step 1] Creating basic HEIF..." << std::endl;
    
    try {
        // Just encode a regular HEIF
        encode_nontiled(input_image, output_heif, compression, quality, 8);
        std::cout << "  ✓ Basic HEIF created: " << output_heif << std::endl;
        return true;
    } catch (const std::exception& e) {
        std::cerr << "  ✗ Error: " << e.what() << std::endl;
        return false;
    }
}

// Step 2: Load and analyze HEIF structure
bool step2_analyze_heif(const char* heif_file) {
    std::cout << "\n[Step 2] Analyzing HEIF structure..." << std::endl;
    
    try {
        HEIFFileModifier modifier;
        if (!modifier.load(heif_file)) {
            return false;
        }
        
        modifier.print_structure();
        std::cout << "  ✓ Analysis complete" << std::endl;
        return true;
    } catch (const std::exception& e) {
        std::cerr << "  ✗ Error: " << e.what() << std::endl;
        return false;
    }
}

// Step 3: Add ogeo brand only
bool step3_add_brand(const char* input_heif, const char* output_heif) {
    std::cout << "\n[Step 3] Adding 'ogeo' brand..." << std::endl;
    
    try {
        HEIFFileModifier modifier;
        
        if (!modifier.load(input_heif)) {
            return false;
        }
        
        if (!modifier.add_ogeo_brand()) {
            return false;
        }
        
        if (!modifier.save(output_heif)) {
            return false;
        }
        
        std::cout << "  ✓ Brand added successfully" << std::endl;
        return true;
    } catch (const std::exception& e) {
        std::cerr << "  ✗ Error: " << e.what() << std::endl;
        return false;
    }
}

// Step 4: Add GeoHEIF properties
bool step4_add_properties(
    const char* input_heif, 
    const char* output_heif,
    const GeoHEIFMetadata& metadata
) {
    std::cout << "\n[Step 4] Adding GeoHEIF properties..." << std::endl;
    
    try {
        // Validate metadata first
        if (!metadata.is_valid()) {
            std::cerr << "  ✗ Invalid metadata: " << metadata.validation_error() << std::endl;
            return false;
        }
        
        HEIFFileModifier modifier;
        
        if (!modifier.load(input_heif)) {
            return false;
        }
        
        if (!modifier.add_geoheif_properties(metadata)) {
            return false;
        }
        
        if (!modifier.save(output_heif)) {
            return false;
        }
        
        std::cout << "  ✓ Properties added successfully" << std::endl;
        return true;
    } catch (const std::exception& e) {
        std::cerr << "  ✗ Error: " << e.what() << std::endl;
        return false;
    }
}

// Complete workflow - all steps combined (SAFE)
bool create_geoheif_safe(
    const char* input_image,
    const char* output_geoheif,
    const GeoHEIFMetadata& metadata,
    const char* temp_dir = nullptr
) {
    std::cout << "\n" << std::string(80, '=') << std::endl;
    std::cout << "Creating GeoHEIF (Safe Step-by-Step)" << std::endl;
    std::cout << std::string(80, '=') << std::endl;
    
    try {
        // Determine temp directory
        std::string temp_path;
        if (temp_dir) {
            temp_path = temp_dir;
        } else {
            // Use same directory as output
            fs::path output_path(output_geoheif);
            temp_path = output_path.parent_path().string();
        }
        
        // Generate temp filenames
        std::string temp1 = temp_path + "/temp_step1.heif";
        std::string temp2 = temp_path + "/temp_step2.heif";
        
        // Step 1: Create basic HEIF
        if (!step1_create_basic_heif(input_image, temp1.c_str())) {
            std::cerr << "\n✗ Failed at step 1" << std::endl;
            return false;
        }
        
        // Step 2: Analyze (optional, for debugging)
        // step2_analyze_heif(temp1.c_str());
        
        // Step 3: Add brand
        if (!step3_add_brand(temp1.c_str(), temp2.c_str())) {
            std::cerr << "\n✗ Failed at step 3" << std::endl;
            std::remove(temp1.c_str());
            return false;
        }
        
        // Step 4: Add properties
        if (!step4_add_properties(temp2.c_str(), output_geoheif, metadata)) {
            std::cerr << "\n✗ Failed at step 4" << std::endl;
            std::remove(temp1.c_str());
            std::remove(temp2.c_str());
            return false;
        }
        
        // Cleanup temp files
        std::remove(temp1.c_str());
        std::remove(temp2.c_str());
        
        std::cout << "\n" << std::string(80, '=') << std::endl;
        std::cout << "✓ GeoHEIF created successfully!" << std::endl;
        std::cout << "  Output: " << output_geoheif << std::endl;
        std::cout << std::string(80, '=') << std::endl;
        
        return true;
        
    } catch (const std::exception& e) {
        std::cerr << "\n✗ Exception: " << e.what() << std::endl;
        return false;
    }
}

// ============================================================================
// Simple Example Functions (Minimal, Safe)
// ============================================================================

// Example 1: Minimal georectified (WGS84, global extent)
void example_minimal_wgs84(const char* input_file, const char* output_file) {
    std::cout << "\n=== Example: Minimal WGS84 GeoHEIF ===" << std::endl;
    
    try {
        // Create minimal metadata
        GeoHEIFMetadata metadata;
        metadata.crs = CoordinateReferenceSystemProperty::from_epsg(4326);
        metadata.affine_transform = ModelTransformationProperty::from_geotiff_style(
            0.1, 0.1, -180.0, 90.0, 0.0, 0.0
        );
        
        // Create GeoHEIF
        create_geoheif_safe(input_file, output_file, metadata);
        
    } catch (const std::exception& e) {
        std::cerr << "Error: " << e.what() << std::endl;
    }
}

// Example 2: UTM projection
void example_utm_projection(const char* input_file, const char* output_file) {
    std::cout << "\n=== Example: UTM Projection GeoHEIF ===" << std::endl;
    
    try {
        // Create UTM metadata
        GeoHEIFMetadata metadata;
        metadata.crs = CoordinateReferenceSystemProperty::from_epsg(32755);  // UTM Zone 55S
        metadata.affine_transform = ModelTransformationProperty::from_geotiff_style(
            10.0,       // 10m pixel size X
            10.0,       // 10m pixel size Y
            300000.0,   // Easting origin
            6100000.0,  // Northing origin
            0.0, 0.0
        );
        
        // Create GeoHEIF
        create_geoheif_safe(input_file, output_file, metadata);
        
    } catch (const std::exception& e) {
        std::cerr << "Error: " << e.what() << std::endl;
    }
}

// Example 3: From GeoTIFF (extract metadata)
void example_from_geotiff_safe(const char* geotiff_file, const char* output_file) {
    std::cout << "\n=== Example: GeoHEIF from GeoTIFF (Safe) ===" << std::endl;
    
    try {
        // Open GeoTIFF with GDAL
        GDALAllRegister();
        GDALDataset* dataset = (GDALDataset*)GDALOpen(geotiff_file, GA_ReadOnly);
        
        if (!dataset) {
            std::cerr << "Cannot open GeoTIFF: " << geotiff_file << std::endl;
            return;
        }
        
        // Extract basic info
        int width = dataset->GetRasterXSize();
        int height = dataset->GetRasterYSize();
        
        std::cout << "  Image size: " << width << "x" << height << std::endl;
        
        // Create metadata
        GeoHEIFMetadata metadata;
        
        // Get CRS
        const char* proj_wkt = dataset->GetProjectionRef();
        if (proj_wkt && strlen(proj_wkt) > 0) {
            std::cout << "  Found projection (WKT)" << std::endl;
            metadata.crs = CoordinateReferenceSystemProperty::from_wkt2(proj_wkt);
        } else {
            std::cout << "  No projection found, using WGS84" << std::endl;
            metadata.crs = CoordinateReferenceSystemProperty::from_epsg(4326);
        }
        
        // Get geotransform
        double gt[6];
        if (dataset->GetGeoTransform(gt) == CE_None) {
            std::cout << "  Found geotransform" << std::endl;
            
            ModelTransformationProperty transform;
            transform.is_2d = true;
            transform.m00 = gt[1];   // pixel width
            transform.m01 = gt[2];   // rotation
            transform.m03 = gt[0];   // origin X
            transform.m10 = gt[4];   // rotation
            transform.m11 = gt[5];   // pixel height (negative)
            transform.m13 = gt[3];   // origin Y
            
            metadata.affine_transform = transform;
            
            std::cout << "  Pixel size: " << gt[1] << " x " << -gt[5] << std::endl;
        } else {
            std::cout << "  No geotransform found" << std::endl;
            GDALClose(dataset);
            return;
        }
        
        GDALClose(dataset);
        
        // Create GeoHEIF
        create_geoheif_safe(geotiff_file, output_file, metadata);
        
    } catch (const std::exception& e) {
        std::cerr << "Error: " << e.what() << std::endl;
    }
}

// ============================================================================
// Interactive Testing Functions
// ============================================================================

// Test individual steps
void test_individual_steps(const char* input_file, const char* output_dir) {
    std::cout << "\n=== Testing Individual Steps ===" << std::endl;
    
    try {
        std::string temp1 = std::string(output_dir) + "/test_step1.heif";
        std::string temp2 = std::string(output_dir) + "/test_step2.heif";
        std::string temp3 = std::string(output_dir) + "/test_step3.heif";
        
        // Test Step 1
        std::cout << "\nTesting Step 1..." << std::endl;
        if (step1_create_basic_heif(input_file, temp1.c_str())) {
            std::cout << "✓ Step 1 OK" << std::endl;
        } else {
            std::cout << "✗ Step 1 FAILED" << std::endl;
            return;
        }
        
        // Test Step 2
        std::cout << "\nTesting Step 2..." << std::endl;
        if (step2_analyze_heif(temp1.c_str())) {
            std::cout << "✓ Step 2 OK" << std::endl;
        } else {
            std::cout << "✗ Step 2 FAILED" << std::endl;
            return;
        }
        
        // Test Step 3
        std::cout << "\nTesting Step 3..." << std::endl;
        if (step3_add_brand(temp1.c_str(), temp2.c_str())) {
            std::cout << "✓ Step 3 OK" << std::endl;
        } else {
            std::cout << "✗ Step 3 FAILED" << std::endl;
            return;
        }
        
        // Test Step 4
        std::cout << "\nTesting Step 4..." << std::endl;
        GeoHEIFMetadata metadata;
        metadata.crs = CoordinateReferenceSystemProperty::from_epsg(4326);
        metadata.affine_transform = ModelTransformationProperty::from_geotiff_style(
            0.1, 0.1, -180.0, 90.0, 0.0, 0.0
        );
        
        if (step4_add_properties(temp2.c_str(), temp3.c_str(), metadata)) {
            std::cout << "✓ Step 4 OK" << std::endl;
        } else {
            std::cout << "✗ Step 4 FAILED" << std::endl;
            return;
        }
        
        std::cout << "\n✓ All individual steps passed!" << std::endl;
        
        // Analyze final result
        std::cout << "\nFinal result:" << std::endl;
        step2_analyze_heif(temp3.c_str());
        
    } catch (const std::exception& e) {
        std::cerr << "✗ Exception: " << e.what() << std::endl;
    }
}

} // namespace geoheif

#endif // HEIF_GEOHEIF_COMPLETE_EXAMPLE_H

### Usage examples

In [ ]:
// // Test the complete GeoHEIF implementation

// // Example 1: Simple georectified image
// geoheif::example_complete_georectified(
//     input_file,
//     (fs::path(output_dir) / "output_geoheif_complete.heif").string().c_str()
// );

// // Example 2: From GeoTIFF
// geoheif::example_complete_from_geotiff(
//     input_file,  // Assuming it's a GeoTIFF
//     (fs::path(output_dir) / "output_geoheif_from_geotiff.heif").string().c_str()
// );

// // Example 3: Multispectral
// geoheif::example_complete_multispectral(
//     input_file,
//     (fs::path(output_dir) / "output_geoheif_multispectral.heif").string().c_str()
// );

// // Or use the two-step process manually:
// // Step 1: Encode regular HEIF
// std::string temp_heif = (fs::path(output_dir) / "temp.heif").string();
// encode_nontiled(input_file, temp_heif.c_str(), ExtendedCompressionType::HEVC, 70);

// // Step 2: Add GeoHEIF metadata
// auto metadata = geoheif::create_simple_georectified_metadata(
//     4326, -180.0, -90.0, 180.0, 90.0, 3600, 1800
// );

// geoheif::create_geoheif_from_heif(
//     temp_heif.c_str(),
//     (fs::path(output_dir) / "output_manual_geoheif.heif").string().c_str(),
//     metadata
// );

## Modify the HEIF with the GeoTIFF CRS

### GeoTIFF CRS Extractor

In [ ]:
#ifndef HEIF_GEOTIFF_CRS_EXTRACTOR_H
#define HEIF_GEOTIFF_CRS_EXTRACTOR_H

namespace geoheif {

// ============================================================================
// GeoTIFF CRS and Georeference Extractor
// ============================================================================

struct GeoTIFFGeoreference {
    // Image dimensions
    int width;
    int height;
    int bands;
    
    // CRS information
    bool has_crs;
    std::string crs_wkt;
    std::string crs_proj4;
    int epsg_code;  // -1 if not EPSG
    
    // Geotransform
    bool has_geotransform;
    double geotransform[6];
    
    // Computed bounds
    double min_x, max_x, min_y, max_y;
    double pixel_size_x, pixel_size_y;
    
    // GCPs (Ground Control Points) - for georeferenceable images
    bool has_gcps;
    int gcp_count;
    std::vector<TiePoint> gcps;
    
    GeoTIFFGeoreference()
        : width(0), height(0), bands(0)
        , has_crs(false), epsg_code(-1)
        , has_geotransform(false)
        , min_x(0), max_x(0), min_y(0), max_y(0)
        , pixel_size_x(0), pixel_size_y(0)
        , has_gcps(false), gcp_count(0)
    {
        for (int i = 0; i < 6; i++) geotransform[i] = 0.0;
    }
};

class GeoTIFFExtractor {
public:
    // Extract all georeference information from GeoTIFF
    static bool extract(const char* geotiff_file, GeoTIFFGeoreference& georef) {
        std::cout << "\n=== Extracting GeoTIFF Georeference ===" << std::endl;
        std::cout << "File: " << geotiff_file << std::endl;
        
        GDALAllRegister();
        
        GDALDataset* dataset = (GDALDataset*)GDALOpen(geotiff_file, GA_ReadOnly);
        if (!dataset) {
            std::cerr << "✗ Cannot open GeoTIFF file" << std::endl;
            return false;
        }
        
        try {
            // Get image dimensions
            georef.width = dataset->GetRasterXSize();
            georef.height = dataset->GetRasterYSize();
            georef.bands = dataset->GetRasterCount();
            
            std::cout << "\n📐 Image Properties:" << std::endl;
            std::cout << "   Size: " << georef.width << " x " << georef.height << std::endl;
            std::cout << "   Bands: " << georef.bands << std::endl;
            
            // Get CRS information
            std::cout << "\n🌍 Coordinate Reference System:" << std::endl;
            const char* proj_wkt = dataset->GetProjectionRef();
            
            if (proj_wkt && strlen(proj_wkt) > 0) {
                georef.has_crs = true;
                georef.crs_wkt = proj_wkt;
                
                // Try to get EPSG code
                OGRSpatialReference srs;
                if (srs.importFromWkt(proj_wkt) == OGRERR_NONE) {
                    const char* auth_name = srs.GetAuthorityName(nullptr);
                    const char* auth_code = srs.GetAuthorityCode(nullptr);
                    
                    if (auth_name && auth_code && 
                        std::string(auth_name) == "EPSG") {
                        georef.epsg_code = std::atoi(auth_code);
                        std::cout << "   ✓ EPSG Code: " << georef.epsg_code << std::endl;
                    }
                    
                    // Get PROJ4 string
                    char* proj4 = nullptr;
                    if (srs.exportToProj4(&proj4) == OGRERR_NONE && proj4) {
                        georef.crs_proj4 = proj4;
                        CPLFree(proj4);
                        std::cout << "   ✓ PROJ4: " << georef.crs_proj4 << std::endl;
                    }
                    
                    // Get CRS name
                    const char* crs_name = srs.GetName();
                    if (crs_name) {
                        std::cout << "   ✓ Name: " << crs_name << std::endl;
                    }
                } else {
                    std::cout << "   ⚠ Has WKT but cannot parse as OGRSpatialReference" << std::endl;
                }
            } else {
                std::cout << "   ⚠ No CRS information found" << std::endl;
            }
            
            // Get geotransform (affine transformation)
            std::cout << "\n📊 Geotransform:" << std::endl;
            if (dataset->GetGeoTransform(georef.geotransform) == CE_None) {
                georef.has_geotransform = true;
                
                // Parse geotransform
                // GT[0] = X origin (top-left X)
                // GT[1] = pixel width
                // GT[2] = rotation (0 if north-up)
                // GT[3] = Y origin (top-left Y)
                // GT[4] = rotation (0 if north-up)
                // GT[5] = pixel height (negative if north-up)
                
                double origin_x = georef.geotransform[0];
                double origin_y = georef.geotransform[3];
                georef.pixel_size_x = georef.geotransform[1];
                georef.pixel_size_y = std::abs(georef.geotransform[5]);
                
                // Compute bounds
                georef.min_x = origin_x;
                georef.max_x = origin_x + (georef.width * georef.pixel_size_x);
                georef.max_y = origin_y;
                georef.min_y = origin_y - (georef.height * georef.pixel_size_y);
                
                std::cout << "   ✓ Origin: (" << origin_x << ", " << origin_y << ")" << std::endl;
                std::cout << "   ✓ Pixel size: " << georef.pixel_size_x 
                          << " x " << georef.pixel_size_y << std::endl;
                std::cout << "   ✓ Rotation: (" << georef.geotransform[2] 
                          << ", " << georef.geotransform[4] << ")" << std::endl;
                std::cout << "   ✓ Bounds: [" << georef.min_x << ", " << georef.min_y 
                          << "] to [" << georef.max_x << ", " << georef.max_y << "]" << std::endl;
            } else {
                std::cout << "   ⚠ No geotransform found" << std::endl;
            }
            
            // Get GCPs (Ground Control Points)
            std::cout << "\n📍 Ground Control Points:" << std::endl;
            georef.gcp_count = dataset->GetGCPCount();
            
            if (georef.gcp_count > 0) {
                georef.has_gcps = true;
                const GDAL_GCP* gcps = dataset->GetGCPs();
                
                std::cout << "   ✓ Found " << georef.gcp_count << " GCPs" << std::endl;
                
                // Convert to our TiePoint format (limit to first 10 for display)
                int display_count = std::min(georef.gcp_count, 10);
                for (int i = 0; i < georef.gcp_count; i++) {
                    TiePoint tp(
                        static_cast<uint32_t>(gcps[i].dfGCPPixel),
                        static_cast<uint32_t>(gcps[i].dfGCPLine),
                        gcps[i].dfGCPX,
                        gcps[i].dfGCPY
                    );
                    
                    if (gcps[i].dfGCPZ != 0.0) {
                        tp.z = gcps[i].dfGCPZ;
                    }
                    
                    georef.gcps.push_back(tp);
                    
                    if (i < display_count) {
                        std::cout << "      GCP " << (i+1) << ": Pixel(" 
                                  << tp.i << ", " << tp.j << ") -> Model(" 
                                  << tp.x << ", " << tp.y;
                        if (tp.z.has_value()) {
                            std::cout << ", " << *tp.z;
                        }
                        std::cout << ")" << std::endl;
                    }
                }
                
                if (georef.gcp_count > display_count) {
                    std::cout << "      ... and " << (georef.gcp_count - display_count) 
                              << " more GCPs" << std::endl;
                }
            } else {
                std::cout << "   ⚠ No GCPs found" << std::endl;
            }
            
            GDALClose(dataset);
            
            std::cout << "\n✓ Extraction complete" << std::endl;
            return true;
            
        } catch (const std::exception& e) {
            std::cerr << "✗ Exception: " << e.what() << std::endl;
            GDALClose(dataset);
            return false;
        }
    }
    
    // Convert to GeoHEIF metadata
    static GeoHEIFMetadata to_geoheif_metadata(const GeoTIFFGeoreference& georef) {
        std::cout << "\n=== Converting to GeoHEIF Metadata ===" << std::endl;
        
        GeoHEIFMetadata metadata;
        
        // Add CRS
        if (georef.has_crs) {
            if (georef.epsg_code > 0) {
                // Use EPSG code (CURIE format)
                std::cout << "   Using EPSG:" << georef.epsg_code << " (CURIE)" << std::endl;
                metadata.crs = CoordinateReferenceSystemProperty::from_epsg(georef.epsg_code);
            } else {
                // Use WKT
                std::cout << "   Using WKT definition" << std::endl;
                metadata.crs = CoordinateReferenceSystemProperty::from_wkt2(georef.crs_wkt);
            }
        } else {
            std::cout << "   ⚠ No CRS, using default WGS84" << std::endl;
            metadata.crs = CoordinateReferenceSystemProperty::from_epsg(4326);
        }
        
        // Add georeferencing
        if (georef.has_geotransform) {
            // Georectified (affine transform)
            std::cout << "   Creating affine transform (georectified)" << std::endl;
            
            ModelTransformationProperty transform;
            transform.is_2d = true;
            
            // Direct mapping from GeoTIFF geotransform
            transform.m00 = georef.geotransform[1];  // pixel width
            transform.m01 = georef.geotransform[2];  // rotation
            transform.m03 = georef.geotransform[0];  // origin X
            
            transform.m10 = georef.geotransform[4];  // rotation
            transform.m11 = georef.geotransform[5];  // pixel height (negative for north-up)
            transform.m13 = georef.geotransform[3];  // origin Y
            
            metadata.affine_transform = transform;
            
            std::cout << "      Matrix: [" << transform.m00 << ", " << transform.m01 
                      << ", " << transform.m03 << "]" << std::endl;
            std::cout << "              [" << transform.m10 << ", " << transform.m11 
                      << ", " << transform.m13 << "]" << std::endl;
            
        } else if (georef.has_gcps) {
            // Georeferenceable (tie points)
            std::cout << "   Creating tie points (georeferenceable)" << std::endl;
            
            ModelTiepointProperty tp_prop;
            tp_prop.is_2d = true;
            tp_prop.tie_points = georef.gcps;
            
            metadata.tie_points = tp_prop;
            
            std::cout << "      Added " << georef.gcps.size() << " tie points" << std::endl;
        } else {
            std::cout << "   ⚠ No georeferencing information" << std::endl;
        }
        
        // Validate
        if (metadata.is_valid()) {
            std::cout << "   ✓ Metadata valid" << std::endl;
        } else {
            std::cout << "   ✗ Metadata invalid: " << metadata.validation_error() << std::endl;
        }
        
        return metadata;
    }
};

} // namespace geoheif

#endif // HEIF_GEOTIFF_CRS_EXTRACTOR_H

### GeoTIFF to GeoHEIF Use Case

In [ ]:
#ifndef HEIF_GEOTIFF_TO_GEOHEIF_USE_CASE_H
#define HEIF_GEOTIFF_TO_GEOHEIF_USE_CASE_H

namespace geoheif {

// ============================================================================
// USE CASE: Convert GeoTIFF to GeoHEIF
// ============================================================================

class GeoTIFFToGeoHEIF {
public:

    // NEW: Unified method for adding GeoTIFF metadata to existing HEIF
    // This replaces the standalone extract_and_apply_geoheif_metadata
    static bool add_geotiff_metadata_to_heif(
        const char* geotiff_file,
        const char* heif_file,
        const char* output_geoheif = nullptr  // If nullptr, modifies heif_file in-place
    ) {
        std::cout << "\n" << std::string(80, '=') << std::endl;
        std::cout << "Adding GeoTIFF Metadata to HEIF" << std::endl;
        std::cout << std::string(80, '=') << std::endl;
        
        try {
            // Step 1: Extract georeference from GeoTIFF
            std::cout << "\n[1/4] Extracting georeference from GeoTIFF..." << std::endl;
            GeoTIFFGeoreference georef;
            if (!GeoTIFFExtractor::extract(geotiff_file, georef)) {
                return false;
            }
            
            // Step 2: Convert to GeoHEIF metadata
            std::cout << "\n[2/4] Converting to GeoHEIF metadata..." << std::endl;
            GeoHEIFMetadata metadata = GeoTIFFExtractor::to_geoheif_metadata(georef);
            
            // Validate
            if (!metadata.is_valid()) {
                std::cerr << "✗ Invalid metadata: " << metadata.validation_error() << std::endl;
                return false;
            }
            
            // Step 3: Load HEIF and add metadata
            std::cout << "\n[3/4] Adding metadata to HEIF..." << std::endl;
            HEIFFileModifier modifier;
            
            if (!modifier.load(heif_file)) {
                return false;
            }
            
            if (!modifier.add_ogeo_brand()) {
                return false;
            }
            
            if (!modifier.add_geoheif_properties(metadata)) {
                return false;
            }
            
            // Step 4: Save result
            std::cout << "\n[4/4] Saving GeoHEIF..." << std::endl;
            
            // Determine output path
            std::string final_output = output_geoheif ? output_geoheif : heif_file;
            
            // If in-place modification, use temp file
            if (!output_geoheif || std::string(output_geoheif) == std::string(heif_file)) {
                std::string temp_output = std::string(heif_file) + ".geoheif.tmp";
                
                if (!modifier.save(temp_output.c_str())) {
                    return false;
                }
                
                // Replace original
                std::remove(heif_file);
                std::rename(temp_output.c_str(), heif_file);
                
                std::cout << "  ✓ Modified in-place: " << heif_file << std::endl;
            } else {
                // Save to different file
                if (!modifier.save(output_geoheif)) {
                    return false;
                }
                
                std::cout << "  ✓ Saved to: " << output_geoheif << std::endl;
            }
            
            std::cout << "\n" << std::string(80, '=') << std::endl;
            std::cout << "✓ GeoHEIF metadata added successfully!" << std::endl;
            std::cout << "  GeoTIFF: " << geotiff_file << std::endl;
            std::cout << "  HEIF: " << heif_file << std::endl;
            std::cout << "  GeoHEIF: " << final_output << std::endl;
            std::cout << std::string(80, '=') << std::endl;
            
            return true;
            
        } catch (const std::exception& e) {
            std::cerr << "✗ Exception: " << e.what() << std::endl;
            return false;
        }
    }
       
    // Method 1: Extract from GeoTIFF, encode image, add metadata
    static bool convert_complete(
        const char* geotiff_file,
        const char* output_geoheif,
        ExtendedCompressionType compression = ExtendedCompressionType::HEVC,
        int quality = 70
    ) {
        std::cout << "\n" << std::string(80, '=') << std::endl;
        std::cout << "USE CASE: GeoTIFF → GeoHEIF (Complete Conversion)" << std::endl;
        std::cout << std::string(80, '=') << std::endl;
        
        try {
            // Step 1: Extract georeference from GeoTIFF
            std::cout << "\n[1/4] Extracting georeference from GeoTIFF..." << std::endl;
            GeoTIFFGeoreference georef;
            if (!GeoTIFFExtractor::extract(geotiff_file, georef)) {
                return false;
            }
            
            // Step 2: Convert to GeoHEIF metadata
            std::cout << "\n[2/4] Converting to GeoHEIF metadata..." << std::endl;
            GeoHEIFMetadata metadata = GeoTIFFExtractor::to_geoheif_metadata(georef);
            
            // Step 3: Encode image as HEIF
            std::cout << "\n[3/4] Encoding image as HEIF..." << std::endl;
            std::string temp_heif = std::string(output_geoheif) + ".temp.heif";
            encode_nontiled(geotiff_file, temp_heif.c_str(), compression, quality, 8);
            
            // Step 4: Add GeoHEIF metadata
            std::cout << "\n[4/4] Adding GeoHEIF metadata..." << std::endl;
            HEIFFileModifier modifier;
            
            if (!modifier.load(temp_heif)) {
                std::remove(temp_heif.c_str());
                return false;
            }
            
            if (!modifier.add_ogeo_brand()) {
                std::remove(temp_heif.c_str());
                return false;
            }
            
            if (!modifier.add_geoheif_properties(metadata)) {
                std::remove(temp_heif.c_str());
                return false;
            }
            
            if (!modifier.save(output_geoheif)) {
                std::remove(temp_heif.c_str());
                return false;
            }
            
            // Cleanup
            std::remove(temp_heif.c_str());
            
            std::cout << "\n" << std::string(80, '=') << std::endl;
            std::cout << "✓ Conversion complete!" << std::endl;
            std::cout << "  Input:  " << geotiff_file << std::endl;
            std::cout << "  Output: " << output_geoheif << std::endl;
            std::cout << std::string(80, '=') << std::endl;
            
            return true;
            
        } catch (const std::exception& e) {
            std::cerr << "✗ Exception: " << e.what() << std::endl;
            return false;
        }
    }
    
    // Method 2: Extract from GeoTIFF, add to existing HEIF
        // UPDATED: Refactored to use the new unified method
    static bool add_to_existing_heif(
        const char* geotiff_file,
        const char* input_heif,
        const char* output_geoheif
    ) {
        return add_geotiff_metadata_to_heif(geotiff_file, input_heif, output_geoheif);
    }
    
    // Method 3: Print GeoTIFF info only (no conversion)
    static bool print_geotiff_info(const char* geotiff_file) {
        std::cout << "\n" << std::string(80, '=') << std::endl;
        std::cout << "GeoTIFF Information Report" << std::endl;
        std::cout << std::string(80, '=') << std::endl;
        
        GeoTIFFGeoreference georef;
        if (!GeoTIFFExtractor::extract(geotiff_file, georef)) {
            return false;
        }
        
        // Print summary
        std::cout << "\n📋 Summary:" << std::endl;
        std::cout << "   Image: " << georef.width << "x" << georef.height 
                  << " (" << georef.bands << " bands)" << std::endl;
        std::cout << "   CRS: " << (georef.has_crs ? "Yes" : "No");
        if (georef.epsg_code > 0) {
            std::cout << " (EPSG:" << georef.epsg_code << ")";
        }
        std::cout << std::endl;
        std::cout << "   Geotransform: " << (georef.has_geotransform ? "Yes" : "No") << std::endl;
        std::cout << "   GCPs: " << (georef.has_gcps ? "Yes" : "No");
        if (georef.has_gcps) {
            std::cout << " (" << georef.gcp_count << " points)";
        }
        std::cout << std::endl;
        
        // Determine type
        std::cout << "\n🏷️  Georeference Type: ";
        if (georef.has_geotransform) {
            std::cout << "Georectified (affine transform)" << std::endl;
        } else if (georef.has_gcps) {
            std::cout << "Georeferenceable (GCPs only)" << std::endl;
        } else {
            std::cout << "None (plain image)" << std::endl;
        }
        
        std::cout << "\n" << std::string(80, '=') << std::endl;
        
        return true;
    }
    
    // Method 4: Compare GeoTIFF and GeoHEIF metadata
    static bool compare_metadata(
        const char* geotiff_file,
        const char* geoheif_file
    ) {
        std::cout << "\n" << std::string(80, '=') << std::endl;
        std::cout << "Metadata Comparison: GeoTIFF vs GeoHEIF" << std::endl;
        std::cout << std::string(80, '=') << std::endl;
        
        // Extract from GeoTIFF
        GeoTIFFGeoreference georef;
        if (!GeoTIFFExtractor::extract(geotiff_file, georef)) {
            return false;
        }
        
        // Load GeoHEIF
        HEIFFileModifier modifier;
        if (!modifier.load(geoheif_file)) {
            return false;
        }
        
        std::cout << "\n📊 Comparison:" << std::endl;
        std::cout << "\n  GeoTIFF:" << std::endl;
        std::cout << "    - Dimensions: " << georef.width << "x" << georef.height << std::endl;
        std::cout << "    - CRS: " << (georef.has_crs ? "Present" : "Missing") << std::endl;
        std::cout << "    - Georeference: " << (georef.has_geotransform ? "Affine" : 
                                                  georef.has_gcps ? "GCPs" : "None") << std::endl;
        
        std::cout << "\n  GeoHEIF:" << std::endl;
        modifier.print_structure();
        
        return true;
    }
};

} // namespace geoheif

#endif // HEIF_GEOTIFF_TO_GEOHEIF_USE_CASE_H

### GeoTIFF to GeoHEIF Usage Examples

In [ ]:
// ============================================================================
// USE CASE EXAMPLES
// ============================================================================

// Use Case 1: Print GeoTIFF Information
// {
//     const char* input_file = "srcdata/ACT2017.tiff"; 
//     std::cout << "\n\n";
//     std::cout << "╔════════════════════════════════════════════════════════════════╗" << std::endl;
//     std::cout << "║  USE CASE 1: Inspect GeoTIFF Metadata                         ║" << std::endl;
//     std::cout << "╚════════════════════════════════════════════════════════════════╝" << std::endl;
    
//     geoheif::GeoTIFFToGeoHEIF::print_geotiff_info(input_file);
// }

// Use Case 2: Complete Conversion (GeoTIFF → GeoHEIF)
// {
//     const char* input_file = "srcdata/ACT2017.tiff";
//     const char* output_dir = "output";
//     std::cout << "\n\n";
//     std::cout << "╔════════════════════════════════════════════════════════════════╗" << std::endl;
//     std::cout << "║  USE CASE 2: Complete Conversion (GeoTIFF → GeoHEIF)          ║" << std::endl;
//     std::cout << "╚════════════════════════════════════════════════════════════════╝" << std::endl;
    
//     geoheif::GeoTIFFToGeoHEIF::convert_complete(
//         input_file,
//         (fs::path(output_dir) / "output_converted_complete.heif").string().c_str(),
//         ExtendedCompressionType::HEVC,
//         70
//     );
// }

// Use Case 3: Extract CRS from GeoTIFF, Add to Existing HEIF
{
    const char* input_file = "srcdata/ACT2017.tiff";
    const char* output_dir = "output";    std::cout << "\n\n";
    std::string plain_heif = (fs::path(output_dir) / "output_tiled_pyramid_5levels_tili_256_deflate.heif").string(); 
    std::cout << "╔════════════════════════════════════════════════════════════════╗" << std::endl;
    std::cout << "║  USE CASE 3: Extract CRS → Add to Existing HEIF               ║" << std::endl;
    std::cout << "╚════════════════════════════════════════════════════════════════╝" << std::endl;
    
    // First, create a plain HEIF
    // std::string plain_heif = (fs::path(output_dir) / "plain_image.heif").string();
    // std::cout << "\nCreating plain HEIF for testing..." << std::endl;
    // encode_nontiled(input_file, plain_heif.c_str(), ExtendedCompressionType::HEVC, 70);
    
    // Now add GeoTIFF metadata to it
    geoheif::GeoTIFFToGeoHEIF::add_to_existing_heif(
        input_file,  // Source GeoTIFF (for CRS extraction)
        plain_heif.c_str(),  // Existing HEIF
        (fs::path(output_dir) / "output_heif_with_crs.heif").string().c_str()  // Output GeoHEIF
    );
}

// // Use Case 4: Compare Original and Converted
// {
//     std::cout << "\n\n";
//     std::cout << "╔════════════════════════════════════════════════════════════════╗" << std::endl;
//     std::cout << "║  USE CASE 4: Compare GeoTIFF vs GeoHEIF Metadata              ║" << std::endl;
//     std::cout << "╚════════════════════════════════════════════════════════════════╝" << std::endl;
    
//     geoheif::GeoTIFFToGeoHEIF::compare_metadata(
//         input_file,  // Original GeoTIFF
//         (fs::path(output_dir) / "output_converted_complete.heif").string().c_str()  // GeoHEIF
//     );
// }

### Advanced Usage Case - Batch Processing

In [ ]:
#ifndef HEIF_BATCH_GEOTIFF_TO_GEOHEIF_H
#define HEIF_BATCH_GEOTIFF_TO_GEOHEIF_H

namespace geoheif {

// Batch process multiple GeoTIFFs
class BatchGeoTIFFProcessor {
public:
    struct ConversionResult {
        std::string input_file;
        std::string output_file;
        bool success;
        std::string error_message;
        double processing_time_seconds;
        
        // Source info
        int width, height, bands;
        bool had_crs;
        int epsg_code;
        double file_size_mb;
    };
    
    static std::vector<ConversionResult> convert_directory(
        const char* input_dir,
        const char* output_dir,
        const char* pattern = "*.tif",
        ExtendedCompressionType compression = ExtendedCompressionType::HEVC,
        int quality = 70
    ) {
        std::cout << "\n" << std::string(80, '=') << std::endl;
        std::cout << "Batch Processing: GeoTIFF Directory → GeoHEIF" << std::endl;
        std::cout << std::string(80, '=') << std::endl;
        
        std::vector<ConversionResult> results;
        
        try {
            // Find all matching files
            std::vector<fs::path> geotiff_files;
            for (const auto& entry : fs::directory_iterator(input_dir)) {
                if (entry.is_regular_file()) {
                    std::string ext = entry.path().extension().string();
                    if (ext == ".tif" || ext == ".tiff" || ext == ".TIF" || ext == ".TIFF") {
                        geotiff_files.push_back(entry.path());
                    }
                }
            }
            
            std::cout << "\nFound " << geotiff_files.size() << " GeoTIFF files" << std::endl;
            
            // Process each file
            int success_count = 0;
            for (size_t i = 0; i < geotiff_files.size(); i++) {
                std::cout << "\n[" << (i+1) << "/" << geotiff_files.size() << "] Processing: " 
                          << geotiff_files[i].filename() << std::endl;
                
                ConversionResult result;
                result.input_file = geotiff_files[i].string();
                
                // Generate output filename
                fs::path output_path = fs::path(output_dir) / geotiff_files[i].stem();
                output_path.replace_extension(".heif");
                result.output_file = output_path.string();
                
                // Get file size
                result.file_size_mb = fs::file_size(geotiff_files[i]) / (1024.0 * 1024.0);
                
                auto start_time = std::chrono::high_resolution_clock::now();
                
                // Extract info
                GeoTIFFGeoreference georef;
                if (GeoTIFFExtractor::extract(result.input_file.c_str(), georef)) {
                    result.width = georef.width;
                    result.height = georef.height;
                    result.bands = georef.bands;
                    result.had_crs = georef.has_crs;
                    result.epsg_code = georef.epsg_code;
                    
                    // Convert
                    result.success = GeoTIFFToGeoHEIF::convert_complete(
                        result.input_file.c_str(),
                        result.output_file.c_str(),
                        compression,
                        quality
                    );
                } else {
                    result.success = false;
                    result.error_message = "Failed to extract georeference";
                }
                
                auto end_time = std::chrono::high_resolution_clock::now();
                result.processing_time_seconds = 
                    std::chrono::duration<double>(end_time - start_time).count();
                
                if (result.success) {
                    success_count++;
                    std::cout << "  ✓ Success (" << result.processing_time_seconds << "s)" << std::endl;
                } else {
                    std::cout << "  ✗ Failed: " << result.error_message << std::endl;
                }
                
                results.push_back(result);
            }
            
            // Print summary
            std::cout << "\n" << std::string(80, '=') << std::endl;
            std::cout << "Batch Processing Complete" << std::endl;
            std::cout << "  Total: " << results.size() << std::endl;
            std::cout << "  Success: " << success_count << std::endl;
            std::cout << "  Failed: " << (results.size() - success_count) << std::endl;
            std::cout << std::string(80, '=') << std::endl;
            
        } catch (const std::exception& e) {
            std::cerr << "Batch processing error: " << e.what() << std::endl;
        }
        
        return results;
    }
    
    // Print batch results
    static void print_results(const std::vector<ConversionResult>& results) {
        std::cout << "\n📊 Batch Processing Results:" << std::endl;
        std::cout << std::string(120, '-') << std::endl;
        std::cout << std::left << std::setw(30) << "File" 
                  << std::setw(12) << "Size (MB)"
                  << std::setw(15) << "Dimensions"
                  << std::setw(10) << "EPSG"
                  << std::setw(12) << "Time (s)"
                  << "Status" << std::endl;
        std::cout << std::string(120, '-') << std::endl;
        
        for (const auto& r : results) {
            fs::path p(r.input_file);
            std::cout << std::left << std::setw(30) << p.filename().string()
                      << std::setw(12) << std::fixed << std::setprecision(2) << r.file_size_mb
                      << std::setw(15) << (std::to_string(r.width) + "x" + std::to_string(r.height))
                      << std::setw(10) << (r.epsg_code > 0 ? std::to_string(r.epsg_code) : "-")
                      << std::setw(12) << std::fixed << std::setprecision(1) << r.processing_time_seconds
                      << (r.success ? "✓" : "✗ " + r.error_message) << std::endl;
        }
        std::cout << std::string(120, '-') << std::endl;
    }
};

} // namespace geoheif

#endif // HEIF_BATCH_GEOTIFF_TO_GEOHEIF_H

### GeoTIFF to GeoHEIF Batch Processing Example

In [ ]:
// // Use Case 5: Batch Process Directory
// {
//     std::cout << "\n\n";
//     std::cout << "╔════════════════════════════════════════════════════════════════╗" << std::endl;
//     std::cout << "║  USE CASE 5: Batch Process GeoTIFF Directory                  ║" << std::endl;
//     std::cout << "╚════════════════════════════════════════════════════════════════╝" << std::endl;
    
//     // Process all GeoTIFFs in a directory
//     auto results = geoheif::BatchGeoTIFFProcessor::convert_directory(
//         "srcdata",           // Input directory with GeoTIFFs
//         output_dir,          // Output directory for GeoHEIFs
//         "*.tif",             // Pattern
//         ExtendedCompressionType::HEVC,
//         70
//     );
    
//     // Print results table
//     geoheif::BatchGeoTIFFProcessor::print_results(results);
// }

# Work Point before inspection

## GeoHEIF Inspection

### GeoHEIF Inspector Core

In [ ]:
#ifndef HEIF_GEOHEIF_INSPECTOR_H
#define HEIF_GEOHEIF_INSPECTOR_H

namespace geoheif {

// ============================================================================
// GeoHEIF Inspector - Read and Extract Metadata
// ============================================================================

// Use standard integer types instead of libheif types
using item_id_type = uint32_t;

// Define PyramidLevelInfo before GeoHEIFInfo
struct PyramidLevelInfo {
    int level_index;           // 0 = full resolution, 1 = first overview, etc.
    item_id_type item_id;      // HEIF item ID for this level
    
    // Dimensions
    int width;                 // Actual image width
    int height;                // Actual image height
    int padded_width;          // Width after padding (if tiled)
    int padded_height;         // Height after padding (if tiled)
    
    // Tiling info (if applicable)
    bool is_tiled;
    std::string tiling_scheme; // "grid", "unci", "tili", or "none"
    int tile_width;
    int tile_height;
    int num_tile_cols;
    int num_tile_rows;
    
    // Geospatial extent (computed from affine transform + dimensions)
    bool has_bbox;
    double min_x, min_y, max_x, max_y;
    double pixel_size_x, pixel_size_y;
    
    // Compression info
    std::string compression_type;
    
    PyramidLevelInfo()
        : level_index(0), item_id(0)
        , width(0), height(0)
        , padded_width(0), padded_height(0)
        , is_tiled(false)
        , tiling_scheme("none")
        , tile_width(0), tile_height(0)
        , num_tile_cols(0), num_tile_rows(0)
        , has_bbox(false)
        , min_x(0), min_y(0), max_x(0), max_y(0)
        , pixel_size_x(0), pixel_size_y(0) {}
};

struct GeoHEIFInfo {
    // File information
    std::string filename;
    size_t file_size;
    
    // HEIF brands
    std::vector<std::string> compatible_brands;
    std::string major_brand;
    bool has_ogeo_brand;
    
    // Image properties
    int width;
    int height;
    int bit_depth;
    
    // GeoHEIF properties found
    bool has_mcrs;  // CRS
    bool has_mtxf;  // Affine transform
    bool has_tiep;  // Tie points
    bool has_edim;  // Extra dimensions
    bool has_edvl;  // Dimension values
    bool has_pcel;  // Cell properties
    bool has_pcat;  // Categories
    
    // Extracted CRS information
    struct CRSInfo {
        bool present;
        uint32_t encoding;  // crsu, curi, wkt2
        std::string encoding_name;
        std::string crs_string;
        std::optional<float> epoch;
        
        // Parsed information
        int epsg_code;  // -1 if not EPSG
        std::string crs_name;
        
        CRSInfo() : present(false), encoding(0), epsg_code(-1) {}
    } crs;
    
    // Extracted georeference
    struct GeoreferenceInfo {
        bool is_georectified;  // Has affine transform
        bool is_georeferenceable;  // Has tie points
        
        // Affine transform (if georectified)
        bool is_2d;
        double m00, m01, m03;  // First row
        double m10, m11, m13;  // Second row
        double m02, m12;       // 3D additional
        double m20, m21, m22, m23;  // Third row for 3D
        
        // Tie points (if georeferenceable)
        std::vector<TiePoint> tie_points;
        
        // Computed bounding box
        bool has_bbox;
        double min_x, min_y, max_x, max_y;
        double pixel_size_x, pixel_size_y;
        
        GeoreferenceInfo() 
            : is_georectified(false), is_georeferenceable(false)
            , is_2d(true)
            , m00(1), m01(0), m03(0)
            , m10(0), m11(1), m13(0)
            , m02(0), m12(0)
            , m20(0), m21(0), m22(1), m23(0)
            , has_bbox(false)
            , min_x(0), min_y(0), max_x(0), max_y(0)
            , pixel_size_x(0), pixel_size_y(0) {}
    } georeference;
    
    // Extra dimensions (if datacube)
    struct DimensionInfo {
        std::string name;
        std::string definition;
        std::string type_name;  // "regular", "irregular", "categorical"
        
        // For regular
        double minimum, maximum, resolution;
        
        // For irregular/categorical
        std::vector<std::string> values;
        
        std::string unit;
    };
    std::vector<DimensionInfo> extra_dimensions;
    
    bool is_pyramid;                                  // Does this file contain a pyramid?
    std::vector<PyramidLevelInfo> pyramid_levels;     // Information for each level

    // Constructor
    GeoHEIFInfo() 
        : file_size(0)
        , has_ogeo_brand(false)
        , is_pyramid(false)
        , width(0), height(0), bit_depth(8)
        , has_mcrs(false), has_mtxf(false), has_tiep(false)
        , has_edim(false), has_edvl(false), has_pcel(false), has_pcat(false) {}

    // Helper: Get info for specific pyramid level
    const PyramidLevelInfo* get_level(int level_index) const {
        for (const auto& level : pyramid_levels) {
            if (level.level_index == level_index) {
                return &level;
            }
        }
        return nullptr;
    }
    
    // Helper: Get full resolution level (level 0)
    const PyramidLevelInfo* get_full_resolution() const {
        return get_level(0);
    }
    
    // Helper: Get closest level for a given scale
    const PyramidLevelInfo* get_level_for_scale(double target_pixel_size) const {
        if (pyramid_levels.empty()) return nullptr;
        
        const PyramidLevelInfo* best = &pyramid_levels[0];
        double best_diff = std::abs(best->pixel_size_x - target_pixel_size);
        
        for (const auto& level : pyramid_levels) {
            double diff = std::abs(level.pixel_size_x - target_pixel_size);
            if (diff < best_diff) {
                best = &level;
                best_diff = diff;
            }
        }
        
        return best;
    }


};


class GeoHEIFInspector {
private:
    struct BoxLocation {
        size_t offset;
        size_t size;
        std::string type;
        size_t header_size;
        item_id_type associated_item_id;
        
        BoxLocation() : offset(0), size(0), header_size(8), associated_item_id(0) {}
        BoxLocation(size_t off, size_t sz, const std::string& t, size_t hs = 8)
            : offset(off), size(sz), type(t), header_size(hs), associated_item_id(0) {}
    };

    std::vector<uint8_t> file_data;
    std::vector<BoxLocation> boxes;

    // Map item IDs to their properties
    struct ItemInfo {
        item_id_type item_id;
        int width;
        int height;
        bool is_primary;
        std::string item_type;  // "grid", "unci", "tili", or codec type
        
        ItemInfo() : item_id(0), width(0), height(0), is_primary(false) {}
    };
    std::map<item_id_type, ItemInfo> item_registry;

    
    // Safe memory readers
    uint32_t read_uint32(size_t offset) const {
        if (offset + 4 > file_data.size()) return 0;
        return (static_cast<uint32_t>(file_data[offset]) << 24) |
               (static_cast<uint32_t>(file_data[offset + 1]) << 16) |
               (static_cast<uint32_t>(file_data[offset + 2]) << 8) |
               static_cast<uint32_t>(file_data[offset + 3]);
    }
    
    uint16_t read_uint16(size_t offset) const {
        if (offset + 2 > file_data.size()) return 0;
        return (static_cast<uint16_t>(file_data[offset]) << 8) |
               static_cast<uint16_t>(file_data[offset + 1]);
    }
    
    uint8_t read_uint8(size_t offset) const {
        if (offset >= file_data.size()) return 0;
        return file_data[offset];
    }
    
    double read_float64(size_t offset) const {
        if (offset + 8 > file_data.size()) return 0.0;
        uint64_t bits = (static_cast<uint64_t>(read_uint32(offset)) << 32) |
                        static_cast<uint64_t>(read_uint32(offset + 4));
        double value;
        std::memcpy(&value, &bits, sizeof(double));
        return value;
    }
    
    float read_float32(size_t offset) const {
        if (offset + 4 > file_data.size()) return 0.0f;
        uint32_t bits = read_uint32(offset);
        float value;
        std::memcpy(&value, &bits, sizeof(float));
        return value;
    }
    
    std::string read_utf8string(size_t offset, size_t max_length = 1024) const {
        std::string result;
        for (size_t i = 0; i < max_length && (offset + i) < file_data.size(); i++) {
            uint8_t ch = file_data[offset + i];
            if (ch == 0) break;
            result += static_cast<char>(ch);
        }
        return result;
    }
    
    std::string fourcc_to_string(uint32_t fourcc) const {
        char str[5];
        str[0] = (fourcc >> 24) & 0xFF;
        str[1] = (fourcc >> 16) & 0xFF;
        str[2] = (fourcc >> 8) & 0xFF;
        str[3] = fourcc & 0xFF;
        str[4] = '\0';
        return std::string(str);
    }
        // Parse boxes recursively
    bool parse_boxes_recursive(size_t offset, size_t end_offset, int depth = 0) {
        const int MAX_DEPTH = 20;
        
        if (depth > MAX_DEPTH) return false;
        
        while (offset < end_offset && offset < file_data.size()) {
            if (offset + 8 > file_data.size() || offset + 8 > end_offset) break;
            
            uint32_t size = read_uint32(offset);
            char type[5] = {0};
            std::memcpy(type, &file_data[offset + 4], 4);
            
            // Validate type
            bool valid_type = true;
            for (int i = 0; i < 4; i++) {
                if (!std::isprint(static_cast<unsigned char>(type[i]))) {
                    valid_type = false;
                    break;
                }
            }
            if (!valid_type) break;
            
            size_t box_size = size;
            size_t header_size = 8;
            
            if (size == 1) {
                if (offset + 16 > file_data.size()) break;
                uint64_t size64 = (static_cast<uint64_t>(read_uint32(offset + 8)) << 32) |
                                  static_cast<uint64_t>(read_uint32(offset + 12));
                box_size = static_cast<size_t>(size64);
                header_size = 16;
            } else if (size == 0) {
                box_size = file_data.size() - offset;
            }
            
            if (box_size < header_size || offset + box_size > file_data.size()) break;
            
            BoxLocation loc(offset, box_size, std::string(type), header_size);
            boxes.push_back(loc);
            
            // Parse container boxes recursively
            std::string type_str(type);
            static const std::vector<std::string> container_boxes = {
                "meta", "iprp", "ipco", "iinf", "dinf", "grpl", "moov", "iloc"
            };
            
            bool is_container = std::find(container_boxes.begin(), 
                                         container_boxes.end(), 
                                         type_str) != container_boxes.end();
            
            if (is_container) {
                size_t content_start = offset + header_size;
                if (type_str == "meta") content_start += 4; // Skip version/flags
                
                if (content_start < offset + box_size) {
                    parse_boxes_recursive(content_start, offset + box_size, depth + 1);
                }
            }
            
            offset += box_size;
        }
        
        return true;
    }
    
    // Find box by type
    const BoxLocation* find_box(const std::string& box_type) const {
        for (const auto& box : boxes) {
            if (box.type == box_type) return &box;
        }
        return nullptr;
    }
    
    // Find all boxes of a type
    std::vector<const BoxLocation*> find_all_boxes(const std::string& box_type) const {
        std::vector<const BoxLocation*> results;
        for (const auto& box : boxes) {
            if (box.type == box_type) results.push_back(&box);
        }
        return results;
    }

    // Parse ftyp box to extract brands
    bool parse_ftyp(GeoHEIFInfo& info, size_t offset, size_t size) {
        // ftyp: size(4) type(4) major_brand(4) minor_version(4) compatible_brands...
        if (offset + 16 > file_data.size()) return false;
        
        size_t pos = offset + 8;  // Skip size and type
        
        // Read major brand
        info.major_brand = fourcc_to_string(read_uint32(pos));
        pos += 4;
        
        // Skip minor version
        pos += 4;
        
        // Read compatible brands
        while (pos + 4 <= offset + size) {
            std::string brand = fourcc_to_string(read_uint32(pos));
            info.compatible_brands.push_back(brand);
            
            if (brand == "ogeo") {
                info.has_ogeo_brand = true;
            }
            
            pos += 4;
        }
        
        return true;
    }
    
    // Parse mcrs box (CRS)
    bool parse_mcrs(GeoHEIFInfo& info, size_t offset, size_t size) {
        // mcrs: size(4) type(4) version(1) flags(3) crsEncoding(4) crs(string) [epoch(4)]
        if (offset + 16 > file_data.size()) return false;
        
        info.has_mcrs = true;
        info.crs.present = true;
        
        size_t pos = offset + 8;  // Skip size and type
        
        // Read version and flags
        uint8_t version = read_uint8(pos++);
        uint32_t flags = (read_uint8(pos) << 16) | (read_uint8(pos+1) << 8) | read_uint8(pos+2);
        pos += 3;
        
        // Read CRS encoding
        info.crs.encoding = read_uint32(pos);
        pos += 4;
        
        // Decode encoding type
        std::string enc = fourcc_to_string(info.crs.encoding);
        info.crs.encoding_name = enc;
        
        // Read CRS string
        info.crs.crs_string = read_utf8string(pos);
        pos += info.crs.crs_string.length() + 1;  // +1 for null terminator
        
        // Read epoch if present (flags & 1)
        if (flags & 1) {
            info.crs.epoch = read_float32(pos);
        }
        
        // Try to extract EPSG code
        if (enc == "curi" || enc == "crsu") {
            // Parse EPSG code from CURIE or URI
            size_t epsg_pos = info.crs.crs_string.find("EPSG");
            if (epsg_pos != std::string::npos) {
                size_t colon_pos = info.crs.crs_string.find(':', epsg_pos);
                if (colon_pos != std::string::npos) {
                    std::string code_str = info.crs.crs_string.substr(colon_pos + 1);
                    // Remove trailing ']' if CURIE
                    size_t bracket_pos = code_str.find(']');
                    if (bracket_pos != std::string::npos) {
                        code_str = code_str.substr(0, bracket_pos);
                    }
                    try {
                        info.crs.epsg_code = std::stoi(code_str);
                    } catch (...) {
                        info.crs.epsg_code = -1;
                    }
                }
            }
        }
        
        return true;
    }
    
    // Parse mtxf box (affine transform)
    bool parse_mtxf(GeoHEIFInfo& info, size_t offset, size_t size) {
        // mtxf: size(4) type(4) version(1) flags(3) matrix values...
        if (offset + 12 > file_data.size()) return false;
        
        info.has_mtxf = true;
        info.georeference.is_georectified = true;
        
        size_t pos = offset + 8;  // Skip size and type
        
        // Read version and flags
        uint8_t version = read_uint8(pos++);
        uint32_t flags = (read_uint8(pos) << 16) | (read_uint8(pos+1) << 8) | read_uint8(pos+2);
        pos += 3;
        
        // Check if 2D (flag bit 0 set)
        bool is_2d = (flags & 0x01) != 0;
        info.georeference.is_2d = is_2d;
        
        if (is_2d) {
            // Read 2D matrix (6 doubles)
            info.georeference.m00 = read_float64(pos); pos += 8;
            info.georeference.m01 = read_float64(pos); pos += 8;
            info.georeference.m03 = read_float64(pos); pos += 8;
            info.georeference.m10 = read_float64(pos); pos += 8;
            info.georeference.m11 = read_float64(pos); pos += 8;
            info.georeference.m13 = read_float64(pos); pos += 8;
        } else {
            // Read 3D matrix (12 doubles)
            info.georeference.m00 = read_float64(pos); pos += 8;
            info.georeference.m01 = read_float64(pos); pos += 8;
            info.georeference.m02 = read_float64(pos); pos += 8;
            info.georeference.m03 = read_float64(pos); pos += 8;
            info.georeference.m10 = read_float64(pos); pos += 8;
            info.georeference.m11 = read_float64(pos); pos += 8;
            info.georeference.m12 = read_float64(pos); pos += 8;
            info.georeference.m13 = read_float64(pos); pos += 8;
            info.georeference.m20 = read_float64(pos); pos += 8;
            info.georeference.m21 = read_float64(pos); pos += 8;
            info.georeference.m22 = read_float64(pos); pos += 8;
            info.georeference.m23 = read_float64(pos); pos += 8;
        }
        
        return true;
    }
    
    // Parse tiep box (tie points)
    bool parse_tiep(GeoHEIFInfo& info, size_t offset, size_t size) {
        // tiep: size(4) type(4) version(1) flags(3) count(2) [i j x y [z]]...
        if (offset + 14 > file_data.size()) return false;
        
        info.has_tiep = true;
        info.georeference.is_georeferenceable = true;
        
        size_t pos = offset + 8;  // Skip size and type
        
        // Read version and flags
        uint8_t version = read_uint8(pos++);
        uint32_t flags = (read_uint8(pos) << 16) | (read_uint8(pos+1) << 8) | read_uint8(pos+2);
        pos += 3;
        
        // Check if 2D (flag bit 0 set)
        bool is_2d = (flags & 0x01) != 0;
        info.georeference.is_2d = is_2d;
        
        // Read tie point count
        uint16_t count = read_uint16(pos);
        pos += 2;
        
        // Read each tie point
        for (int i = 0; i < count && pos < offset + size; i++) {
            uint32_t pi = read_uint32(pos); pos += 4;
            uint32_t pj = read_uint32(pos); pos += 4;
            double x = read_float64(pos); pos += 8;
            double y = read_float64(pos); pos += 8;
            
            TiePoint tp(pi, pj, x, y);
            
            if (!is_2d && pos + 8 <= offset + size) {
                tp.z = read_float64(pos);
                pos += 8;
            }
            
            info.georeference.tie_points.push_back(tp);
        }
        
        return true;
    }
    
    // Parse ispe box (Image Spatial Extents)
    bool parse_ispe(GeoHEIFInfo& info, size_t offset, size_t size) {
        // ispe: size(4) type(4) version(1) flags(3) image_width(4) image_height(4)
        if (offset + 16 > file_data.size()) return false;
        
        size_t pos = offset + 8;  // Skip size and type
        
        // Read version and flags (FullBox)
        uint8_t version = read_uint8(pos++);
        uint32_t flags = (read_uint8(pos) << 16) | (read_uint8(pos+1) << 8) | read_uint8(pos+2);
        pos += 3;
        
        // Read image dimensions
        info.width = read_uint32(pos); pos += 4;
        info.height = read_uint32(pos); pos += 4;
        
        std::cout << "   ✓ ispe (Image Spatial Extents): " 
                << info.width << "x" << info.height << std::endl;
        
        return true;
    }

    // Compute bounding box from georeference and image dimensions
    void compute_bounding_box(GeoHEIFInfo& info) {
        if (info.width == 0 || info.height == 0) {
            return;
        }
        
        if (info.georeference.is_georectified) {
            // Use affine transform to compute corners
            auto& g = info.georeference;
            
            // Compute pixel size
            g.pixel_size_x = std::abs(g.m00);  // Assuming no rotation
            g.pixel_size_y = std::abs(g.m11);
            
            // Corners in pixel space: (0,0), (width, 0), (0, height), (width, height)
            std::vector<std::pair<double, double>> model_corners;
            
            // Transform each corner
            for (int i = 0; i <= info.width; i += info.width) {
                for (int j = 0; j <= info.height; j += info.height) {
                    double x = g.m00 * i + g.m01 * j + g.m03;
                    double y = g.m10 * i + g.m11 * j + g.m13;
                    model_corners.push_back({x, y});
                }
            }
            
            // Find min/max
            g.min_x = model_corners[0].first;
            g.max_x = model_corners[0].first;
            g.min_y = model_corners[0].second;
            g.max_y = model_corners[0].second;
            
            for (const auto& corner : model_corners) {
                g.min_x = std::min(g.min_x, corner.first);
                g.max_x = std::max(g.max_x, corner.first);
                g.min_y = std::min(g.min_y, corner.second);
                g.max_y = std::max(g.max_y, corner.second);
            }
            
            g.has_bbox = true;
            
        } else if (info.georeference.is_georeferenceable && !info.georeference.tie_points.empty()) {
            // Use tie points to estimate bounds
            auto& g = info.georeference;
            
            g.min_x = info.georeference.tie_points[0].x;
            g.max_x = info.georeference.tie_points[0].x;
            g.min_y = info.georeference.tie_points[0].y;
            g.max_y = info.georeference.tie_points[0].y;
            
            for (const auto& tp : info.georeference.tie_points) {
                g.min_x = std::min(g.min_x, tp.x);
                g.max_x = std::max(g.max_x, tp.x);
                g.min_y = std::min(g.min_y, tp.y);
                g.max_y = std::max(g.max_y, tp.y);
            }
            
            g.has_bbox = true;
            
            // Estimate pixel size
            if (info.georeference.tie_points.size() >= 2) {
                // Very rough estimate from first two points
                double dx = std::abs(info.georeference.tie_points[1].x - info.georeference.tie_points[0].x);
                double dy = std::abs(info.georeference.tie_points[1].y - info.georeference.tie_points[0].y);
                int di = std::abs(static_cast<int>(info.georeference.tie_points[1].i) - 
                                 static_cast<int>(info.georeference.tie_points[0].i));
                int dj = std::abs(static_cast<int>(info.georeference.tie_points[1].j) - 
                                 static_cast<int>(info.georeference.tie_points[0].j));
                
                if (di > 0) g.pixel_size_x = dx / di;
                if (dj > 0) g.pixel_size_y = dy / dj;
            }
        }
    }

//-------function-for-pyramid-levels-------
    // Parse iinf (Item Information)
    void parse_item_information() {
        std::cout << "\n📋 Parsing Item Information..." << std::endl;
        
        auto* iinf = find_box("iinf");
        if (!iinf) {
            std::cout << "   No iinf box found" << std::endl;
            return;
        }
        
        size_t pos = iinf->offset + 8;
        
        uint8_t version = read_uint8(pos++);
        uint32_t flags = (read_uint8(pos) << 16) | (read_uint8(pos+1) << 8) | read_uint8(pos+2);
        pos += 3;
        
        uint32_t entry_count;
        if (version == 0) {
            entry_count = read_uint16(pos);
            pos += 2;
        } else {
            entry_count = read_uint32(pos);
            pos += 4;
        }
        
        std::cout << "   Found " << entry_count << " items" << std::endl;
        
        size_t end_pos = iinf->offset + iinf->size;
        
        while (pos < end_pos && pos + 8 < file_data.size()) {
            uint32_t infe_size = read_uint32(pos);
            std::string infe_type = fourcc_to_string(read_uint32(pos + 4));
            
            if (infe_type != "infe" || infe_size < 8) break;
            if (pos + infe_size > end_pos) break;
            
            parse_infe_box(pos, infe_size);
            
            pos += infe_size;
        }
    }
    
    // Parse infe (Item Info Entry)
    void parse_infe_box(size_t offset, size_t size) {
        size_t pos = offset + 8;
        
        if (pos + 4 > file_data.size()) return;
        
        uint8_t version = read_uint8(pos++);
        uint32_t flags = (read_uint8(pos) << 16) | (read_uint8(pos+1) << 8) | read_uint8(pos+2);
        pos += 3;
        
        ItemInfo info;
        
        if (version <= 1) {
            info.item_id = read_uint16(pos);
            pos += 2;
        } else {
            info.item_id = read_uint32(pos);
            pos += 4;
        }
        
        pos += 2;  // Skip item_protection_index
        
        if (pos + 4 <= offset + size) {
            info.item_type = fourcc_to_string(read_uint32(pos));
            pos += 4;
        }
        
        std::string item_name = read_utf8string(pos);
        
        item_registry[info.item_id] = info;
        
        std::cout << "      Item " << info.item_id << ": type=" << info.item_type 
                  << " name=" << (item_name.empty() ? "(none)" : item_name) << std::endl;
    }
    
    // Parse pitm (Primary Item)
    void parse_primary_item() {
        auto* pitm = find_box("pitm");
        if (!pitm) return;
        
        size_t pos = pitm->offset + 8;
        
        uint8_t version = read_uint8(pos++);
        pos += 3;
        
        item_id_type primary_id;
        if (version == 0) {
            primary_id = read_uint16(pos);
        } else {
            primary_id = read_uint32(pos);
        }
        
        if (item_registry.find(primary_id) != item_registry.end()) {
            item_registry[primary_id].is_primary = true;
            std::cout << "   Primary item: " << primary_id << std::endl;
        }
    }
    
    // Parse all ispe boxes
    void parse_all_ispe_boxes() {
        std::cout << "\n📐 Parsing Image Spatial Extents (ispe)..." << std::endl;
        
        auto* ipco = find_box("ipco");
        if (!ipco) {
            std::cout << "   No ipco box found" << std::endl;
            return;
        }
        
        size_t pos = ipco->offset + 8;
        size_t end_pos = ipco->offset + ipco->size;
        
        std::vector<std::pair<size_t, int>> ispe_entries;
        int property_index = 1;
        
        while (pos < end_pos && pos + 8 < file_data.size()) {
            uint32_t prop_size = read_uint32(pos);
            std::string prop_type = fourcc_to_string(read_uint32(pos + 4));
            
            if (prop_size < 8 || pos + prop_size > end_pos) break;
            
            if (prop_type == "ispe") {
                ispe_entries.push_back({pos, property_index});
                std::cout << "   Found ispe at property index " << property_index << std::endl;
            }
            
            pos += prop_size;
            property_index++;
        }
        
        parse_ipma_and_associate_ispe(ispe_entries);
    }
    
    // Parse ipma and associate ispe
    void parse_ipma_and_associate_ispe(const std::vector<std::pair<size_t, int>>& ispe_entries) {
        auto* ipma = find_box("ipma");
        if (!ipma) {
            std::cout << "   No ipma box found" << std::endl;
            return;
        }
        
        size_t pos = ipma->offset + 8;
        
        uint8_t version = read_uint8(pos++);
        uint32_t flags = (read_uint8(pos) << 16) | (read_uint8(pos+1) << 8) | read_uint8(pos+2);
        pos += 3;
        
        uint32_t entry_count = read_uint32(pos);
        pos += 4;
        
        std::cout << "   Processing " << entry_count << " item-property associations" << std::endl;
        
        for (uint32_t i = 0; i < entry_count && pos < ipma->offset + ipma->size; i++) {
            item_id_type item_id;
            if (version < 1) {
                item_id = read_uint16(pos);
                pos += 2;
            } else {
                item_id = read_uint32(pos);
                pos += 4;
            }
            
            uint8_t association_count = read_uint8(pos++);
            
            for (int j = 0; j < association_count && pos < ipma->offset + ipma->size; j++) {
                uint16_t property_index;
                
                if (flags & 1) {
                    property_index = read_uint16(pos);
                    pos += 2;
                    property_index &= 0x7FFF;
                } else {
                    property_index = read_uint8(pos++);
                    property_index &= 0x7F;
                }
                
                for (const auto& [ispe_offset, ispe_index] : ispe_entries) {
                    if (ispe_index == property_index) {
                        parse_ispe_and_associate(ispe_offset, item_id);
                    }
                }
            }
        }
    }
    
    // Parse ispe and associate with item
    void parse_ispe_and_associate(size_t offset, item_id_type item_id) {
        size_t pos = offset + 8;
        
        uint8_t version = read_uint8(pos++);
        uint32_t flags = (read_uint8(pos) << 16) | (read_uint8(pos+1) << 8) | read_uint8(pos+2);
        pos += 3;
        
        uint32_t width = read_uint32(pos); pos += 4;
        uint32_t height = read_uint32(pos); pos += 4;
        
        if (item_registry.find(item_id) != item_registry.end()) {
            item_registry[item_id].width = width;
            item_registry[item_id].height = height;
            
            std::cout << "      Item " << item_id << " dimensions: " 
                      << width << "x" << height << std::endl;
        }
    }
    
    // Detect tiling scheme for item
    std::string detect_tiling_scheme_for_item(item_id_type item_id) {
        if (item_registry[item_id].item_type == "grid") return "grid";
        if (item_registry[item_id].item_type == "unci") return "unci";
        if (item_registry[item_id].item_type == "tili") return "tili";
        return "none";
    }
    
    // Parse pyramid entity group
    bool parse_pyramid_entity_group(GeoHEIFInfo& info) {
        std::cout << "\n🏔️ Detecting Pyramid Structure..." << std::endl;
        
        parse_item_information();
        parse_primary_item();
        parse_all_ispe_boxes();
        
        auto grpl_boxes = find_all_boxes("grpl");
        
        if (grpl_boxes.empty()) {
            std::cout << "   No entity groups found" << std::endl;
            
            // Check for single image (non-pyramid)
            for (const auto& [item_id, item_info] : item_registry) {
                if (item_info.is_primary && item_info.width > 0 && item_info.height > 0) {
                    std::cout << "   Single image detected (non-pyramid)" << std::endl;
                    
                    PyramidLevelInfo level;
                    level.level_index = 0;
                    level.item_id = item_id;
                    level.width = item_info.width;
                    level.height = item_info.height;
                    level.tiling_scheme = detect_tiling_scheme_for_item(item_id);
                    level.is_tiled = (level.tiling_scheme != "none");
                    
                    if (level.is_tiled) {
                        extract_tiling_details(level);
                    }
                    
                    info.pyramid_levels.push_back(level);
                    info.is_pyramid = false;
                    
                    return true;
                }
            }
            
            return false;
        }
        
        std::cout << "   Found " << grpl_boxes.size() << " entity group(s)" << std::endl;
        
        bool found_pyramid = false;
        std::vector<item_id_type> pyramid_item_ids;
        
        for (auto* grpl_box : grpl_boxes) {
            // grpl is a CONTAINER box, not a FullBox
            // It contains entity group boxes directly
            size_t pos = grpl_box->offset + 8;  // Skip size and type only
            
            // IMPORTANT: grpl may be a FullBox - check the spec
            // If it IS a FullBox, uncomment these lines:
            // uint8_t version = read_uint8(pos++);
            // uint32_t flags = (read_uint8(pos) << 16) | (read_uint8(pos+1) << 8) | read_uint8(pos+2);
            // pos += 3;
            
            while (pos + 8 < grpl_box->offset + grpl_box->size) {
                uint32_t group_size = read_uint32(pos);
                if (group_size < 8 || pos + group_size > file_data.size()) break;
                
                uint32_t group_type = read_uint32(pos + 4);
                std::string group_type_str = fourcc_to_string(group_type);
                
                std::cout << "   Entity group type: " << group_type_str 
                        << " (size: " << group_size << ")" << std::endl;
                
                if (group_type_str == "pymd") {
                    std::cout << "   ✓ Pyramid entity group found!" << std::endl;
                    found_pyramid = true;

                    size_t group_pos = pos + 8;

                    // Read FullBox header (version + flags)
                    uint8_t version = read_uint8(group_pos);
                    group_pos++;
                    
                    uint32_t flags = (read_uint8(group_pos) << 16) | 
                                    (read_uint8(group_pos+1) << 8) | 
                                    read_uint8(group_pos+2);
                    group_pos += 3;
                    
                    std::cout << "      pymd version: " << static_cast<int>(version) << std::endl;
                    std::cout << "      pymd flags: 0x" << std::hex << flags << std::dec << std::endl;

                    // Read group_id (present in version 1 and later)
                    if (version >= 1) {
                        uint32_t group_id = read_uint32(group_pos);
                        group_pos += 4;
                        std::cout << "      Group ID: " << group_id << std::endl;
                    }

                    // Read number of entities
                    uint32_t num_entities = read_uint32(group_pos);
                    group_pos += 4;

                    std::cout << "      Number of levels: " << num_entities << std::endl;

                    // Read all entity IDs
                    for (uint32_t i = 0; i < num_entities && group_pos + 4 <= pos + group_size; i++) {
                        item_id_type item_id = read_uint32(group_pos);
                        group_pos += 4;

                        pyramid_item_ids.push_back(item_id);
                        std::cout << "      Level " << i << " item ID: " << item_id << std::endl;
                    }
                }
                
                pos += group_size;
            }
        }
        
        if (found_pyramid && !pyramid_item_ids.empty()) {
            info.is_pyramid = true;
            extract_pyramid_level_info(info, pyramid_item_ids);
            return true;
        }
        
        return false;
    }
    
    // Extract pyramid level info
    void extract_pyramid_level_info(GeoHEIFInfo& info, const std::vector<item_id_type>& item_ids) {
        std::cout << "\n📐 Extracting Pyramid Level Information..." << std::endl;

        info.pyramid_levels.clear();

        for (size_t i = 0; i < item_ids.size(); i++) {
            item_id_type item_id = item_ids[i];

            if (item_registry.find(item_id) == item_registry.end()) {
                std::cerr << "   ⚠ Item " << item_id << " not found in registry" << std::endl;
                continue;
            }

            const auto& item_info = item_registry[item_id];

            PyramidLevelInfo level;
            level.level_index = static_cast<int>(i);
            level.item_id = item_id;
            level.width = item_info.width;
            level.height = item_info.height;
            level.compression_type = item_info.item_type;

            std::cout << "   Level " << i << " (item " << item_id << "): "
                    << level.width << "x" << level.height
                    << " [" << level.compression_type << "]" << std::endl;
            
            level.tiling_scheme = detect_tiling_scheme_for_item(item_id);
            level.is_tiled = (level.tiling_scheme != "none");
            
            if (level.is_tiled) {
                std::cout << "      ✓ Tiled: " << level.tiling_scheme << std::endl;
                extract_tiling_details(level);
            }
            
            if (info.georeference.is_georectified && level.width > 0 && level.height > 0) {
                compute_level_bounding_box(level, info.georeference);
                
                if (level.has_bbox) {
                    std::cout << "      BBox: [" << std::fixed << std::setprecision(6)
                              << level.min_x << ", " << level.min_y 
                              << "] to [" << level.max_x << ", " << level.max_y << "]" << std::endl;
                    std::cout << "      Pixel size: " << level.pixel_size_x 
                              << " x " << level.pixel_size_y << std::endl;
                }
            }
            
            info.pyramid_levels.push_back(level);
        }
        
        std::cout << "   ✓ Extracted " << info.pyramid_levels.size() << " pyramid levels" << std::endl;
    }
    
    // Extract tiling details
    void extract_tiling_details(PyramidLevelInfo& level) {
        if (level.tiling_scheme == "grid") {
            extract_grid_tiling(level);
        } else if (level.tiling_scheme == "unci") {
            extract_unci_tiling(level);
        } else if (level.tiling_scheme == "tili") {
            extract_tili_tiling(level);
        }
    }
    
    // Extract grid tiling
    void extract_grid_tiling(PyramidLevelInfo& level) {
        auto grid_boxes = find_all_boxes("grid");
        
        for (auto* grid_box : grid_boxes) {
            size_t pos = grid_box->offset + 8;
            
            uint8_t version = read_uint8(pos++);
            uint32_t flags = (read_uint8(pos) << 16) | (read_uint8(pos+1) << 8) | read_uint8(pos+2);
            pos += 3;
            
            level.num_tile_rows = read_uint8(pos++);
            level.num_tile_cols = read_uint8(pos++);
            
            if (flags & 1) {
                level.padded_width = read_uint32(pos); pos += 4;
                level.padded_height = read_uint32(pos); pos += 4;
            } else {
                level.padded_width = read_uint16(pos); pos += 2;
                level.padded_height = read_uint16(pos); pos += 2;
            }
            
            if (level.num_tile_cols > 0 && level.num_tile_rows > 0) {
                level.tile_width = level.width / level.num_tile_cols;
                level.tile_height = level.height / level.num_tile_rows;
                
                if (level.width % level.num_tile_cols != 0) {
                    level.tile_width = (level.width + level.num_tile_cols - 1) / level.num_tile_cols;
                }
                if (level.height % level.num_tile_rows != 0) {
                    level.tile_height = (level.height + level.num_tile_rows - 1) / level.num_tile_rows;
                }
                
                std::cout << "         Grid: " << level.num_tile_cols << "x" << level.num_tile_rows
                          << " tiles of " << level.tile_width << "x" << level.tile_height << std::endl;
            }
            
            return;
        }
    }
    
    // Extract UNCI tiling
    void extract_unci_tiling(PyramidLevelInfo& level) {
        for (int tile_size : {512, 256, 1024, 128}) {
            if (level.width % tile_size == 0 && level.height % tile_size == 0) {
                level.tile_width = tile_size;
                level.tile_height = tile_size;
                level.num_tile_cols = level.width / tile_size;
                level.num_tile_rows = level.height / tile_size;
                level.padded_width = level.width;
                level.padded_height = level.height;
                
                std::cout << "         UNCI (estimated): " << tile_size << "x" << tile_size << " tiles" << std::endl;
                return;
            }
        }
        
        level.tile_width = 256;
        level.tile_height = 256;
        level.num_tile_cols = (level.width + 255) / 256;
        level.num_tile_rows = (level.height + 255) / 256;
        level.padded_width = level.num_tile_cols * 256;
        level.padded_height = level.num_tile_rows * 256;
        
        std::cout << "         UNCI (default): 256x256 tiles" << std::endl;
    }
    
    // Extract TILI tiling
    void extract_tili_tiling(PyramidLevelInfo& level) {
        auto tili_boxes = find_all_boxes("tili");
        
        for (auto* tili_box : tili_boxes) {
            size_t pos = tili_box->offset + 8;
            
            uint8_t version = read_uint8(pos++);
            uint32_t flags = (read_uint8(pos) << 16) | (read_uint8(pos+1) << 8) | read_uint8(pos+2);
            pos += 3;
            
            level.tile_width = read_uint32(pos); pos += 4;
            level.tile_height = read_uint32(pos); pos += 4;
            
            if (level.tile_width > 0 && level.tile_height > 0) {
                level.num_tile_cols = (level.width + level.tile_width - 1) / level.tile_width;
                level.num_tile_rows = (level.height + level.tile_height - 1) / level.tile_height;
                
                level.padded_width = level.num_tile_cols * level.tile_width;
                level.padded_height = level.num_tile_rows * level.tile_height;
                
                std::cout << "         TILI: " << level.tile_width << "x" << level.tile_height 
                          << " tiles (" << level.num_tile_cols << "x" << level.num_tile_rows << ")" << std::endl;
            }
            
            return;
        }
    }
    
    // Compute bounding box for level
    void compute_level_bounding_box(PyramidLevelInfo& level, 
                                    const GeoHEIFInfo::GeoreferenceInfo& georeference) {
        if (!georeference.is_georectified) return;
        
        std::vector<std::pair<double, double>> model_corners;
        
        for (int i : {0, level.width}) {
            for (int j : {0, level.height}) {
                double x = georeference.m00 * i + georeference.m01 * j + georeference.m03;
                double y = georeference.m10 * i + georeference.m11 * j + georeference.m13;
                model_corners.push_back({x, y});
            }
        }
        
        level.min_x = level.max_x = model_corners[0].first;
        level.min_y = level.max_y = model_corners[0].second;
        
        for (const auto& [x, y] : model_corners) {
            level.min_x = std::min(level.min_x, x);
            level.max_x = std::max(level.max_x, x);
            level.min_y = std::min(level.min_y, y);
            level.max_y = std::max(level.max_y, y);
        }
        
        level.pixel_size_x = std::abs(georeference.m00);
        level.pixel_size_y = std::abs(georeference.m11);
        level.has_bbox = true;
    }

    
public:
    // Main inspection function
    static bool inspect(const char* geoheif_file, GeoHEIFInfo& info) {
        std::cout << "\n" << std::string(80, '=') << std::endl;
        std::cout << "GeoHEIF Inspector" << std::endl;
        std::cout << std::string(80, '=') << std::endl;
        std::cout << "File: " << geoheif_file << std::endl;
        
        try {
            GeoHEIFInspector inspector;
            info.filename = geoheif_file;
            
            // Load file
            std::ifstream file(geoheif_file, std::ios::binary | std::ios::ate);
            if (!file) {
                std::cerr << "✗ Cannot open file" << std::endl;
                return false;
            }
            
            info.file_size = file.tellg();
            file.seekg(0, std::ios::beg);
            
            inspector.file_data.resize(info.file_size);
            file.read(reinterpret_cast<char*>(inspector.file_data.data()), info.file_size);
            
            std::cout << "Size: " << info.file_size << " bytes (" 
                      << (info.file_size / 1024.0) << " KB)" << std::endl;
            
            // Parse boxes using HEIFFileModifier
            //HEIFFileModifier modifier;
            //if (!modifier.load(geoheif_file)) {
            //    return false;
            //}
            inspector.boxes.clear();
            inspector.parse_boxes_recursive(0, info.file_size);
            
            std::cout << "Found " << inspector.boxes.size() << " boxes" << std::endl;
            
            // Parse ftyp box
            std::cout << "\n🏷️  Brands:" << std::endl;
            auto* ftyp = inspector.find_box("ftyp");
            if (ftyp) {
                inspector.parse_ftyp(info, ftyp->offset, ftyp->size);
                std::cout << "   Major brand: " << info.major_brand << std::endl;
                std::cout << "   Compatible brands: ";
                for (const auto& brand : info.compatible_brands) {
                    std::cout << brand << " ";
                }
                std::cout << std::endl;
                std::cout << "   Has 'ogeo' brand: " << (info.has_ogeo_brand ? "✓ Yes" : "✗ No") << std::endl;
            }
            
            // Parse GeoHEIF property boxes
            std::cout << "\n🌍 GeoHEIF Properties:" << std::endl;
            
            // mcrs (CRS)
            auto* mcrs = inspector.find_box("mcrs");
            if (mcrs) {
                inspector.parse_mcrs(info, mcrs->offset, mcrs->size);
                std::cout << "   ✓ mcrs (CRS): " << info.crs.encoding_name 
                          << " = " << info.crs.crs_string << std::endl;
                if (info.crs.epsg_code > 0) {
                    std::cout << "      EPSG: " << info.crs.epsg_code << std::endl;
                }
            } else {
                std::cout << "   ✗ mcrs (CRS): Not found" << std::endl;
            }
            
            // mtxf (affine transform)
            auto* mtxf = inspector.find_box("mtxf");
            if (mtxf) {
                inspector.parse_mtxf(info, mtxf->offset, mtxf->size);
                std::cout << "   ✓ mtxf (Affine Transform): Found" << std::endl;
            }
            
            // tiep (tie points)
            auto* tiep = inspector.find_box("tiep");
            if (tiep) {
                inspector.parse_tiep(info, tiep->offset, tiep->size);
                std::cout << "   ✓ tiep (Tie Points): " << info.georeference.tie_points.size() 
                          << " points" << std::endl;
            }
            
            // Check for other GeoHEIF boxes
            info.has_edim = (inspector.find_box("edim") != nullptr);
            info.has_edvl = (inspector.find_box("edvl") != nullptr);
            info.has_pcel = (inspector.find_box("pcel") != nullptr);
            info.has_pcat = (inspector.find_box("pcat") != nullptr);
            
            if (info.has_edim) std::cout << "   ✓ edim (Extra Dimensions)" << std::endl;
            if (info.has_edvl) std::cout << "   ✓ edvl (Dimension Values)" << std::endl;
            if (info.has_pcel) std::cout << "   ✓ pcel (Cell Properties)" << std::endl;
            if (info.has_pcat) std::cout << "   ✓ pcat (Categories)" << std::endl;
            
            // Get image dimensions (from ispe or similar)
            // For now, set placeholder - would need to parse meta/iprp/ipco/ispe
            // This would require more detailed HEIF parsing
            info.width = 0;   // TODO: Extract from ispe property
            info.height = 0;  // TODO: Extract from ispe property
            
            // Get image dimensions from ispe
            auto* ispe = inspector.find_box("ispe");
            if (ispe) {
                inspector.parse_ispe(info, ispe->offset, ispe->size);
            } else {
                std::cout << "   ⚠ ispe (Image Spatial Extents): Not found" << std::endl;
            }

            // Compute bounding box if we have dimensions
            if (info.width > 0 && info.height > 0) {
                inspector.compute_bounding_box(info);
                if (info.georeference.has_bbox) {
                    std::cout << "\n📐 Computed Bounding Box:" << std::endl;
                    std::cout << "   Min: (" << info.georeference.min_x 
                            << ", " << info.georeference.min_y << ")" << std::endl;
                    std::cout << "   Max: (" << info.georeference.max_x 
                            << ", " << info.georeference.max_y << ")" << std::endl;
                }
            }

            // Parse pyramid structure
            inspector.parse_pyramid_entity_group(info);
            
            // Set primary image dimensions
            if (!info.pyramid_levels.empty()) {
                info.width = info.pyramid_levels[0].width;
                info.height = info.pyramid_levels[0].height;
            }
            
            std::cout << "\n✓ Inspection complete" << std::endl;
            return true;
            
        } catch (const std::exception& e) {
            std::cerr << "✗ Exception: " << e.what() << std::endl;
            return false;
        }
    }
    
    // Print detailed report
    static void print_report(const GeoHEIFInfo& info) {
        std::cout << "\n" << std::string(80, '=') << std::endl;
        std::cout << "GeoHEIF Inspection Report" << std::endl;
        std::cout << std::string(80, '=') << std::endl;
        
        std::cout << "\n📄 File Information:" << std::endl;
        std::cout << "   File: " << info.filename << std::endl;
        std::cout << "   Size: " << info.file_size << " bytes (" 
                  << std::fixed << std::setprecision(2) 
                  << (info.file_size / 1024.0 / 1024.0) << " MB)" << std::endl;
        
        std::cout << "\n🏷️  HEIF Brands:" << std::endl;
        std::cout << "   Major: " << info.major_brand << std::endl;
        std::cout << "   Compatible: ";
        for (size_t i = 0; i < info.compatible_brands.size(); i++) {
            if (i > 0) std::cout << ", ";
            std::cout << info.compatible_brands[i];
        }
        std::cout << std::endl;
        std::cout << "   GeoHEIF ('ogeo'): " << (info.has_ogeo_brand ? "✓ YES" : "✗ NO") << std::endl;
        
        if (info.crs.present) {
            std::cout << "\n🌍 Coordinate Reference System:" << std::endl;
            std::cout << "   Encoding: " << info.crs.encoding_name << std::endl;
            std::cout << "   Definition: " << info.crs.crs_string << std::endl;
            
            if (info.crs.epsg_code > 0) {
                std::cout << "   EPSG Code: " << info.crs.epsg_code << std::endl;
            }
            
            if (info.crs.epoch.has_value()) {
                std::cout << "   Epoch: " << *info.crs.epoch << std::endl;
            }
        }
        
        if (info.georeference.is_georectified || info.georeference.is_georeferenceable) {
            std::cout << "\n📊 Georeference:" << std::endl;
            
            if (info.georeference.is_georectified) {
                std::cout << "   Type: Georectified (Affine Transform)" << std::endl;
                std::cout << "   Dimensions: " << (info.georeference.is_2d ? "2D" : "3D") << std::endl;
                
                auto& g = info.georeference;
                std::cout << "   Transform Matrix:" << std::endl;
                std::cout << "      [" << std::setw(12) << g.m00 << ", " << std::setw(12) << g.m01 
                          << ", " << std::setw(12) << g.m03 << "]" << std::endl;
                std::cout << "      [" << std::setw(12) << g.m10 << ", " << std::setw(12) << g.m11 
                          << ", " << std::setw(12) << g.m13 << "]" << std::endl;
                
                if (!g.is_2d) {
                    std::cout << "      [" << std::setw(12) << g.m20 << ", " << std::setw(12) << g.m21 
                              << ", " << std::setw(12) << g.m22 << ", " << std::setw(12) << g.m23 << "]" << std::endl;
                }
                
                if (g.pixel_size_x > 0 && g.pixel_size_y > 0) {
                    std::cout << "   Pixel Size: " << g.pixel_size_x << " x " << g.pixel_size_y << std::endl;
                }
            }
            
            if (info.georeference.is_georeferenceable) {
                std::cout << "   Type: Georeferenceable (Tie Points)" << std::endl;
                std::cout << "   Tie Points: " << info.georeference.tie_points.size() << std::endl;
                
                int display_count = std::min(5, static_cast<int>(info.georeference.tie_points.size()));
                for (int i = 0; i < display_count; i++) {
                    const auto& tp = info.georeference.tie_points[i];
                    std::cout << "      " << (i+1) << ". Pixel(" << tp.i << ", " << tp.j 
                              << ") → Model(" << tp.x << ", " << tp.y;
                    if (tp.z.has_value()) {
                        std::cout << ", " << *tp.z;
                    }
                    std::cout << ")" << std::endl;
                }
                
                if (info.georeference.tie_points.size() > display_count) {
                    std::cout << "      ... and " << (info.georeference.tie_points.size() - display_count) 
                              << " more" << std::endl;
                }
            }
            
            if (info.georeference.has_bbox) {
                std::cout << "\n📐 Bounding Box:" << std::endl;
                std::cout << "   Min X: " << std::fixed << std::setprecision(6) << info.georeference.min_x << std::endl;
                std::cout << "   Max X: " << info.georeference.max_x << std::endl;
                std::cout << "   Min Y: " << info.georeference.min_y << std::endl;
                std::cout << "   Max Y: " << info.georeference.max_y << std::endl;
                std::cout << "   Width:  " << (info.georeference.max_x - info.georeference.min_x) << std::endl;
                std::cout << "   Height: " << (info.georeference.max_y - info.georeference.min_y) << std::endl;
            }
        }
        
        if (info.has_edim || info.has_edvl || info.has_pcel || info.has_pcat) {
            std::cout << "\n📦 Additional Properties:" << std::endl;
            if (info.has_edim) std::cout << "   ✓ Extra Dimensions (datacube)" << std::endl;
            if (info.has_edvl) std::cout << "   ✓ Dimension Values" << std::endl;
            if (info.has_pcel) std::cout << "   ✓ Cell Properties" << std::endl;
            if (info.has_pcat) std::cout << "   ✓ Property Categories" << std::endl;
        }
        
        std::cout << "\n" << std::string(80, '=') << std::endl;
    }
    
    // Extract just CRS and bbox (quick function)
    static bool extract_crs_bbox(const char* geoheif_file, 
                                  std::string& crs_string,
                                  double& min_x, double& min_y,
                                  double& max_x, double& max_y) {
        GeoHEIFInfo info;
        if (!inspect(geoheif_file, info)) {
            return false;
        }
        
        if (info.crs.present) {
            crs_string = info.crs.crs_string;
        } else {
            return false;
        }
        
        if (info.georeference.has_bbox) {
            min_x = info.georeference.min_x;
            min_y = info.georeference.min_y;
            max_x = info.georeference.max_x;
            max_y = info.georeference.max_y;
            return true;
        }
        
        return false;
    }
};

} // namespace geoheif

#endif // HEIF_GEOHEIF_INSPECTOR_H

### GeoHEIF Inspector Usage Examples

In [ ]:
#ifndef HEIF_GEOHEIF_INSPECTOR_EXAMPLES_H
#define HEIF_GEOHEIF_INSPECTOR_EXAMPLES_H

namespace geoheif {

// Example 1: Full inspection with detailed report
void example_inspect_full(const char* geoheif_file) {
    std::cout << "\n╔════════════════════════════════════════════════════════════════╗" << std::endl;
    std::cout << "║  Example: Full GeoHEIF Inspection                             ║" << std::endl;
    std::cout << "╚════════════════════════════════════════════════════════════════╝" << std::endl;
    
    GeoHEIFInfo info;
    if (GeoHEIFInspector::inspect(geoheif_file, info)) {
        GeoHEIFInspector::print_report(info);
    }
}

// Example 2: Quick CRS and bbox extraction
void example_extract_crs_bbox(const char* geoheif_file) {
    std::cout << "\n╔════════════════════════════════════════════════════════════════╗" << std::endl;
    std::cout << "║  Example: Quick CRS & BBox Extraction                         ║" << std::endl;
    std::cout << "╚════════════════════════════════════════════════════════════════╝" << std::endl;
    
    std::string crs;
    double min_x, min_y, max_x, max_y;
    
    if (GeoHEIFInspector::extract_crs_bbox(geoheif_file, crs, min_x, min_y, max_x, max_y)) {
        std::cout << "\n✓ Extraction successful:" << std::endl;
        std::cout << "   CRS: " << crs << std::endl;
        std::cout << "   BBox: [" << min_x << ", " << min_y << "] to [" 
                  << max_x << ", " << max_y << "]" << std::endl;
    } else {
        std::cout << "\n✗ Failed to extract CRS/BBox" << std::endl;
    }
}

// Example 3: Batch inspection of directory
void example_batch_inspect(const char* directory) {
    std::cout << "\n╔════════════════════════════════════════════════════════════════╗" << std::endl;
    std::cout << "║  Example: Batch Inspection                                    ║" << std::endl;
    std::cout << "╚════════════════════════════════════════════════════════════════╝" << std::endl;
    
    std::cout << "\n📊 Scanning directory: " << directory << std::endl;
    std::cout << std::string(100, '-') << std::endl;
    std::cout << std::left << std::setw(30) << "File"
              << std::setw(12) << "ogeo brand"
              << std::setw(20) << "CRS"
              << "BBox" << std::endl;
    std::cout << std::string(100, '-') << std::endl;
    
    for (const auto& entry : fs::directory_iterator(directory)) {
        if (entry.is_regular_file()) {
            std::string ext = entry.path().extension().string();
            if (ext == ".heif" || ext == ".heic" || ext == ".avif") {
                GeoHEIFInfo info;
                if (GeoHEIFInspector::inspect(entry.path().string().c_str(), info)) {
                    std::cout << std::left << std::setw(30) << entry.path().filename().string()
                              << std::setw(12) << (info.has_ogeo_brand ? "✓" : "✗");
                    
                    if (info.crs.present) {
                        std::string crs_short = info.crs.crs_string;
                        if (crs_short.length() > 18) {
                            crs_short = crs_short.substr(0, 15) + "...";
                        }
                        std::cout << std::setw(20) << crs_short;
                    } else {
                        std::cout << std::setw(20) << "-";
                    }
                    
                    if (info.georeference.has_bbox) {
                        std::cout << "[" << std::fixed << std::setprecision(1) 
                                  << info.georeference.min_x << "," << info.georeference.min_y 
                                  << " to " << info.georeference.max_x << "," << info.georeference.max_y << "]";
                    } else {
                        std::cout << "-";
                    }
                    
                    std::cout << std::endl;
                }
            }
        }
    }
    
    std::cout << std::string(100, '-') << std::endl;
}

// Example 4: Compare GeoTIFF vs GeoHEIF
void example_compare_geotiff_geoheif(const char* geotiff_file, const char* geoheif_file) {
    std::cout << "\n╔════════════════════════════════════════════════════════════════╗" << std::endl;
    std::cout << "║  Example: Compare GeoTIFF vs GeoHEIF Metadata                 ║" << std::endl;
    std::cout << "╚════════════════════════════════════════════════════════════════╝" << std::endl;
    
    // Extract from GeoTIFF
    GeoTIFFGeoreference gtiff;
    GeoTIFFExtractor::extract(geotiff_file, gtiff);
    
    // Inspect GeoHEIF
    GeoHEIFInfo geoheif;
    GeoHEIFInspector::inspect(geoheif_file, geoheif);
    
    // Compare
    std::cout << "\n📊 Comparison:" << std::endl;
    std::cout << std::string(80, '-') << std::endl;
    std::cout << std::left << std::setw(25) << "Property" 
              << std::setw(30) << "GeoTIFF" 
              << "GeoHEIF" << std::endl;
    std::cout << std::string(80, '-') << std::endl;
    
    // Dimensions
    std::cout << std::left << std::setw(25) << "Dimensions"
              << std::setw(30) << (std::to_string(gtiff.width) + "x" + std::to_string(gtiff.height))
              << (std::to_string(geoheif.width) + "x" + std::to_string(geoheif.height)) << std::endl;
    
    // CRS
    std::cout << std::left << std::setw(25) << "Has CRS"
              << std::setw(30) << (gtiff.has_crs ? "✓" : "✗")
              << (geoheif.crs.present ? "✓" : "✗") << std::endl;
    
    if (gtiff.epsg_code > 0 && geoheif.crs.epsg_code > 0) {
        std::cout << std::left << std::setw(25) << "EPSG Code"
                  << std::setw(30) << gtiff.epsg_code
                  << geoheif.crs.epsg_code << std::endl;
        
        if (gtiff.epsg_code == geoheif.crs.epsg_code) {
            std::cout << "   ✓ EPSG codes match!" << std::endl;
        }
    }
    
    // Georeference type
    std::string gtiff_type = gtiff.has_geotransform ? "Affine" : 
                            gtiff.has_gcps ? "GCPs" : "None";
    std::string geoheif_type = geoheif.georeference.is_georectified ? "Affine" :
                               geoheif.georeference.is_georeferenceable ? "GCPs" : "None";
    
    std::cout << std::left << std::setw(25) << "Georeference Type"
              << std::setw(30) << gtiff_type
              << geoheif_type << std::endl;
    
    // Bounding box comparison
    if (gtiff.has_geotransform && geoheif.georeference.has_bbox) {
        std::cout << "\nBounding Box Comparison:" << std::endl;
        std::cout << "   " << std::left << std::setw(10) << "" 
                  << std::setw(25) << "GeoTIFF" << "GeoHEIF" << std::endl;
        std::cout << "   " << std::left << std::setw(10) << "Min X:" 
                  << std::setw(25) << gtiff.min_x << geoheif.georeference.min_x << std::endl;
        std::cout << "   " << std::left << std::setw(10) << "Max X:" 
                  << std::setw(25) << gtiff.max_x << geoheif.georeference.max_x << std::endl;
        std::cout << "   " << std::left << std::setw(10) << "Min Y:" 
                  << std::setw(25) << gtiff.min_y << geoheif.georeference.min_y << std::endl;
        std::cout << "   " << std::left << std::setw(10) << "Max Y:" 
                  << std::setw(25) << gtiff.max_y << geoheif.georeference.max_y << std::endl;
        
        // Check if they match (within tolerance)
        double tolerance = 0.001;
        bool match = (std::abs(gtiff.min_x - geoheif.georeference.min_x) < tolerance &&
                     std::abs(gtiff.max_x - geoheif.georeference.max_x) < tolerance &&
                     std::abs(gtiff.min_y - geoheif.georeference.min_y) < tolerance &&
                     std::abs(gtiff.max_y - geoheif.georeference.max_y) < tolerance);
        
        if (match) {
            std::cout << "\n   ✓ Bounding boxes match!" << std::endl;
        } else {
            std::cout << "\n   ⚠ Bounding boxes differ" << std::endl;
        }
    }
    
    std::cout << std::string(80, '-') << std::endl;
}

} // namespace geoheif

#endif // HEIF_GEOHEIF_INSPECTOR_EXAMPLES_H

### GeoHEIF Inspector Useage in Notebook

In [ ]:
// // ============================================================================
// // GEOHEIF INSPECTION EXAMPLES
// // ============================================================================

// Example 1: Inspect a GeoHEIF file
{
    const char* output_dir = "output";
    std::cout << "\n\n";
    std::cout << "╔════════════════════════════════════════════════════════════════╗" << std::endl;
    std::cout << "║  INSPECTION EXAMPLE 1: Full Report                            ║" << std::endl;
    std::cout << "╚════════════════════════════════════════════════════════════════╝" << std::endl;
    
    std::string geoheif_file = (fs::path(output_dir) / "output_heif_with_crs.heif").string();
    geoheif::example_inspect_full(geoheif_file.c_str());
}

// // Example 2: Quick CRS and BBox extraction
// {
//     const char* output_dir = "output";
//     std::cout << "\n\n";
//     std::cout << "╔════════════════════════════════════════════════════════════════╗" << std::endl;
//     std::cout << "║  INSPECTION EXAMPLE 2: Quick CRS & BBox                       ║" << std::endl;
//     std::cout << "╚════════════════════════════════════════════════════════════════╝" << std::endl;
    
//     std::string geoheif_file = (fs::path(output_dir) / "output_heif_with_crs.heif").string();
//     geoheif::example_extract_crs_bbox(geoheif_file.c_str());
// }

// // Example 3: Batch inspect directory
// {
//     std::cout << "\n\n";
//     std::cout << "╔════════════════════════════════════════════════════════════════╗" << std::endl;
//     std::cout << "║  INSPECTION EXAMPLE 3: Batch Directory Inspection             ║" << std::endl;
//     std::cout << "╚════════════════════════════════════════════════════════════════╝" << std::endl;
    
//     geoheif::example_batch_inspect(output_dir);
// }

// Example 4: Compare GeoTIFF vs GeoHEIF
// {
//     const char* input_file = "srcdata/ACT2017.tiff";
//     const char* output_dir = "output";
//     std::cout << "\n\n";
//     std::cout << "╔════════════════════════════════════════════════════════════════╗" << std::endl;
//     std::cout << "║  INSPECTION EXAMPLE 4: Compare GeoTIFF vs GeoHEIF             ║" << std::endl;
//     std::cout << "╚════════════════════════════════════════════════════════════════╝" << std::endl;
    
//     std::string geoheif_file = (fs::path(output_dir) / "output_heif_with_crs.heif").string();
//     geoheif::example_compare_geotiff_geoheif(input_file, geoheif_file.c_str());
// }

### Helper functiion to convert geographic coordinate to pixel coodinate at sspecific level

In [ ]:
#ifndef HELPER_CONVERT_GEOGRAPHIC_COODINATES_TO_PIXEL_COORDINATES_H
#define HELPER_CONVERT_GEOGRAPHIC_COODINATES_TO_PIXEL_COORDINATES_H
// Convert geographic coordinate to pixel coordinate at specific level
/* struct PixelCoordinate {
    double i;  // column (x)
    double j;  // row (y)
    bool valid;
};

PixelCoordinate geo_to_pixel(
    double geo_x, double geo_y,
    const ModelTransformationProperty& transform,
    int level_width, int level_height
) {
    PixelCoordinate result;
    
    if (transform.is_2d) {
        // Inverse affine transformation
        // [x]   [m00 m01] [i]   [m03]
        // [y] = [m10 m11] [j] + [m13]
        
        // Solve for [i, j]:
        double det = transform.m00 * transform.m11 - transform.m01 * transform.m10;
        
        if (std::abs(det) < 1e-10) {
            result.valid = false;
            return result;
        }
        
        double dx = geo_x - transform.m03;
        double dy = geo_y - transform.m13;
        
        result.i = (transform.m11 * dx - transform.m01 * dy) / det;
        result.j = (-transform.m10 * dx + transform.m00 * dy) / det;
        
        // Check bounds
        result.valid = (result.i >= 0 && result.i < level_width &&
                       result.j >= 0 && result.j < level_height);
    } else {
        // 3D transformation (more complex)
        result.valid = false;
    }
    
    return result;
} */
#endif // HELPER_CONVERT_GEOGRAPHIC_COODINATES_TO_PIXEL_COORDINATES_H

### Example Usage of Convert from coodiantes to pixel

In [ ]:
/* 
{
    // Inspect a GeoHEIF file
    GeoHEIFInfo info;
    if (GeoHEIFInspector::inspect("output.heif", info)) {
        // Check if we have georeferencing
        if (info.crs.present && info.georeference.has_bbox) {
            std::cout << "\n=== Geospatial Information ===" << std::endl;
            std::cout << "CRS: " << info.crs.crs_string << std::endl;
            std::cout << "Bounding Box:" << std::endl;
            std::cout << "  (" << info.georeference.min_x << ", " 
                    << info.georeference.min_y << ")" << std::endl;
            std::cout << "  to" << std::endl;
            std::cout << "  (" << info.georeference.max_x << ", " 
                    << info.georeference.max_y << ")" << std::endl;
            
            // Convert a geographic coordinate to pixels
            if (info.georeference.is_georectified) {
                auto pixel = geo_to_pixel(
                    -35.3, 149.1,  // Canberra coordinates
                    *info.affine_transform,
                    info.width, info.height
                );
                
                if (pixel.valid) {
                    std::cout << "\nPixel location: (" 
                            << pixel.i << ", " << pixel.j << ")" << std::endl;
                }
            }
        }
    }
} */

# Working Point

# Work Stop Point

## Enhanced Tiling Support with All Schemes

This section adds support for:
1. TILI (tiled image) encoding
2. Explicit choice between grid/unci/tili schemes
3. All compression algorithms with any tiling scheme
4. Pyramids/overviews with configurable levels and consistent tiling

In [ ]:
#ifndef HEIF_TILING_SCHEME_H
#define HEIF_TILING_SCHEME_H

// Tiling scheme enumeration
enum class TilingScheme : uint8_t {
    Grid,     // Standard grid layout (works with all compressions)
    UNCI,     // Uncompressed with optional compression (ISO 23001-17)
    TILI      // Tiled image (experimental)
};

struct TilingSchemeInfo {
    TilingScheme scheme;
    std::string_view name;
    std::string_view description;
    bool experimental;
    bool supports_all_compressions;
};

constexpr inline std::array<TilingSchemeInfo, 3> TILING_SCHEMES = {{
    {TilingScheme::Grid, "grid", "Standard grid layout", false, true},
    {TilingScheme::UNCI, "unci", "Uncompressed tiling (ISO 23001-17)", false, false},
    {TilingScheme::TILI, "tili", "Tiled image (experimental)", true, true}
}};

inline std::string_view get_tiling_scheme_name(TilingScheme scheme) {
    for (const auto& info : TILING_SCHEMES) {
        if (info.scheme == scheme) {
            return info.name;
        }
    }
    return "unknown";
}

inline bool is_compression_supported_for_scheme(ExtendedCompressionType compression, 
                                               TilingScheme scheme) {
    if (scheme == TilingScheme::Grid || scheme == TilingScheme::TILI) {
        return true;  // Grid and TILI support all compressions
    }
    
    if (scheme == TilingScheme::UNCI) {
        // UNCI only supports uncompressed or specific compressions
        using enum ExtendedCompressionType;
        return (compression == None || compression == Uncompressed ||
                compression == DEFLATE || compression == ZLIB || 
                compression == Brotli);
    }
    
    return false;
}

void list_tiling_schemes() {
    std::cout << "\n=== Available Tiling Schemes ===" << std::endl;
    std::cout << std::left << std::setw(10) << "Scheme" 
              << std::setw(40) << "Description"
              << std::setw(15) << "Experimental"
              << std::setw(20) << "All Compressions" << std::endl;
    std::cout << std::string(85, '-') << std::endl;
    
    for (const auto& info : TILING_SCHEMES) {
        std::cout << std::left << std::setw(10) << info.name
                  << std::setw(40) << info.description
                  << std::setw(15) << (info.experimental ? "Yes" : "No")
                  << std::setw(20) << (info.supports_all_compressions ? "Yes" : "Limited")
                  << std::endl;
    }
}

std::cout << "Tiling schemes defined." << std::endl;

#endif // HEIF_TILING_SCHEME_H

## Pad Image to Match Tile Grid

In [ ]:
#ifndef HEIF_IMAGE_PADDING_H
#define HEIF_IMAGE_PADDING_H

heif_image* pad_image_for_tiling(heif_image* source, 
                                 int tile_width, int tile_height,
                                 uint16_t padding_value = 0) {  // Changed to uint16_t to support up to 16-bit
    int source_width = heif_image_get_primary_width(source);
    int source_height = heif_image_get_primary_height(source);
    
    // Calculate padded dimensions (round up to next tile boundary)
    int num_cols = (source_width + tile_width - 1) / tile_width;
    int num_rows = (source_height + tile_height - 1) / tile_height;
    int padded_width = num_cols * tile_width;
    int padded_height = num_rows * tile_height;
    
    // If already aligned, return as-is
    if (source_width == padded_width && source_height == padded_height) {
        return source;
    }
    
    std::cout << "    Padding image: " << source_width << "x" << source_height 
              << " → " << padded_width << "x" << padded_height 
              << " (padding value: " << padding_value << ")" << std::endl;
    
    // Create padded image
    heif_colorspace colorspace = heif_image_get_colorspace(source);
    heif_chroma chroma = heif_image_get_chroma_format(source);
    
    heif_image* padded = nullptr;
    heif_error error = heif_image_create(padded_width, padded_height, 
                                        colorspace, chroma, &padded);
    check_error_simple(error, "create padded image");
    
    // Get all channels
    std::vector<heif_channel> channels;
    if (heif_image_has_channel(source, heif_channel_Y)) channels.push_back(heif_channel_Y);
    if (heif_image_has_channel(source, heif_channel_Cb)) channels.push_back(heif_channel_Cb);
    if (heif_image_has_channel(source, heif_channel_Cr)) channels.push_back(heif_channel_Cr);
    if (heif_image_has_channel(source, heif_channel_R)) channels.push_back(heif_channel_R);
    if (heif_image_has_channel(source, heif_channel_G)) channels.push_back(heif_channel_G);
    if (heif_image_has_channel(source, heif_channel_B)) channels.push_back(heif_channel_B);
    if (heif_image_has_channel(source, heif_channel_Alpha)) channels.push_back(heif_channel_Alpha);
    
    // Copy and pad each channel
    for (auto channel : channels) {
        int bits = heif_image_get_bits_per_pixel(source, channel);
        heif_image_add_plane(padded, channel, padded_width, padded_height, bits);
        
        int src_stride, dst_stride;
        const uint8_t* src_data = heif_image_get_plane_readonly(source, channel, &src_stride);
        uint8_t* dst_data = heif_image_get_plane(padded, channel, &dst_stride);
        
        std::cout << "      Channel bits per pixel: " << bits << std::endl;
        
        if (bits <= 8) {
            // 8-bit data: use uint8_t
            uint8_t pad_val_8 = static_cast<uint8_t>(padding_value & 0xFF);
            
            // Copy source rows
            for (int y = 0; y < source_height; y++) {
                memcpy(dst_data + y * dst_stride,
                       src_data + y * src_stride,
                       source_width);
                
                // Pad right edge with padding_value
                if (source_width < padded_width) {
                    memset(dst_data + y * dst_stride + source_width, 
                           pad_val_8, 
                           padded_width - source_width);
                }
            }
            
            // Pad bottom rows with padding_value
            for (int y = source_height; y < padded_height; y++) {
                memset(dst_data + y * dst_stride,
                       pad_val_8,
                       padded_width);
            }
        } else {
            // 16-bit data: use uint16_t
            const uint16_t* src_data_16 = reinterpret_cast<const uint16_t*>(src_data);
            uint16_t* dst_data_16 = reinterpret_cast<uint16_t*>(dst_data);
            int src_stride_16 = src_stride / 2;
            int dst_stride_16 = dst_stride / 2;
            
            // Clamp padding value to max for bit depth
            uint16_t max_val = (1 << bits) - 1;
            uint16_t pad_val_16 = std::min(padding_value, max_val);
            
            // Copy source rows
            for (int y = 0; y < source_height; y++) {
                // Copy existing data
                for (int x = 0; x < source_width; x++) {
                    dst_data_16[y * dst_stride_16 + x] = src_data_16[y * src_stride_16 + x];
                }
                
                // Pad right edge
                if (source_width < padded_width) {
                    for (int x = source_width; x < padded_width; x++) {
                        dst_data_16[y * dst_stride_16 + x] = pad_val_16;
                    }
                }
            }
            
            // Pad bottom rows
            for (int y = source_height; y < padded_height; y++) {
                for (int x = 0; x < padded_width; x++) {
                    dst_data_16[y * dst_stride_16 + x] = pad_val_16;
                }
            }
        }
    }
    
    return padded;
}

// Check if image needs padding for tiling scheme
bool needs_padding_for_scheme(TilingScheme scheme, int width, int height, 
                              int tile_width, int tile_height) {
    if (scheme == TilingScheme::Grid) {
        return false;  // Grid handles partial tiles
    }
    
    // UNCI and TILI require exact divisibility
    return (width % tile_width != 0) || (height % tile_height != 0);
}

#endif // HEIF_IMAGE_PADDING_H

## Offset offset documentation

In [ ]:
#ifndef HEIF_OFFSET_TABLE_CONFIG_FINAL_H
#define HEIF_OFFSET_TABLE_CONFIG_FINAL_H

// Final, accurate OffsetTableConfig based on actual libheif API
struct OffsetTableConfig {
    uint8_t offset_field_length;    // Bits: 0, 32, 40, 48, 64
    uint8_t size_field_length;      // Bits: 0, 24, 32, 40, 48, 64
    uint8_t tiles_are_sequential;   // 0 = use tables, 1 = sequential
    
    enum class Preset : uint8_t {
        OffsetAndSize,      // Both offset and size tables
        OffsetOnly,         // Only offset table
        SizeOnly,           // Only size table
        Sequential,         // No tables (sequential)
        Default
    };
    
    // Constructors
    OffsetTableConfig() 
        : offset_field_length(32)
        , size_field_length(24)
        , tiles_are_sequential(0) {}
    
    explicit OffsetTableConfig(Preset preset) {
        switch (preset) {
            case Preset::OffsetAndSize:
                offset_field_length = 32;
                size_field_length = 24;
                tiles_are_sequential = 0;
                break;
            case Preset::OffsetOnly:
                offset_field_length = 32;
                size_field_length = 0;
                tiles_are_sequential = 1;
                break;
            case Preset::SizeOnly:
                offset_field_length = 0;
                size_field_length = 24;
                tiles_are_sequential = 1;
                break;
            case Preset::Sequential:
                offset_field_length = 0;
                size_field_length = 0;
                tiles_are_sequential = 1;
                break;
            case Preset::Default:
            default:
                offset_field_length = 32;
                size_field_length = 24;
                tiles_are_sequential = 0;
                break;
        }
    }
    
    OffsetTableConfig(uint8_t offset_bits, uint8_t size_bits, bool sequential = false)
        : offset_field_length(offset_bits)
        , size_field_length(size_bits)
        , tiles_are_sequential(sequential ? 1 : 0) {}
    
    // Validation
    bool is_valid() const {
        if (offset_field_length != 0 && offset_field_length != 32 && 
            offset_field_length != 40 && offset_field_length != 48 && 
            offset_field_length != 64) {
            return false;
        }
        
        if (size_field_length != 0 && size_field_length != 24 && 
            size_field_length != 32 && size_field_length != 40 && 
            size_field_length != 48 && size_field_length != 64) {
            return false;
        }
        
        return true;
    }
    
    // Check if directly controllable via API (ACCURATE VERSION)
    bool is_directly_controllable(TilingScheme scheme) const {
        switch (scheme) {
            case TilingScheme::Grid:
                // Grid: Uses iloc box, not directly controllable
                return false;
                
            case TilingScheme::UNCI:
                // UNCI: API does NOT expose offset/size fields yet
                // heif_unci_image_parameters has TODO for interleave/padding
                // icef box exists internally but not in public API
                return false;
                
            case TilingScheme::TILI:
                // TILI: Full API support via heif_tiled_image_parameters
                return true;
                
            default:
                return false;
        }
    }
    
    std::string api_status(TilingScheme scheme) const {
        switch (scheme) {
            case TilingScheme::Grid:
                return "❌ Grid: No API support\n"
                       "   • Uses file-level iloc box\n"
                       "   • Tiles stored as separate items\n"
                       "   • Offset/size not configurable per-grid";
                
            case TilingScheme::UNCI:
                return "❌ UNCI: Not yet in API (v1.x)\n"
                       "   • heif_unci_image_parameters incomplete\n"
                       "   • TODO: interleave type, padding\n"
                       "   • icef box exists internally but not exposed\n"
                       "   • May be added in future libheif version";
                
            case TilingScheme::TILI:
                return "✅ TILI: Full API support\n"
                       "   • heif_tiled_image_parameters.offset_field_length\n"
                       "   • heif_tiled_image_parameters.size_field_length\n"
                       "   • heif_tiled_image_parameters.tiles_are_sequential\n"
                       "   • Direct control over tile offset table";
                
            default:
                return "Unknown scheme";
        }
    }
    
    std::string description() const {
        if (tiles_are_sequential) {
            return "Sequential (no offset table)";
        }
        
        std::string desc;
        if (offset_field_length > 0 && size_field_length > 0) {
            desc = std::format("Offset+Size ({}+{} bits)", 
                             offset_field_length, size_field_length);
        } else if (offset_field_length > 0) {
            desc = std::format("Offset-only ({} bits)", offset_field_length);
        } else if (size_field_length > 0) {
            desc = std::format("Size-only ({} bits)", size_field_length);
        } else {
            desc = "No offset table";
        }
        return desc;
    }
    
    size_t table_entry_size_bytes() const {
        if (tiles_are_sequential) return 0;
        return (offset_field_length + size_field_length) / 8;
    }
};

// Print comprehensive comparison table
void print_offset_table_api_support() {
    std::cout << "\n╔════════════════════════════════════════════════════════════════════════╗" << std::endl;
    std::cout << "║  Tile Offset/Size Table Configuration - API Support Status (libheif)  ║" << std::endl;
    std::cout << "╚════════════════════════════════════════════════════════════════════════╝" << std::endl;
    
    std::cout << "\n┌─────────┬────────────────┬──────────────────────────────────────────────┐" << std::endl;
    std::cout << "│ Scheme  │ API Support    │ Details                                      │" << std::endl;
    std::cout << "├─────────┼────────────────┼──────────────────────────────────────────────┤" << std::endl;
    std::cout << "│ Grid    │ ❌ NO          │ • Uses file-level iloc box                   │" << std::endl;
    std::cout << "│         │                │ • Each tile = separate image item            │" << std::endl;
    std::cout << "│         │                │ • No per-grid offset configuration           │" << std::endl;
    std::cout << "│         │                │ • Best for: Standard multi-codec tiling      │" << std::endl;
    std::cout << "├─────────┼────────────────┼──────────────────────────────────────────────┤" << std::endl;
    std::cout << "│ UNCI    │ ❌ NO          │ • heif_unci_image_parameters incomplete      │" << std::endl;
    std::cout << "│         │ (current v1.x) │ • Has: tile_width, tile_height, compression  │" << std::endl;
    std::cout << "│         │                │ • Missing: offset/size table config          │" << std::endl;
    std::cout << "│         │ 🔧 FUTURE      │ • TODO comment: \"interleave type, padding\"   │" << std::endl;
    std::cout << "│         │                │ • icef box exists but not in public API      │" << std::endl;
    std::cout << "│         │                │ • May come in future libheif release         │" << std::endl;
    std::cout << "├─────────┼────────────────┼──────────────────────────────────────────────┤" << std::endl;
    std::cout << "│ TILI    │ ✅ YES         │ • heif_tiled_image_parameters provides:      │" << std::endl;
    std::cout << "│         │ FULL SUPPORT   │   - offset_field_length (0,32,40,48,64 bits) │" << std::endl;
    std::cout << "│         │                │   - size_field_length (0,24,32,40,48,64 bits)│" << std::endl;
    std::cout << "│         │                │   - tiles_are_sequential (0 or 1)            │" << std::endl;
    std::cout << "│         │                │ • Complete control over tile offset table    │" << std::endl;
    std::cout << "│         │                │ • Best for: Advanced offset table control    │" << std::endl;
    std::cout << "└─────────┴────────────────┴──────────────────────────────────────────────┘" << std::endl;
    
    std::cout << "\n📊 SUMMARY:" << std::endl;
    std::cout << "   • Only TILI provides direct offset/size table configuration" << std::endl;
    std::cout << "   • UNCI may add this in future (see TODO in heif_unci_image_parameters)" << std::endl;
    std::cout << "   • Grid uses standard HEIF item location mechanism" << std::endl;
    
    std::cout << "\n💡 RECOMMENDATION:" << std::endl;
    std::cout << "   Use TilingScheme::TILI for explicit offset/size table control" << std::endl;
    
    std::cout << "\n" << std::string(76, '=') << std::endl;
}

// List presets with accurate applicability
void list_offset_table_presets_with_support() {
    std::cout << "\n=== Offset Table Configuration Presets ===" << std::endl;
    std::cout << std::left << std::setw(20) << "Preset" 
              << std::setw(15) << "Offset Bits"
              << std::setw(15) << "Size Bits"
              << std::setw(15) << "Sequential"
              << "Description" << std::endl;
    std::cout << std::string(100, '-') << std::endl;
    
    auto print_preset = [](const char* name, OffsetTableConfig::Preset preset) {
        OffsetTableConfig config(preset);
        std::cout << std::left << std::setw(20) << name
                  << std::setw(15) << static_cast<int>(config.offset_field_length)
                  << std::setw(15) << static_cast<int>(config.size_field_length)
                  << std::setw(15) << (config.tiles_are_sequential ? "Yes" : "No")
                  << config.description() << std::endl;
    };
    
    print_preset("OffsetAndSize", OffsetTableConfig::Preset::OffsetAndSize);
    print_preset("OffsetOnly", OffsetTableConfig::Preset::OffsetOnly);
    print_preset("SizeOnly", OffsetTableConfig::Preset::SizeOnly);
    print_preset("Sequential", OffsetTableConfig::Preset::Sequential);
    print_preset("Default", OffsetTableConfig::Preset::Default);
    
    std::cout << "\n⚠️  IMPORTANT: These configurations are ONLY applied to TILI scheme!" << std::endl;
    std::cout << "   • Grid and UNCI do not support offset table configuration in current API" << std::endl;
    std::cout << "   • Config will be documented but not applied for Grid/UNCI" << std::endl;
}

std::cout << "Accurate offset table configuration structures defined." << std::endl;

#endif // HEIF_OFFSET_TABLE_CONFIG_FINAL_H

## Encoding function with offset configuration

In [ ]:
#ifndef HEIF_ENCODE_TILED_WITH_OFFSET_CONFIG_H
#define HEIF_ENCODE_TILED_WITH_OFFSET_CONFIG_H

// Universal tiled encoding function with offset table configuration
void encode_tiled(const char* input_file, 
                 const char* output_file,
                 int tile_width, 
                 int tile_height,
                 TilingScheme scheme,
                 ExtendedCompressionType compression,
                 int quality = 50,
                 int output_bit_depth = 8,
                 uint16_t padding_value = 0,
                 const OffsetTableConfig& offset_config = OffsetTableConfig()) {
    
    std::cout << "\n=== Encoding Tiled Image ===" << std::endl;
    std::cout << "Tiling Scheme: " << get_tiling_scheme_name(scheme) << std::endl;
    std::cout << "Compression: " << get_compression_name(compression) << std::endl;
    std::cout << "Tile Size: " << tile_width << "x" << tile_height << std::endl;
    std::cout << "Padding Value: " << padding_value << std::endl;
    std::cout << "Offset Table Config: " << offset_config.description() << std::endl;

    // Show offset config status
    if (offset_config.is_directly_controllable(scheme)) {
        std::cout << "\n✅ Offset Table Config: WILL BE APPLIED" << std::endl;
        std::cout << "   " << offset_config.description() << std::endl;
    } else {
        std::cout << "\n⚠️  Offset Table Config: NOT APPLIED (not supported by scheme)" << std::endl;
        std::cout << "   Requested: " << offset_config.description() << std::endl;
        std::cout << "\n" << offset_config.api_status(scheme) << std::endl;
    }
    
    // Validate offset configuration
    if (!offset_config.is_valid()) {
        std::cerr << "Error: Invalid offset table configuration" << std::endl;
        return;
    }
    
    // Validate compression for scheme
    if (!is_compression_supported_for_scheme(compression, scheme)) {
        std::cerr << "Error: Compression " << get_compression_name(compression) 
                  << " not supported with " << get_tiling_scheme_name(scheme) 
                  << " scheme" << std::endl;
        return;
    }
    
    // Load input
    GDALImageData gdal_data = GDALImageLoader::load(input_file);
    heif_image* full_image = GDALImageLoader::to_heif_image(gdal_data, output_bit_depth);
    
    int width = gdal_data.width;
    int height = gdal_data.height;
    
    heif_context* ctx = heif_context_alloc();
    
    // Get encoder
    heif_compression_format heif_format = to_heif_compression_format(compression);
    heif_encoder* encoder = nullptr;
    heif_error error = heif_context_get_encoder_for_format(ctx, heif_format, &encoder);
    check_error_simple(error, "get encoder");
    
    if (heif_format != heif_compression_uncompressed && !is_lossless_compression(compression)) {
        heif_encoder_set_lossy_quality(encoder, quality);
    }
    
    // Calculate grid dimensions
    int num_cols = (width + tile_width - 1) / tile_width;
    int num_rows = (height + tile_height - 1) / tile_height;
    
    std::cout << "Creating " << num_cols << "x" << num_rows << " tile grid" << std::endl;
    
    heif_encoding_options* options = heif_encoding_options_alloc();
    options->unci_compression = to_unci_compression(compression);
    
    heif_image_handle* tiled_handle = nullptr;
    heif_image* working_image = full_image;
    bool image_was_padded = false;
    
    // Check if padding is needed for this scheme
    if (needs_padding_for_scheme(scheme, width, height, tile_width, tile_height)) {
        std::cout << "  Image dimensions not aligned with tile size, padding..." << std::endl;
        working_image = pad_image_for_tiling(full_image, tile_width, tile_height, padding_value);
        image_was_padded = true;
        
        // Update dimensions to padded size
        width = heif_image_get_primary_width(working_image);
        height = heif_image_get_primary_height(working_image);
        num_cols = width / tile_width;
        num_rows = height / tile_height;
    }
    
    // Calculate expected table size
    size_t entry_size = offset_config.table_entry_size_bytes();
    size_t total_tiles = num_cols * num_rows;
    size_t table_size = entry_size * total_tiles;
    
    std::cout << "  Offset table: " << entry_size << " bytes/tile × " 
              << total_tiles << " tiles = " << table_size << " bytes total" << std::endl;
    
    // Create tiled image based on scheme
    switch (scheme) {
        case TilingScheme::Grid: {
            // Note: Grid scheme in libheif may not directly support custom offset tables
            // But we can document the configuration for reference
            std::cout << "  Grid scheme - offset config for reference: " 
                      << offset_config.description() << std::endl;
            
            error = heif_context_add_grid_image(ctx, width, height,
                                               num_cols, num_rows,
                                               options, &tiled_handle);
            check_error_simple(error, "add grid image");
            break;
        }
        
#ifdef WITH_UNCOMPRESSED_CODEC
        case TilingScheme::UNCI: {
            heif_unci_image_parameters params;
            memset(&params, 0, sizeof(params));
            params.version = 1;
            params.image_width = width;
            params.image_height = height;
            params.tile_width = tile_width;
            params.tile_height = tile_height;
            params.compression = to_unci_compression(compression);
            
            // Apply offset table configuration to UNCI
            // Note: Check libheif documentation for exact field names
            // These may need to be adapted based on the actual UNCI API

            std::cout << "\n  UNCI Parameters (Current API v1):" << std::endl;
            std::cout << "    ✓ image_width: " << params.image_width << std::endl;
            std::cout << "    ✓ image_height: " << params.image_height << std::endl;
            std::cout << "    ✓ tile_width: " << params.tile_width << std::endl;
            std::cout << "    ✓ tile_height: " << params.tile_height << std::endl;
            std::cout << "    ✓ compression: " << static_cast<int>(params.compression) << std::endl;
            
            std::cout << "\n    ⚠️  NOT AVAILABLE in API (yet):" << std::endl;
            std::cout << "    ❌ offset_field_length" << std::endl;
            std::cout << "    ❌ size_field_length" << std::endl;
            std::cout << "    ❌ interleave_type (TODO in libheif)" << std::endl;
            std::cout << "    ❌ padding (TODO in libheif)" << std::endl;
            
            std::cout << "\n    📝 Your requested config (for documentation):" << std::endl;

            std::cout << "  UNCI Configuration:" << std::endl;
            std::cout << "    Offset field: " << static_cast<int>(offset_config.offset_field_length) << " bits" << std::endl;
            std::cout << "    Size field: " << static_cast<int>(offset_config.size_field_length) << " bits" << std::endl;
            std::cout << "    Sequential: " << (offset_config.tiles_are_sequential ? "yes" : "no") << std::endl;
            
            // Create prototype tile
            heif_image* prototype_tile = nullptr;
            error = heif_image_extract_area(working_image, 0, 0, tile_width, tile_height,
                                           heif_get_global_security_limits(),
                                           &prototype_tile);
            check_error_simple(error, "extract prototype tile");
            
            error = heif_context_add_empty_unci_image(ctx, &params, options,
                                                     prototype_tile, &tiled_handle);
            heif_image_release(prototype_tile);
            check_error_simple(error, "add UNCI image");
            break;
        }
#endif
        
#ifdef HEIF_ENABLE_EXPERIMENTAL_FEATURES
        case TilingScheme::TILI: {
            heif_tiled_image_parameters tiled_params;
            memset(&tiled_params, 0, sizeof(tiled_params));
            tiled_params.version = 1;
            tiled_params.image_width = width;
            tiled_params.image_height = height;
            tiled_params.tile_width = tile_width;
            tiled_params.tile_height = tile_height;
            
            // Apply offset table configuration to TILI
            tiled_params.offset_field_length = offset_config.offset_field_length;
            tiled_params.size_field_length = offset_config.size_field_length;
            tiled_params.tiles_are_sequential = offset_config.tiles_are_sequential;
            
            std::cout << "  TILI Configuration:" << std::endl;
            std::cout << "    Offset field: " << static_cast<int>(tiled_params.offset_field_length) << " bits" << std::endl;
            std::cout << "    Size field: " << static_cast<int>(tiled_params.size_field_length) << " bits" << std::endl;
            std::cout << "    Sequential: " << (tiled_params.tiles_are_sequential ? "yes" : "no") << std::endl;
            std::cout << "    Expected table size: " << table_size << " bytes" << std::endl;
            
            error = heif_context_add_tiled_image(ctx, &tiled_params, options,
                                                encoder, &tiled_handle);
            check_error_simple(error, "add TILI image");
            break;
        }
#endif
        
        default:
            std::cerr << "Unsupported tiling scheme" << std::endl;
            if (image_was_padded && working_image != full_image) {
                heif_image_release(working_image);
            }
            heif_encoding_options_free(options);
            heif_encoder_release(encoder);
            heif_context_free(ctx);
            heif_image_release(full_image);
            return;
    }
    
    // Encode all tiles
    for (int ty = 0; ty < num_rows; ty++) {
        for (int tx = 0; tx < num_cols; tx++) {
            if ((ty * num_cols + tx) % 100 == 0) {
                std::cout << "\rEncoding tile (" << tx << "," << ty << ")..." << std::flush;
            }
            
            heif_image* tile_image = nullptr;
            error = heif_image_extract_area(working_image,
                                          tx * tile_width, ty * tile_height,
                                          tile_width, tile_height,
                                          heif_get_global_security_limits(),
                                          &tile_image);
            check_error_simple(error, "extract tile");
            
            error = heif_context_add_image_tile(ctx, tiled_handle, tx, ty,
                                               tile_image, encoder);
            check_error_simple(error, "add tile");
            
            heif_image_release(tile_image);
        }
    }
    std::cout << "\nAll tiles encoded." << std::endl;
    
    heif_context_set_primary_image(ctx, tiled_handle);
    
    error = heif_context_write_to_file(ctx, output_file);
    check_error_simple(error, "write to file");
    
    std::cout << "✓ Successfully encoded tiled image: " << output_file << std::endl;
    
    if (image_was_padded && working_image != full_image) {
        heif_image_release(working_image);
    }
    heif_image_handle_release(tiled_handle);
    heif_encoding_options_free(options);
    heif_encoder_release(encoder);
    heif_context_free(ctx);
    heif_image_release(full_image);
}

std::cout << "Tiled encoding with offset configuration defined." << std::endl;

#endif // HEIF_ENCODE_TILED_WITH_OFFSET_CONFIG_H

## Comprehensive Tiled Encoding Functions with Automatic Limit Adjustment

In [ ]:
#ifndef HEIF_ENCODE_TILED_COMPREHENSIVE_H
#define HEIF_ENCODE_TILED_COMPREHENSIVE_H


// Backward compatible versions
void encode_tiled_grid(const char* input_file, const char* output_file,
                      int tile_width, int tile_height,
                      ExtendedCompressionType compression,
                      int quality = 50, int output_bit_depth = 8,
                      uint16_t padding_value = 0,
                      const OffsetTableConfig& offset_config = OffsetTableConfig()) {
    encode_tiled(input_file, output_file, tile_width, tile_height,
                TilingScheme::Grid, compression, quality, output_bit_depth, padding_value, offset_config);
}

#ifdef WITH_UNCOMPRESSED_CODEC
void encode_tiled_unci(const char* input_file, const char* output_file,
                      int tile_width, int tile_height,
                      ExtendedCompressionType compression,
                      int quality = 50, int output_bit_depth = 8,
                      uint16_t padding_value = 0,
                      const OffsetTableConfig& offset_config = OffsetTableConfig()) {
    encode_tiled(input_file, output_file, tile_width, tile_height,
                TilingScheme::UNCI, compression, quality, output_bit_depth, padding_value, offset_config);
}
#endif

#ifdef HEIF_ENABLE_EXPERIMENTAL_FEATURES
void encode_tiled_tili(const char* input_file, const char* output_file,
                      int tile_width, int tile_height,
                      ExtendedCompressionType compression,
                      int quality = 50, int output_bit_depth = 8,
                      uint16_t padding_value = 0,
                      const OffsetTableConfig& offset_config = OffsetTableConfig()) {
    encode_tiled(input_file, output_file, tile_width, tile_height,
                TilingScheme::TILI, compression, quality, output_bit_depth, padding_value, offset_config);
}
#endif

std::cout << "Comprehensive tiled encoding functions defined." << std::endl;

#endif // HEIF_ENCODE_TILED_COMPREHENSIVE_H

## Pyramid/Overview Generation with Configurable Levels and padding (if needed)

In [ ]:
#ifndef HEIF_ENCODE_PYRAMID_COMPREHENSIVE_H
#define HEIF_ENCODE_PYRAMID_COMPREHENSIVE_H

// Structure to define pyramid configuration
struct PyramidConfig {
    int num_levels;                      // Total number of pyramid levels (including full resolution)
    bool use_tiling;                     // Whether to tile each level
    TilingScheme tiling_scheme;          // Tiling scheme to use (if use_tiling is true)
    int tile_width;                      // Tile width (if use_tiling is true)
    int tile_height;                     // Tile height (if use_tiling is true)
    ExtendedCompressionType compression; // Compression for all levels
    int quality;                         // Quality for lossy compression
    int output_bit_depth;                // Bit depth
    uint16_t padding_value;              // Padding value for tiles (if tiling is used)
    OffsetTableConfig offset_config;     // Offset table configuration for tiled levels (if tiling is used)

    // Optional: different settings per level
    std::vector<int> quality_per_level;  // If not empty, use different quality per level

    //constructor with defaults
    PyramidConfig()
        : padding_value(0)
        , offset_config(OffsetTableConfig::Preset::Default) {}
};

// Helper to downsample image
inline heif_image* downsample_image(heif_image* src, int factor) {
    int src_width = heif_image_get_primary_width(src);
    int src_height = heif_image_get_primary_height(src);
    int dst_width = src_width / factor;
    int dst_height = src_height / factor;
    
    heif_image* dst = nullptr;
    heif_error error = heif_image_create(dst_width, dst_height,
                                        heif_image_get_colorspace(src),
                                        heif_image_get_chroma_format(src),
                                        &dst);
    check_error_simple(error, "create downsampled image");
    
    // Simple nearest-neighbor downsampling
    heif_channel channels[] = {heif_channel_R, heif_channel_G, heif_channel_B, 
                               heif_channel_Y, heif_channel_Alpha};
    for (auto channel : channels) {
        if (!heif_image_has_channel(src, channel)) continue;
        
        int bits = heif_image_get_bits_per_pixel(src, channel);
        heif_image_add_plane(dst, channel, dst_width, dst_height, bits);
        
        int src_stride, dst_stride;
        const uint8_t* src_data = heif_image_get_plane_readonly(src, channel, &src_stride);
        uint8_t* dst_data = heif_image_get_plane(dst, channel, &dst_stride);
        
        for (int y = 0; y < dst_height; y++) {
            for (int x = 0; x < dst_width; x++) {
                dst_data[y * dst_stride + x] = src_data[(y * factor) * src_stride + (x * factor)];
            }
        }
    }
    
    return dst;
}


#ifdef HEIF_ENABLE_EXPERIMENTAL_FEATURES
void encode_pyramid_with_config(const char* input_file, 
                               const char* output_file,
                               const PyramidConfig& config) {
    
    std::cout << "\n=== Encoding Pyramid Image ===" << std::endl;
    std::cout << "Levels: " << config.num_levels << std::endl;
    std::cout << "Compression: " << get_compression_name(config.compression) << std::endl;
    if (config.use_tiling) {
        std::cout << "Tiling: " << get_tiling_scheme_name(config.tiling_scheme) 
                  << " (" << config.tile_width << "x" << config.tile_height << ")" << std::endl;
    } else {
        std::cout << "Tiling: None" << std::endl;
    }
    
    // Load input
    GDALImageData gdal_data = GDALImageLoader::load(input_file);
    heif_image* full_image = GDALImageLoader::to_heif_image(gdal_data, config.output_bit_depth);
    
    heif_context* ctx = heif_context_alloc();
    
    // Get encoder
    heif_compression_format heif_format = to_heif_compression_format(config.compression);
    heif_error error;
    /*
    heif_encoder* encoder = nullptr;
    heif_error error = heif_context_get_encoder_for_format(ctx, heif_format, &encoder);
    check_error_simple(error, "get encoder");
    */

    heif_encoding_options* options = heif_encoding_options_alloc();
    options->unci_compression = to_unci_compression(config.compression);
    
    std::vector<heif_item_id> pyramid_ids;
    heif_image* current_level = full_image;
    
    // Encode each pyramid level
    for (int level = 0; level < config.num_levels; level++) {
        int curr_width = heif_image_get_primary_width(current_level);
        int curr_height = heif_image_get_primary_height(current_level);

        // *** Create fresh encoder for THIS level ***
        heif_encoder* encoder = nullptr;
        error = heif_context_get_encoder_for_format(ctx, heif_format, &encoder);
        check_error_simple(error, std::format("get encoder for level {}", level));        
        
        int quality = config.quality;
        if (!config.quality_per_level.empty() && level < config.quality_per_level.size()) {
            quality = config.quality_per_level[level];
        }
        
        if (heif_format != heif_compression_uncompressed && !is_lossless_compression(config.compression)) {
            heif_encoder_set_lossy_quality(encoder, quality);
        }
        
        std::cout << "\nLevel " << level << ": " << curr_width << "x" << curr_height;
        if (!is_lossless_compression(config.compression)) {
            std::cout << " (quality " << quality << ")";
        }
        std::cout << std::endl;
        
        heif_image_handle* level_handle = nullptr;
        heif_image* working_image = current_level;  // Image to use for this level
        bool image_was_padded = false;
        
        if (config.use_tiling) {
            // Check if padding is needed for this scheme
            if (needs_padding_for_scheme(config.tiling_scheme, 
                                        curr_width, curr_height,
                                        config.tile_width, config.tile_height)) {
                std::cout << "  Image dimensions not aligned with tile size, padding..." << std::endl;
                working_image = pad_image_for_tiling(current_level, 
                                                    config.tile_width, 
                                                    config.tile_height,
                                                    config.padding_value);
                image_was_padded = true;
                
                // Update dimensions to padded size
                curr_width = heif_image_get_primary_width(working_image);
                curr_height = heif_image_get_primary_height(working_image);
            }
            
            int num_cols = curr_width / config.tile_width;
            int num_rows = curr_height / config.tile_height;
            
            std::cout << "  Creating " << num_cols << "x" << num_rows << " tile grid" << std::endl;
            
            // Create tiled image structure
            switch (config.tiling_scheme) {
                case TilingScheme::Grid:
                    // Grid can handle original dimensions
                    num_cols = (curr_width + config.tile_width - 1) / config.tile_width;
                    num_rows = (curr_height + config.tile_height - 1) / config.tile_height;
                    
                    error = heif_context_add_grid_image(ctx, curr_width, curr_height,
                                                    num_cols, num_rows,
                                                    options, &level_handle);
                    check_error_simple(error, "add grid image");
                    break;
                
#ifdef WITH_UNCOMPRESSED_CODEC
                case TilingScheme::UNCI: {
                    heif_unci_image_parameters params;
                    memset(&params, 0, sizeof(params));
                    params.version = 1;
                    params.image_width = curr_width;
                    params.image_height = curr_height;
                    params.tile_width = config.tile_width;
                    params.tile_height = config.tile_height;
                    params.compression = to_unci_compression(config.compression);
                    
                    std::cout << "  Creating UNCI image: " << params.image_width 
                            << "x" << params.image_height 
                            << " with " << params.tile_width << "x" << params.tile_height 
                            << " tiles" << std::endl;
                    
                    heif_image* prototype_tile = nullptr;
                    error = heif_image_extract_area(working_image, 0, 0, 
                                                config.tile_width, config.tile_height,
                                                nullptr, &prototype_tile);
                    check_error_simple(error, "extract prototype tile");
                    
                    error = heif_context_add_empty_unci_image(ctx, &params, options,
                                                            prototype_tile, &level_handle);
                    heif_image_release(prototype_tile);
                    check_error_simple(error, "add UNCI image");
                    break;
                }
#endif
            
                case TilingScheme::TILI: {
                    heif_tiled_image_parameters tiled_params;
                    memset(&tiled_params, 0, sizeof(tiled_params));
                    tiled_params.version = 1;
                    tiled_params.image_width = curr_width;
                    tiled_params.image_height = curr_height;
                    tiled_params.tile_width = config.tile_width;
                    tiled_params.tile_height = config.tile_height;

                    // Apply offset configuration from config
                    tiled_params.offset_field_length = config.offset_config.offset_field_length;
                    tiled_params.size_field_length = config.offset_config.size_field_length;
                    tiled_params.tiles_are_sequential = config.offset_config.tiles_are_sequential;

                    std::cout << "  Creating TILI structure for level " << level << std::endl;
                    
                    error = heif_context_add_tiled_image(ctx, &tiled_params, options,
                                                        encoder, &level_handle);
                    
                    if (error.code != heif_error_Ok) {
                        std::cerr << "  TILI failed at level " << level << ": " 
                                << error.message << std::endl;
                        std::cout << "  Falling back to Grid for this level" << std::endl;
                        
                        num_cols = (curr_width + config.tile_width - 1) / config.tile_width;
                        num_rows = (curr_height + config.tile_height - 1) / config.tile_height;
                        
                        error = heif_context_add_grid_image(ctx, curr_width, curr_height,
                                                        num_cols, num_rows,
                                                        options, &level_handle);
                        check_error_simple(error, "add grid image (TILI fallback)");
                    }
                    break;
                }
            }
 
            // Encode tiles for this level -Update Pyramid Encoding to Use Padding for TILI
            std::cout << "  Encoding " << (num_cols * num_rows) << " tiles..." << std::endl;
            
            for (int ty = 0; ty < num_rows; ty++) {
                for (int tx = 0; tx < num_cols; tx++) {
                    int tile_x = tx * config.tile_width;
                    int tile_y = ty * config.tile_height;

                    // *** FIX: Calculate actual tile dimensions ***
                    int actual_tile_width = std::min(config.tile_width, curr_width - tile_x);
                    int actual_tile_height = std::min(config.tile_height, curr_height - tile_y);
                    
                    // Skip if tile is completely outside image bounds
                    if (tile_x >= curr_width || tile_y >= curr_height) {
                        std::cout << "  Skipping out-of-bounds tile (" << tx << "," << ty << ")" << std::endl;
                        continue;
                    }
                    
                    //std::cout << "\rEncoding tile (" << tx << "," << ty << ") at (" 
                    //        << tile_x << "," << tile_y << ") size " 
                    //        << actual_tile_width << "x" << actual_tile_height << "..." << std::flush;
                    
                    // Extract tile - use exact tile size since image is now padded
                    heif_image* tile_image = nullptr;
                    error = heif_image_extract_area(working_image,
                                                tile_x, tile_y,
                                                actual_tile_width, actual_tile_height, //config.tile_width, config.tile_height,
                                                nullptr, &tile_image);
                    
                    if (error.code != heif_error_Ok) {
                        std::cerr << "  Failed to extract tile (" << tx << "," << ty 
                                << ") at level " << level << ": " << error.message << std::endl;
                        
                        // Cleanup
                        if (image_was_padded && working_image != current_level) {
                            heif_image_release(working_image);
                        }
                        if (level_handle) heif_image_handle_release(level_handle);
                        heif_encoding_options_free(options);
                        heif_encoder_release(encoder);
                        heif_context_free(ctx);
                        if (current_level != full_image) heif_image_release(current_level);
                        heif_image_release(full_image);
                        return;
                    }
                    
                    // Add tile
                    error = heif_context_add_image_tile(ctx, level_handle, tx, ty,
                                                    tile_image, encoder);
                    
                    heif_image_release(tile_image);
                    tile_image = nullptr;
                    
                    if (error.code != heif_error_Ok) {
                        std::cerr << "  Failed to add tile (" << tx << "," << ty 
                                << ") at level " << level << ": " << error.message << std::endl;
                        
                        // Cleanup
                        if (image_was_padded && working_image != current_level) {
                            heif_image_release(working_image);
                        }
                        if (level_handle) heif_image_handle_release(level_handle);
                        heif_encoding_options_free(options);
                        heif_encoder_release(encoder);
                        heif_context_free(ctx);
                        if (current_level != full_image) heif_image_release(current_level);
                        heif_image_release(full_image);
                        return;
                    }
                    
                    // Progress
                    if ((ty * num_cols + tx + 1) % 50 == 0) {
                        std::cout << "    Progress: " << (ty * num_cols + tx + 1) 
                                << "/" << (num_cols * num_rows) << std::endl;
                    }
                }
            }
        
  
            std::cout << "  ✓ All tiles encoded for level " << level << std::endl;

            // Release padded image if it was created
            if (image_was_padded && working_image != current_level) {
                heif_image_release(working_image);
                working_image = nullptr;
            }
            
        } else {
            // Non-tiled level
            error = heif_context_encode_image(ctx, current_level, encoder, options, &level_handle);
            check_error_simple(error, "encode level");
        }
        
        // Set first level as primary
        if (level == 0) {
            heif_context_set_primary_image(ctx, level_handle);
        }

        // At end of level, release THIS level's encoder
        heif_encoder_release(encoder);
        encoder = nullptr;
        
        pyramid_ids.push_back(heif_image_handle_get_item_id(level_handle));
        heif_image_handle_release(level_handle);
        level_handle = nullptr;
        
        // Prepare next level
        if (level < config.num_levels - 1) {
            std::cout << "  Downsampling for next level..." << std::endl;
            
            heif_image* downsampled = downsample_image(current_level, 2);
            
            // CRITICAL: Release previous level image (except the original full_image)
            if (current_level != full_image) {
                heif_image_release(current_level);
            }
            current_level = downsampled;
            
            std::cout << "  ✓ Next level prepared: " 
                      << heif_image_get_primary_width(current_level) << "x"
                      << heif_image_get_primary_height(current_level) << std::endl;
        }
    }
    
    // Clean up the last level image (if it's not the original)
    if (current_level != full_image) {
        heif_image_release(current_level);
    }
    current_level = nullptr;
    
    // Create pyramid entity group
    std::cout << "\nCreating pyramid entity group..." << std::endl;
    error = heif_context_add_pyramid_entity_group(ctx, pyramid_ids.data(),
                                                  pyramid_ids.size(), nullptr);
    check_error_simple(error, "add pyramid entity group");
    
    // Write to file
    std::cout << "Writing to file..." << std::endl;
    error = heif_context_write_to_file(ctx, output_file);
    check_error_simple(error, "write to file");
    
    std::cout << "\n✓ Successfully encoded pyramid with " << config.num_levels 
              << " levels: " << output_file << std::endl;
    
    // Final cleanup
    heif_encoding_options_free(options);
    // heif_encoder_release(encoder);
    heif_context_free(ctx);
    heif_image_release(full_image);
}
#endif
// Simplified pyramid encoding (backward compatible)
void encode_pyramid(const char* input_file, 
                   const char* output_file,
                   int num_levels,
                   ExtendedCompressionType compression,
                   int quality = 50,
                   int output_bit_depth = 8) {
    PyramidConfig config;
    memset(&config, 0, sizeof(config));
    config.num_levels = num_levels;
    config.use_tiling = false;
    config.compression = compression;
    config.quality = quality;
    config.output_bit_depth = output_bit_depth;
    
    encode_pyramid_with_config(input_file, output_file, config);
}



// Tiled pyramid encoding
void encode_tiled_pyramid(const char* input_file, 
                         const char* output_file,
                         int num_levels,
                         int tile_width,
                         int tile_height,
                         TilingScheme scheme,
                         ExtendedCompressionType compression,
                         int quality = 50,
                         int output_bit_depth = 8,
                         uint16_t padding_value = 0,
                         const OffsetTableConfig& offset_config = OffsetTableConfig()) {
    PyramidConfig config;
    memset(&config, 0, sizeof(config));
    config.num_levels = num_levels;
    config.use_tiling = true;
    config.tiling_scheme = scheme;
    config.tile_width = tile_width;
    config.tile_height = tile_height;
    config.compression = compression;
    config.quality = quality;
    config.output_bit_depth = output_bit_depth;
    config.padding_value = padding_value;
    config.offset_config = offset_config;
    
    encode_pyramid_with_config(input_file, output_file, config);
}
//#endif

std::cout << "Comprehensive pyramid encoding functions defined." << std::endl;

#endif // HEIF_ENCODE_PYRAMID_COMPREHENSIVE_H

## GeoHEIF Encoding Wrapper Functions

In [ ]:
#ifndef HEIF_ENCODE_GEOHEIF_WRAPPERS_H
#define HEIF_ENCODE_GEOHEIF_WRAPPERS_H

// ============================================================================
// GeoHEIF Encoding Wrappers - Automatic GeoTIFF Metadata Extraction
// ============================================================================

// Encode non-tiled GeoHEIF from GeoTIFF
void encode_nontiled_geoheif(
    const char* geotiff_file, 
    const char* output_file,
    ExtendedCompressionType compression,
    int quality = 50,
    int output_bit_depth = 8
) {
    std::cout << "\n=== Encoding Non-Tiled GeoHEIF ===" << std::endl;
    std::cout << "Input GeoTIFF: " << geotiff_file << std::endl;
    std::cout << "Output GeoHEIF: " << output_file << std::endl;
    
    // Step 1: Encode as regular HEIF
    std::cout << "\n[1/2] Encoding HEIF..." << std::endl;
    encode_nontiled(geotiff_file, output_file, compression, quality, output_bit_depth);
    
    // Step 2: Add GeoTIFF metadata (in-place modification)
    std::cout << "\n[2/2] Adding GeoTIFF metadata..." << std::endl;
    if (!geoheif::GeoTIFFToGeoHEIF::add_geotiff_metadata_to_heif(
            geotiff_file, output_file, nullptr)) {
        std::cerr << "⚠ Warning: Failed to add GeoHEIF metadata, but HEIF was created" << std::endl;
    }
}

// Encode tiled GeoHEIF from GeoTIFF
void encode_tiled_geoheif(
    const char* geotiff_file, 
    const char* output_file,
    int tile_width, 
    int tile_height,
    TilingScheme scheme,
    ExtendedCompressionType compression,
    int quality = 50,
    int output_bit_depth = 8,
    uint16_t padding_value = 0,
    const OffsetTableConfig& offset_config = OffsetTableConfig()
) {
    std::cout << "\n=== Encoding Tiled GeoHEIF ===" << std::endl;
    std::cout << "Input GeoTIFF: " << geotiff_file << std::endl;
    std::cout << "Output GeoHEIF: " << output_file << std::endl;
    std::cout << "Tiling: " << get_tiling_scheme_name(scheme) 
              << " (" << tile_width << "x" << tile_height << ")" << std::endl;
    
    // Step 1: Encode as regular tiled HEIF
    std::cout << "\n[1/2] Encoding tiled HEIF..." << std::endl;
    encode_tiled(geotiff_file, output_file, tile_width, tile_height, scheme,
                compression, quality, output_bit_depth, padding_value, offset_config);
    
    // Step 2: Add GeoTIFF metadata (in-place modification)
    std::cout << "\n[2/2] Adding GeoTIFF metadata..." << std::endl;
    if (!geoheif::GeoTIFFToGeoHEIF::add_geotiff_metadata_to_heif(
            geotiff_file, output_file, nullptr)) {
        std::cerr << "⚠ Warning: Failed to add GeoHEIF metadata, but HEIF was created" << std::endl;
    }
}

// Encode pyramid GeoHEIF from GeoTIFF
void encode_pyramid_geoheif(
    const char* geotiff_file, 
    const char* output_file,
    const PyramidConfig& config
) {
    std::cout << "\n=== Encoding Pyramid GeoHEIF ===" << std::endl;
    std::cout << "Input GeoTIFF: " << geotiff_file << std::endl;
    std::cout << "Output GeoHEIF: " << output_file << std::endl;
    std::cout << "Levels: " << config.num_levels << std::endl;
    
    // Step 1: Encode as regular pyramid HEIF
    std::cout << "\n[1/2] Encoding pyramid HEIF..." << std::endl;
    encode_pyramid_with_config(geotiff_file, output_file, config);
    
    // Step 2: Add GeoTIFF metadata (in-place modification)
    std::cout << "\n[2/2] Adding GeoTIFF metadata..." << std::endl;
    if (!geoheif::GeoTIFFToGeoHEIF::add_geotiff_metadata_to_heif(
            geotiff_file, output_file, nullptr)) {
        std::cerr << "⚠ Warning: Failed to add GeoHEIF metadata, but HEIF was created" << std::endl;
    }
}

// Convenience wrapper for simple pyramid encoding
void encode_pyramid_geoheif(
    const char* geotiff_file, 
    const char* output_file,
    int num_levels,
    ExtendedCompressionType compression,
    int quality = 50,
    int output_bit_depth = 8
) {
    PyramidConfig config;
    config.num_levels = num_levels;
    config.use_tiling = false;
    config.compression = compression;
    config.quality = quality;
    config.output_bit_depth = output_bit_depth;
    
    encode_pyramid_geoheif(geotiff_file, output_file, config);
}

// Convenience wrapper for tiled pyramid encoding
void encode_tiled_pyramid_geoheif(
    const char* geotiff_file, 
    const char* output_file,
    int num_levels,
    int tile_width,
    int tile_height,
    TilingScheme scheme,
    ExtendedCompressionType compression,
    int quality = 50,
    int output_bit_depth = 8,
    uint16_t padding_value = 0,
    const OffsetTableConfig& offset_config = OffsetTableConfig()
) {
    PyramidConfig config;
    config.num_levels = num_levels;
    config.use_tiling = true;
    config.tiling_scheme = scheme;
    config.tile_width = tile_width;
    config.tile_height = tile_height;
    config.compression = compression;
    config.quality = quality;
    config.output_bit_depth = output_bit_depth;
    config.padding_value = padding_value;
    config.offset_config = offset_config;
    
    encode_pyramid_geoheif(geotiff_file, output_file, config);
}

std::cout << "GeoHEIF encoding wrappers defined." << std::endl;

#endif // HEIF_ENCODE_GEOHEIF_WRAPPERS_H

## Test All Tiling Schemes and Pyramid Options

In [ ]:
#ifndef HEIF_TEST_TILING_H
#define HEIF_TEST_TILING_H

void test_all_tiling_schemes(const char* input_file, const char* output_dir = ".") {
    std::cout << "\n" << std::string(80, '=') << std::endl;
    std::cout << "Testing All Tiling Schemes" << std::endl;
    std::cout << std::string(80, '=') << "\n" << std::endl;
    std::string outputFileStr = (fs::path(output_dir) / "test_all_tiling_schemes_output.txt").string();
    const char* output_file = outputFileStr.c_str();

    try {
        // Test Grid tiling with different compressions
        std::cout << "\n1. Grid Tiling Tests:" << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_grid_512_hevc_70.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled(input_file, output_file, 512, 512,
                    TilingScheme::Grid, ExtendedCompressionType::HEVC, 70);
        outputFileStr = (fs::path(output_dir) / "output_grid_512_av1_60.avif").string();
        output_file = outputFileStr.c_str();
        encode_tiled(input_file, output_file, 512, 512,
                    TilingScheme::Grid, ExtendedCompressionType::AV1, 60);
        outputFileStr = (fs::path(output_dir) / "output_grid_512_deflate.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled(input_file, output_file, 512, 512,
                    TilingScheme::Grid, ExtendedCompressionType::DEFLATE);
        
#ifdef WITH_UNCOMPRESSED_CODEC
        // Test UNCI tiling
        std::cout << "\n2. UNCI Tiling Tests:" << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_unci_512_deflate.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled(input_file, output_file, 512, 512,
                    TilingScheme::UNCI, ExtendedCompressionType::DEFLATE);
        outputFileStr = (fs::path(output_dir) / "output_unci_512_zlib.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled(input_file, output_file, 512, 512,
                    TilingScheme::UNCI, ExtendedCompressionType::ZLIB);
#endif
        
#ifdef HEIF_ENABLE_EXPERIMENTAL_FEATURES
        // Test TILI tiling
        std::cout << "\n3. TILI Tiling Tests:" << std::endl;
        outputFileStr = fs::path(output_dir) / "output_tili_512_hevc_70.heif";
        output_file = outputFileStr.c_str();
        encode_tiled(input_file, output_file, 512, 512,
                    TilingScheme::TILI, ExtendedCompressionType::HEVC, 70);

        // Note: AV1 with TILI may not be supported in all implementations. Encoder reported errors.
        // outputFileStr = fs::path(output_dir) / "output_tili_512_av1_60.avif";
        // output_file = outputFileStr.c_str();
        // encode_tiled(input_file, output_file, 512, 512,
        //            TilingScheme::TILI, ExtendedCompressionType::AV1, 60);

        // Test with DEFLATE (should work)
        std::cout << "\n   Testing TILI with DEFLATE (lossless)..." << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_tili_512_deflate.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled(input_file, output_file, 512, 512,
                    TilingScheme::TILI, ExtendedCompressionType::DEFLATE);
        
        // Test with uncompressed (should work)
        std::cout << "\n   Testing TILI with uncompressed..." << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_tili_512_uncompressed.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled(input_file, output_file, 512, 512,
                    TilingScheme::TILI, ExtendedCompressionType::Uncompressed);
        
        // Test pyramids
        std::cout << "\n4. Pyramid Tests:" << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_pyramid_3levels_hevc_70.heif").string();
        output_file = outputFileStr.c_str();
        encode_pyramid(input_file, output_file, 3,
                      ExtendedCompressionType::HEVC, 70);
        outputFileStr = (fs::path(output_dir) / "output_pyramid_5levels_av1_60.avif").string();
        output_file = outputFileStr.c_str();
        encode_pyramid(input_file, output_file, 5,
                      ExtendedCompressionType::AV1, 60);
        
        // Test tiled pyramids
        std::cout << "\n5. Tiled Pyramid Tests:" << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_tiled_pyramid_4levels_grid_hevc_70.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled_pyramid(input_file, output_file,
                           4, 512, 512, TilingScheme::Grid,
                           ExtendedCompressionType::HEVC, 70);
        outputFileStr = (fs::path(output_dir) / "output_tiled_pyramid_4levels_tili_deflate.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled_pyramid(input_file, output_file,
                           4, 512, 512, TilingScheme::TILI,
                           ExtendedCompressionType::DEFLATE);
#endif
        
        std::cout << "\n" << std::string(80, '=') << std::endl;
        std::cout << "✓ All tiling scheme tests completed!" << std::endl;
        std::cout << std::string(80, '=') << "\n" << std::endl;
        
    } catch (const std::exception& e) {
        std::cerr << "Error during testing: " << e.what() << std::endl;
    }
}

#endif // HEIF_TEST_TILING_H

std::cout << "\nTest functions defined." << std::endl;
std::cout << "To test all schemes, call: test_all_tiling_schemes(\"input.tif\", \"output_dir\");" << std::endl;


## Demo and Example Functions (Fixed with proper namespace)

In [ ]:
#ifndef HEIF_DEMO_FUNCTIONS_H
#define HEIF_DEMO_FUNCTIONS_H

void demo_all_compressions(const char* input_file, const char* output_dir) {
    std::cout << "\n" << std::string(80, '=') << std::endl;
    std::cout << "HEIF Encoding Demo - All Compression Methods" << std::endl;
    std::cout << std::string(80, '=') << "\n" << std::endl;
    std::string outputFileStr= (fs::path(output_dir) / "output_all_compressions.heif").string();
    char const* output_file = outputFileStr.c_str();
    
    try {
        // Non-tiled, various compressions
        outputFileStr = (fs::path(output_dir) / "output_nontiled_none.heif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::None);
        outputFileStr = (fs::path(output_dir) / "output_nontiled_hevc_q50.heif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::HEVC, 50);
        outputFileStr = (fs::path(output_dir) / "output_nontiled_hevc_q80.heif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::HEVC, 80);
        outputFileStr = (fs::path(output_dir) / "output_nontiled_av1.avif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::AV1, 60);
        outputFileStr = (fs::path(output_dir) / "output_nontiled_deflate.heif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::DEFLATE);
        outputFileStr = (fs::path(output_dir) / "output_nontiled_zlib.heif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::ZLIB);
        outputFileStr = (fs::path(output_dir) / "output_nontiled_brotli.heif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::Brotli);
        
        // Tiled with different compressions
        outputFileStr = (fs::path(output_dir) / "output_tiled_hevc.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled_grid(input_file, output_file, 512, 512, ExtendedCompressionType::HEVC, 60);
        outputFileStr = (fs::path(output_dir) / "output_tiled_av1.avif").string();
        output_file = outputFileStr.c_str();
        encode_tiled_grid(input_file, output_file, 512, 512, ExtendedCompressionType::AV1, 60);
        outputFileStr = (fs::path(output_dir) / "output_tiled_deflate.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled_grid(input_file, output_file, 512, 512, ExtendedCompressionType::DEFLATE);
        
#ifdef HEIF_ENABLE_EXPERIMENTAL_FEATURES
        // Pyramid
        outputFileStr = (fs::path(output_dir) / "output_pyramid_hevc.heif").string();
        output_file = outputFileStr.c_str();
        encode_pyramid(input_file, output_file, 4, ExtendedCompressionType::HEVC, 60);
        outputFileStr = (fs::path(output_dir) / "output_pyramid_deflate.heif").string();
        output_file = outputFileStr.c_str();
        encode_pyramid(input_file, output_file, 4, ExtendedCompressionType::DEFLATE);
#endif
        
        std::cout << "\n" << std::string(80, '=') << std::endl;
        std::cout << "All encodings completed successfully!" << std::endl;
        std::cout << std::string(80, '=') << "\n" << std::endl;
        
    } catch (const std::exception& e) {
        std::cerr << "Error during encoding: " << e.what() << std::endl;
    }
}

// Simplified demo with just a few formats
void demo_basic_encoding(const char* input_file, const char* output_dir) {
    std::cout << "\n=== Basic Encoding Demo ===" << std::endl;
    std::string outputFileStr = (fs::path(output_dir) / "output_basic.heif").string();
    const char* output_file = outputFileStr.c_str();

    try {
        // Test a few key formats
        std::cout << "\n1. Encoding with HEVC (quality 70)..." << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_hevc_70.heif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::HEVC, 70);
        
        std::cout << "\n2. Encoding with AV1 (quality 60)..." << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_av1_60.avif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::AV1, 60);
        
        std::cout << "\n3. Encoding with DEFLATE (lossless)..." << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_deflate.heif").string();
        output_file = outputFileStr.c_str();
        encode_nontiled(input_file, output_file, ExtendedCompressionType::DEFLATE);
        
        std::cout << "\n4. Encoding tiled image with HEVC..." << std::endl;
        outputFileStr = (fs::path(output_dir) / "output_tiled_512_hevc_70.heif").string();
        output_file = outputFileStr.c_str();
        encode_tiled_grid(input_file, output_file, 512, 512, ExtendedCompressionType::HEVC, 70);
        
        std::cout << "\n✓ Basic encoding demo completed successfully!" << std::endl;
        
    } catch (const std::exception& e) {
        std::cerr << "Error during encoding: " << e.what() << std::endl;
    }
}

// Example: Encode with specific compression type
void encode_example(const char* input_file, 
                   const char* output_file,
                   ExtendedCompressionType compression = ExtendedCompressionType::HEVC,
                   int quality = 70) {
    // Use the helper function to get compression info
    auto compression_name = get_compression_name(compression);
    bool lossless = is_lossless_compression(compression);
    
    std::cout << "\n=== Encoding Example ===" << std::endl;
    std::cout << "Input: " << input_file << std::endl;
    std::cout << "Output: " << output_file << std::endl;
    std::cout << "Compression: " << compression_name << std::endl;
    if (!lossless) {
        std::cout << "Quality: " << quality << std::endl;
    } else {
        std::cout << "Mode: Lossless" << std::endl;
    }
    
    try {
        encode_nontiled(input_file, output_file, compression, quality);
        std::cout << "✓ Encoding completed successfully!" << std::endl;
    } catch (const std::exception& e) {
        std::cerr << "Error: " << e.what() << std::endl;
    }
}

// Print compression info for a specific type
void print_compression_info(ExtendedCompressionType type) {
    const CompressionInfo* info = get_compression_info(type);
    if (!info) {
        std::cout << "Unknown compression type" << std::endl;
        return;
    }
    
    std::cout << "\nCompression Info:" << std::endl;
    std::cout << "  Name: " << info->name << std::endl;
    std::cout << "  Description: " << info->description << std::endl;
    std::cout << "  Lossless: " << (info->lossless ? "Yes" : "No") << std::endl;
    std::cout << "  Supported by libheif: " << (info->supported_by_libheif ? "Yes" : "No") << std::endl;
    std::cout << "  Supported by GDAL: " << (info->supported_by_gdal ? "Yes" : "No") << std::endl;
}

std::cout << "Demo functions defined." << std::endl;
std::cout << "\nAvailable demo functions:" << std::endl;
std::cout << "  - demo_all_compressions(input_file)" << std::endl;
std::cout << "  - demo_basic_encoding(input_file)" << std::endl;
std::cout << "  - encode_example(input_file, output_file, compression, quality)" << std::endl;
std::cout << "  - print_compression_info(compression_type)" << std::endl;

#endif // HEIF_DEMO_FUNCTIONS_H

## Usage Examples

In [ ]:
#ifndef HEIF_USAGE_EXAMPLES_H
#define HEIF_USAGE_EXAMPLES_H

void show_usage_examples() {
    std::cout << "\n" << std::string(80, '=') << std::endl;
    std::cout << "HEIF Encoder Usage Examples" << std::endl;
    std::cout << std::string(80, '=') << "\n" << std::endl;
    
    std::cout << "1. Basic encoding with HEVC:" << std::endl;
    std::cout << "   encode_example(\"input.tif\", \"output.heif\", " << std::endl;
    std::cout << "                  ExtendedCompressionType::HEVC, 70);" << std::endl;
    std::cout << std::endl;
    
    std::cout << "2. Encoding with AV1 (AVIF):" << std::endl;
    std::cout << "   encode_example(\"input.tif\", \"output.avif\", " << std::endl;
    std::cout << "                  ExtendedCompressionType::AV1, 60);" << std::endl;
    std::cout << std::endl;
    
    std::cout << "3. Lossless encoding with DEFLATE:" << std::endl;
    std::cout << "   encode_nontiled(\"input.tif\", \"output.heif\", " << std::endl;
    std::cout << "                   ExtendedCompressionType::DEFLATE);" << std::endl;
    std::cout << std::endl;
    
    std::cout << "4. Tiled encoding:" << std::endl;
    std::cout << "   encode_tiled_grid(\"input.tif\", \"output.heif\", " << std::endl;
    std::cout << "                     512, 512,  // tile size" << std::endl;
    std::cout << "                     ExtendedCompressionType::HEVC, 70);" << std::endl;
    std::cout << std::endl;
    
#ifdef HEIF_ENABLE_EXPERIMENTAL_FEATURES
    std::cout << "5. Pyramid/multi-resolution encoding:" << std::endl;
    std::cout << "   encode_pyramid(\"input.tif\", \"output.heif\", " << std::endl;
    std::cout << "                  4,  // number of levels" << std::endl;
    std::cout << "                  ExtendedCompressionType::HEVC, 70);" << std::endl;
    std::cout << std::endl;
#endif
    
    std::cout << "6. Run all compression demos:" << std::endl;
    std::cout << "   demo_all_compressions(\"input.tif\", \"output_dir\");" << std::endl;
    std::cout << std::endl;
    
    std::cout << "7. Run basic demo:" << std::endl;
    std::cout << "   demo_basic_encoding(\"input.tif\", \"output_dir\");" << std::endl;
    std::cout << std::endl;
    
    std::cout << "Available compression types:" << std::endl;
    std::cout << "  - ExtendedCompressionType::None" << std::endl;
    std::cout << "  - ExtendedCompressionType::HEVC" << std::endl;
    std::cout << "  - ExtendedCompressionType::AV1" << std::endl;
    std::cout << "  - ExtendedCompressionType::VVC" << std::endl;
    std::cout << "  - ExtendedCompressionType::AVC" << std::endl;
    std::cout << "  - ExtendedCompressionType::JPEG" << std::endl;
    std::cout << "  - ExtendedCompressionType::JPEG2000" << std::endl;
    std::cout << "  - ExtendedCompressionType::HTJ2K" << std::endl;
    std::cout << "  - ExtendedCompressionType::DEFLATE" << std::endl;
    std::cout << "  - ExtendedCompressionType::ZLIB" << std::endl;
    std::cout << "  - ExtendedCompressionType::Brotli" << std::endl;
    std::cout << "  - ExtendedCompressionType::ZSTD" << std::endl;
    std::cout << "  - ExtendedCompressionType::LZW" << std::endl;
    std::cout << "  - ExtendedCompressionType::WebP" << std::endl;
    std::cout << "  - ExtendedCompressionType::LERC" << std::endl;
    std::cout << "  - ExtendedCompressionType::PackBits" << std::endl;
    
    std::cout << "\n" << std::string(80, '=') << std::endl;
}

// show_usage_examples();

#endif // HEIF_USAGE_EXAMPLES_H

In [ ]:
// show_usage_examples();

In [ ]:
// // Use default padding (0)
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE);

// // Specify custom padding value (e.g., 255 for white)
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 8, 255);

// // Pyramid with custom padding
// encode_tiled_pyramid(input_file, output_file, 5, 256, 256,
//                     TilingScheme::TILI, ExtendedCompressionType::DEFLATE,
//                     50, 8, 128);  // Pad with mid-gray (128)
// 8-bit data with default padding (0)
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 8, 0);

// // 8-bit data with white padding (255)
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 8, 255);

// // 10-bit data with max value padding (1023 = 2^10 - 1)
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 10, 1023);

// // 12-bit data with mid-gray padding (2048 = 2^11)
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 12, 2048);

// // 16-bit data with custom padding (32768)
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 16, 32768);

// // Pyramid with 12-bit data and black padding
// encode_tiled_pyramid(input_file, output_file, 5, 256, 256,
//                     TilingScheme::TILI, ExtendedCompressionType::DEFLATE,
//                     50, 12, 0);


// // Example 1: Default offset configuration (offset + size)
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE);

// // Example 2: Using preset - offset and size tables
// OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetAndSize);
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 8, 0, config1);

// // Example 3: Using preset - offset only
// OffsetTableConfig config2(OffsetTableConfig::Preset::OffsetOnly);
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 8, 0, config2);

// // Example 4: Using preset - size only
// OffsetTableConfig config3(OffsetTableConfig::Preset::SizeOnly);
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 8, 0, config3);

// // Example 5: Using preset - sequential (no offset table)
// OffsetTableConfig config4(OffsetTableConfig::Preset::Sequential);
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 8, 0, config4);

// // Example 6: Custom offset configuration - 40-bit offset, 32-bit size
// OffsetTableConfig config5(40, 32, false);
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 8, 0, config5);

// // Example 7: Custom for 64-bit offsets (large files)
// OffsetTableConfig config6(64, 32, false);
// encode_tiled_tili(input_file, output_file, 256, 256,
//                   ExtendedCompressionType::DEFLATE, 50, 8, 0, config6);

// // Example 8: Pyramid with custom offset configuration
// OffsetTableConfig pyramid_config(OffsetTableConfig::Preset::OffsetAndSize);
// encode_tiled_pyramid(input_file, output_file, 5, 256, 256,
//                     TilingScheme::TILI, ExtendedCompressionType::DEFLATE,
//                     50, 8, 0, pyramid_config);

// // Example 9: Compare different configurations
// list_offset_table_presets();

// // Example 10: Test all three schemes with same offset config
// OffsetTableConfig test_config(32, 24, false);

// encode_tiled(input_file, "output_grid.heif", 256, 256,
//             TilingScheme::Grid, ExtendedCompressionType::DEFLATE,
//             50, 8, 0, test_config);

// encode_tiled(input_file, "output_unci.heif", 256, 256,
//             TilingScheme::UNCI, ExtendedCompressionType::DEFLATE,
//             50, 8, 0, test_config);

// encode_tiled(input_file, "output_tili.heif", 256, 256,
//             TilingScheme::TILI, ExtendedCompressionType::DEFLATE,
//             50, 8, 0, test_config);

## GeoHEIF Usage Example

In [ ]:
#ifndef GEOHEIF_ENCODING_EXAMPLES_H
#define GEOHEIF_ENCODING_EXAMPLES_H

// Example 1: Simple non-tiled GeoHEIF
void example_geoheif_simple(const char* geotiff_file, const char* output_dir) {
    std::cout << "\n╔════════════════════════════════════════════════════════════════╗" << std::endl;
    std::cout << "║  Example 1: Simple GeoHEIF Encoding                           ║" << std::endl;
    std::cout << "╚════════════════════════════════════════════════════════════════╝" << std::endl;
    
    std::string output = std::string(output_dir) + "/geoheif_simple_hevc70.heif";
    
    encode_nontiled_geoheif(geotiff_file, output.c_str(),
                           ExtendedCompressionType::HEVC, 70);
    
    // Verify result
    geoheif::GeoHEIFInfo info;
    if (geoheif::GeoHEIFInspector::inspect(output.c_str(), info)) {
        std::cout << "\n📊 Verification:" << std::endl;
        std::cout << "   Has ogeo brand: " << (info.has_ogeo_brand ? "✓" : "✗") << std::endl;
        std::cout << "   Has CRS: " << (info.crs.present ? "✓" : "✗") << std::endl;
        std::cout << "   Has georeference: " << (info.georeference.has_bbox ? "✓" : "✗") << std::endl;
    }
}

// Example 2: Tiled GeoHEIF with TILI
void example_geoheif_tiled(const char* geotiff_file, const char* output_dir) {
    std::cout << "\n╔════════════════════════════════════════════════════════════════╗" << std::endl;
    std::cout << "║  Example 2: Tiled GeoHEIF (TILI + DEFLATE)                    ║" << std::endl;
    std::cout << "╚════════════════════════════════════════════════════════════════╝" << std::endl;
    
    std::string output = std::string(output_dir) + "/geoheif_tiled_256_deflate.heif";
    
    encode_tiled_geoheif(geotiff_file, output.c_str(),
                        256, 256, TilingScheme::TILI,
                        ExtendedCompressionType::DEFLATE);
    
    // Verify result
    geoheif::GeoHEIFInfo info;
    if (geoheif::GeoHEIFInspector::inspect(output.c_str(), info)) {
        geoheif::GeoHEIFInspector::print_report(info);
    }
}

// Example 3: Pyramid GeoHEIF
void example_geoheif_pyramid(const char* geotiff_file, const char* output_dir) {
    std::cout << "\n╔════════════════════════════════════════════════════════════════╗" << std::endl;
    std::cout << "║  Example 3: Pyramid GeoHEIF (5 levels, tiled)                 ║" << std::endl;
    std::cout << "╚════════════════════════════════════════════════════════════════╝" << std::endl;
    
    std::string output = std::string(output_dir) + "/geoheif_pyramid_5lvl_tiled.heif";
    
    encode_tiled_pyramid_geoheif(geotiff_file, output.c_str(),
                                5, 256, 256, TilingScheme::TILI,
                                ExtendedCompressionType::DEFLATE);
    
    // Verify result
    geoheif::GeoHEIFInfo info;
    if (geoheif::GeoHEIFInspector::inspect(output.c_str(), info)) {
        std::cout << "\n📊 Pyramid Information:" << std::endl;
        std::cout << "   Is pyramid: " << (info.is_pyramid ? "✓" : "✗") << std::endl;
        std::cout << "   Number of levels: " << info.pyramid_levels.size() << std::endl;
        
        for (const auto& level : info.pyramid_levels) {
            std::cout << "   Level " << level.level_index << ": " 
                      << level.width << "x" << level.height << std::endl;
        }
    }
}

// Example 4: Add metadata to existing HEIF
void example_add_metadata_to_existing(const char* geotiff_file, const char* heif_file, 
                                      const char* output_dir) {
    std::cout << "\n╔════════════════════════════════════════════════════════════════╗" << std::endl;
    std::cout << "║  Example 4: Add Metadata to Existing HEIF                     ║" << std::endl;
    std::cout << "╚════════════════════════════════════════════════════════════════╝" << std::endl;
    
    std::string output = std::string(output_dir) + "/existing_heif_with_geo.heif";
    
    geoheif::GeoTIFFToGeoHEIF::add_geotiff_metadata_to_heif(
        geotiff_file, heif_file, output.c_str()
    );
    
    // Verify result
    geoheif::GeoHEIFInfo info;
    if (geoheif::GeoHEIFInspector::inspect(output.c_str(), info)) {
        geoheif::GeoHEIFInspector::print_report(info);
    }
}

#endif // GEOHEIF_ENCODING_EXAMPLES_H

## GeoHEIF Encoding Function Use Patterns

In [ ]:
// // Direct use of the unified method
// geoheif::GeoTIFFToGeoHEIF::add_geotiff_metadata_to_heif(
//     "input.tif",     // Source GeoTIFF
//     "image.heif",    // Existing HEIF
//     nullptr          // Modify in-place (or specify output path)
// );

// // Or use convenient wrappers
// encode_nontiled_geoheif("input.tif", "output.heif", 
//                         ExtendedCompressionType::HEVC, 70);

## Helper Function to show offset configuration info

In [ ]:
#ifndef HELPER_FUNCTION_OFFSET_TABLES_H
#define HELPER_FUNCTION_OFFSET_TABLES_H
void demonstrate_offset_configurations(const char* input_file, const char* output_dir) {
    std::cout << "\n=== Demonstrating Offset Table Configurations ===" << std::endl;
    
    list_offset_table_presets_with_support();
    
    std::cout << "\n--- Testing Different Configurations ---" << std::endl;
    
    // Test 1: Offset + Size (standard)
    std::cout << "\n1. Testing Offset + Size configuration:" << std::endl;
    OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetAndSize);
    std::string output1 = std::string(output_dir) + "/tili_offset_and_size.heif";
    encode_tiled_tili(input_file, output1.c_str(), 256, 256,
                     ExtendedCompressionType::DEFLATE, 50, 8, 0, config1);
    
    // Test 2: Offset only
    std::cout << "\n2. Testing Offset-only configuration:" << std::endl;
    OffsetTableConfig config2(OffsetTableConfig::Preset::OffsetOnly);
    std::string output2 = std::string(output_dir) + "/tili_offset_only.heif";
    encode_tiled_tili(input_file, output2.c_str(), 256, 256,
                     ExtendedCompressionType::DEFLATE, 50, 8, 0, config2);
    
    // Test 3: Size only
    std::cout << "\n3. Testing Size-only configuration:" << std::endl;
    OffsetTableConfig config3(OffsetTableConfig::Preset::SizeOnly);
    std::string output3 = std::string(output_dir) + "/tili_size_only.heif";
    encode_tiled_tili(input_file, output3.c_str(), 256, 256,
                     ExtendedCompressionType::DEFLATE, 50, 8, 0, config3);
    
    // Test 4: Sequential
    std::cout << "\n4. Testing Sequential configuration (no offset table):" << std::endl;
    OffsetTableConfig config4(OffsetTableConfig::Preset::Sequential);
    std::string output4 = std::string(output_dir) + "/tili_sequential.heif";
    encode_tiled_tili(input_file, output4.c_str(), 256, 256,
                     ExtendedCompressionType::DEFLATE, 50, 8, 0, config4);
    
    std::cout << "\n✓ All offset configuration tests completed!" << std::endl;
    std::cout << "\nCheck output files to compare file sizes and structures." << std::endl;
}
#endif // HELPER_FUNCTION_OFFSET_TABLES_H

In [ ]:
// list_offset_table_presets();
// demonstrate_offset_configurations("input.tif", "output_dir");

## Test Helper Functions

In [ ]:
#ifndef HEIF_TEST_HELPERS_H
#define HEIF_TEST_HELPERS_H

// Test that helper functions work correctly
void test_helper_functions() {
    std::cout << "\n=== Testing Helper Functions ===" << std::endl;
    
    // Test get_compression_info
    std::cout << "\n1. Testing get_compression_info():" << std::endl;
    print_compression_info(ExtendedCompressionType::HEVC);
    print_compression_info(ExtendedCompressionType::DEFLATE);
    
    // Test get_compression_name
    std::cout << "\n2. Testing get_compression_name():" << std::endl;
    std::cout << "  HEVC: " << get_compression_name(ExtendedCompressionType::HEVC) << std::endl;
    std::cout << "  AV1: " << get_compression_name(ExtendedCompressionType::AV1) << std::endl;
    std::cout << "  DEFLATE: " << get_compression_name(ExtendedCompressionType::DEFLATE) << std::endl;
    
    // Test is_lossless_compression
    std::cout << "\n3. Testing is_lossless_compression():" << std::endl;
    std::cout << "  HEVC is lossless: " << (is_lossless_compression(ExtendedCompressionType::HEVC) ? "Yes" : "No") << std::endl;
    std::cout << "  DEFLATE is lossless: " << (is_lossless_compression(ExtendedCompressionType::DEFLATE) ? "Yes" : "No") << std::endl;
    
    // Test to_heif_compression_format
    std::cout << "\n4. Testing to_heif_compression_format():" << std::endl;
    auto hevc_fmt = to_heif_compression_format(ExtendedCompressionType::HEVC);
    auto av1_fmt = to_heif_compression_format(ExtendedCompressionType::AV1);
    std::cout << "  HEVC format code: " << static_cast<int>(hevc_fmt) << std::endl;
    std::cout << "  AV1 format code: " << static_cast<int>(av1_fmt) << std::endl;
    
    // Test to_unci_compression
    std::cout << "\n5. Testing to_unci_compression():" << std::endl;
    auto deflate_unci = to_unci_compression(ExtendedCompressionType::DEFLATE);
    auto zlib_unci = to_unci_compression(ExtendedCompressionType::ZLIB);
    std::cout << "  DEFLATE unci code: " << static_cast<int>(deflate_unci) << std::endl;
    std::cout << "  ZLIB unci code: " << static_cast<int>(zlib_unci) << std::endl;
    
    std::cout << "\n✓ All helper function tests passed" << std::endl;
}

// test_helper_functions();

#endif // HEIF_TEST_HELPERS_H

In [ ]:
// test_helper_functions();


# Working POINT 2

## List Available Features

### Detect TILI Support at Runtime

In [ ]:
#ifndef HEIF_TILI_RUNTIME_CHECK_H
#define HEIF_TILI_RUNTIME_CHECK_H

bool is_tili_actually_supported() {
#ifdef HEIF_ENABLE_EXPERIMENTAL_FEATURES
    // Create a minimal test context to check if TILI is available
    heif_context* test_ctx = heif_context_alloc();
    
    heif_tiled_image_parameters test_params;
    memset(&test_params, 0, sizeof(test_params));
    test_params.version = 1;
    test_params.image_width = 512;
    test_params.image_height = 512;
    test_params.tile_width = 256;
    test_params.tile_height = 256;
    //test_params.offset_field_length = 32;
    //test_params.size_field_length = 24;
    //test_params.tiles_are_sequential = 1;
    
    // Get a dummy encoder
    heif_encoder* test_encoder = nullptr;
    heif_error error = heif_context_get_encoder_for_format(
        test_ctx, heif_compression_HEVC, &test_encoder);
    
    if (error.code != heif_error_Ok) {
        heif_context_free(test_ctx);
        std::cout << "TILI check: Cannot get encoder" << std::endl;
        return false;
    }
    
    // Try to create TILI image
    heif_image_handle* test_handle = nullptr;
    error = heif_context_add_tiled_image(test_ctx, &test_params, nullptr,
                                        test_encoder, &test_handle);
    
    bool supported = (error.code == heif_error_Ok || 
                     error.code == heif_error_Usage_error); // Usage error is OK, means API exists
    
    // Check for specific "unsupported" errors
    if (error.code == heif_error_Unsupported_filetype ||
        error.code == heif_error_Unsupported_feature) {
        supported = false;
    }
    
    if (test_handle) {
        heif_image_handle_release(test_handle);
    }
    heif_encoder_release(test_encoder);
    heif_context_free(test_ctx);
    
    if (!supported) {
        std::cout << "TILI check: " << error.message << std::endl;
        std::cout << "TILI is NOT supported in this libheif build" << std::endl;
    } else {
        std::cout << "TILI check: TILI appears to be supported" << std::endl;
    }
    
    return supported;
#else
    std::cout << "TILI check: Experimental features not enabled" << std::endl;
    return false;
#endif
}

// Cache the result
inline bool check_tili_support_cached() {
    static bool checked = false;
    static bool supported = false;
    
    if (!checked) {
        std::cout << "\n=== Checking TILI Support ===" << std::endl;
        supported = is_tili_actually_supported();
        checked = true;
        std::cout << "TILI Support: " << (supported ? "YES ✓" : "NO ✗") << std::endl;
        std::cout << std::string(40, '=') << "\n" << std::endl;
    }
    
    return supported;
}

#endif // HEIF_TILI_RUNTIME_CHECK_H

In [ ]:
// if (!check_tili_support_cached()) {
//     std::cout << "TILI not supported in this libheif build." << std::endl;
//     std::cout << "Falling back to Grid scheme..." << std::endl;
// }
// else {
//     std::cout << "TILI is supported. Using TILI tiling scheme." << std::endl;
// } 

### Compressions

In [ ]:
// // Display all available compression algorithms
// list_all_compression_algorithms();

// // Display available encoders
// std::cout << "\n=== Available Encoders ===\n" << std::endl;
// list_encoders(heif_compression_HEVC);
// list_encoders(heif_compression_AV1);
// list_encoders(heif_compression_uncompressed);

### Tiling Schemes

In [ ]:
// List capabilities
list_tiling_schemes();


## Quick Start Example

Uncomment and modify the following to encode your GeoTIFF:

In [ ]:
{
std::cout << "Quick start example:" << std::endl;
// define some variables for input and output paths
std::string inputFileStr = (fs::current_path() / "srcdata" / "ACT2017.tiff").string();
std::string outputDirStr = (fs::current_path() / "benchmark_output").string();
std::string outputFileStr = (fs::current_path() / "benchmark_output" / "output.heif").string();
const char* input_file = inputFileStr.c_str();
const char* output_file = outputFileStr.c_str();
const char* output_dir = outputDirStr.c_str();

if (!fs::exists(std::string(input_file))) {
    std::cout << "ERROR: " << std::string(input_file) << " Path not found.\n";
} else {
    std::cout << std::string(input_file) << " File or directory exists!\n";

// Uncomment and set your input file path:
// const char* input_file = "path/to/your/input.tif";
//std::cout << "Current working directory: " << fs::current_path() << std::endl;
//std::cout << "Looking for input file in: " << (fs::current_path() / "srcdata" / "ACT2017.tiff") << std::endl;
// 1. Store the string in a local variable first
//fs::path myPath = fs::current_path() / "srcdata" / "ACT2017.tiff";
//std::string pathStr = myPath.string(); 

// 2. Get the pointer from that variable
//const char* cStr = pathStr.c_str();

//std::cout << "Path as C-string: " << cStr << std::endl;


// Example 1: Single image with HEVC
// encode_example(input_file, "output.heif", ExtendedCompressionType::HEVC, 70);
//outputFileStr = (fs::current_path() / "benchmark_output" / "output_hevc_70.heif").string();
//std::cout << "Output file path: " << outputFileStr << std::endl;
//output_file = outputFileStr.c_str();
//std::cout << "Output file pointer: " << output_file << std::endl;
//encode_example(input_file, output_file, ExtendedCompressionType::HEVC, 70);

// Example 2: Single image with AV1
// encode_example(input_file, "output.avif", ExtendedCompressionType::AV1, 60);
//outputFileStr = (fs::current_path() / "benchmark_output" / "output_av1_60.heif").string();
//std::cout << "Output file path: " << outputFileStr << std::endl;
//output_file = outputFileStr.c_str();
//std::cout << "Output file pointer: " << output_file << std::endl;
//encode_example(input_file, output_file, ExtendedCompressionType::AV1, 60);

// Example 3: Lossless with DEFLATE
// encode_nontiled(input_file, "output_lossless.heif", ExtendedCompressionType::DEFLATE);
//outputFileStr = (fs::current_path() / "benchmark_output" / "output_lossless_deflate.heif").string();
//std::cout << "Output file path: " << outputFileStr << std::endl;
//output_file = outputFileStr.c_str();
//std::cout << "Output file pointer: " << output_file << std::endl;
//encode_nontiled(input_file, output_file, ExtendedCompressionType::DEFLATE);

// Example 4: Tiled image
// encode_tiled_grid(input_file, "output_tiled.heif", 512, 512, 
//                   ExtendedCompressionType::HEVC, 70);
//outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_512_hevc_70.heif").string();
//std::cout << "Output file path: " << outputFileStr << std::endl;
//output_file = outputFileStr.c_str();
//std::cout << "Output file pointer: " << output_file << std::endl;
//encode_tiled_grid(input_file, output_file, 512, 512, 
//                  ExtendedCompressionType::HEVC, 70);
//outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_256_deflate_100.heif").string();
//std::cout << "Output file path: " << outputFileStr << std::endl;
//output_file = outputFileStr.c_str();
//std::cout << "Output file pointer: " << output_file << std::endl;
//encode_tiled_grid(input_file, output_file, 256, 256, 
//                  ExtendedCompressionType::DEFLATE, 100);

// Example 5: Run basic demo (creates multiple output files)
// demo_basic_encoding(input_file, output_dir);
// demo_basic_encoding(input_file, output_dir);

// Example 6: Run all compression demos (creates many output files)
// demo_all_compressions(input_file, output_dir);
// demo_all_compressions(input_file, output_dir);

// Example 7: Test all tiling schemes
// test_all_tiling_schemes(input_file, output_dir);
// test_all_tiling_schemes(input_file, output_dir);


// Example 8: Create image with unci tiling scheme 
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_unci_256_deflate_100.heif").string();
// std::cout << "Output file path: " << outputFileStr << std::endl;
// output_file = outputFileStr.c_str();
// std::cout << "Output file pointer: " << output_file << std::endl;
// encode_tiled_unci(input_file, output_file, 256, 256, 
//                  ExtendedCompressionType::DEFLATE, 100);

// Example 9: Create image with tili tiling scheme (experimental, may not be supported by all decoders)
//outputFileStr =  (fs::current_path() / "benchmark_output" / "output_tili_512_deflate.heif").string();
// std::cout << "Output file path: " << outputFileStr << std::endl;
// output_file = outputFileStr.c_str();
// encode_tiled_tili(input_file, output_file, 512, 512,
//              ExtendedCompressionType::DEFLATE);

// Exmapele 10: Create image with tili tiling scheme using ecnode_tiled and specifying TILI scheme
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tili_512_deflate2.heif").string();
// std::cout << "Output file path: " << outputFileStr << std::endl;
// output_file = outputFileStr.c_str();
// encode_tiled(input_file, output_file, 512, 512,
//            TilingScheme::TILI, ExtendedCompressionType::DEFLATE);

// Example 11: Create tiled image with tili tiling scheme and HEVC compression using encode_tiled and specifying TILI scheme
// outputFileStr = (fs::current_path() / "benchmark_output" /  "output_tili_512_hevc_70.heif").string();
// output_file = outputFileStr.c_str();
// encode_tiled(input_file, output_file, 512, 512,
//           TilingScheme::TILI, ExtendedCompressionType::HEVC, 70);

// Example 12: Create a tiled with TILI and AV1 compression (may not be supported in all implementations, encoder may report errors)
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tili_512_av1_60.avif").string();
// output_file = outputFileStr.c_str();
// encode_tiled(input_file, output_file, 512, 512,
//           TilingScheme::TILI, ExtendedCompressionType::AV1, 60);

// Example 13: Create a tiled image with TILI tiling scheme and DEFLATE compression (lossless, should work in all implementations that support TILI)
// Test with DEFLATE (should work)
// std::cout << "\n   Testing TILI with DEFLATE (lossless)..." << std::endl;
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tili_512_deflate.heif").string();
// output_file = outputFileStr.c_str();
// encode_tiled(input_file, output_file, 512, 512,
//            TilingScheme::TILI, ExtendedCompressionType::DEFLATE);


// Example 14: Create a tiled image with TILI tiling scheme and uncompressed (lossless, should work in all implementations that support TILI)
// Test with uncompressed (should work)
//std::cout << "\n   Testing TILI with uncompressed..." << std::endl;
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tili_512_uncompressed.heif").string();
// output_file = outputFileStr.c_str();
// encode_tiled(input_file, output_file, 512, 512,
//           TilingScheme::TILI, ExtendedCompressionType::Uncompressed);

// Example 15: Create pyramids with HEVC compression
// Test pyramids
// std::cout << "\n Pyramid Tests:" << std::endl;
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_pyramid_3levels_hevc_70.heif").string();
// output_file = outputFileStr.c_str();
// encode_pyramid(input_file, output_file, 3,
//                 ExtendedCompressionType::HEVC, 70);

// Example 16: Create pyramids with AV1 compression (may not be supported in all implementations, encoder may report errors)
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_pyramid_5levels_av1_60.avif").string();
// output_file = outputFileStr.c_str();
// encode_pyramid(input_file, output_file, 5,
//                 ExtendedCompressionType::AV1, 60);

// Example 17: Create tiled pyramids with different tiling schemes and compressions (some combinations may not be supported, encoder may report errors)
// Test tiled pyramids
// std::cout << "\nTiled Pyramid Tests:" << std::endl;
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_4levels_grid_hevc_70.heif").string();
// output_file = outputFileStr.c_str();
// encode_tiled_pyramid(input_file, output_file,
//                     4, 512, 512, TilingScheme::Grid,
//                     ExtendedCompressionType::HEVC, 70);

// Example 18: Create tiled pyramid with TILI tiling scheme and DEFLATE compression (lossless, should work in all implementations that support TILI)
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_4levels_tili_deflate.heif").string();
// output_file = outputFileStr.c_str();
// encode_tiled_pyramid(input_file, output_file,
//                     4, 512, 512, TilingScheme::TILI,
//                     ExtendedCompressionType::DEFLATE);

// Example 18: Create tiled pyramid with TILI tiling scheme and UNCOMPRESSED) - works after memmory cleaning after each level processing and padding tile to exact same tile size for TILI
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_4levels_tili_uncompressed.heif").string();
// output_file = outputFileStr.c_str();
// encode_tiled_pyramid(input_file, output_file,
//                     4, 512, 512, TilingScheme::TILI,
//                     ExtendedCompressionType::Uncompressed);
// Example 19: Create tiled pyramid with TILI tiling scheme and DEFLATE compression but with smaller tile size (256x256) - works after memmory cleaning after each level processing and padding tile to exact same tile size for TILI
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_5levels_tili_256_deflate.heif").string();
// output_file = outputFileStr.c_str();
// encode_tiled_pyramid(input_file, output_file,
//                      5, 256, 256, TilingScheme::TILI,
//                      ExtendedCompressionType::DEFLATE, 100, 8, 255);

// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_5levels_tili_256_deflate.heif").string();
// output_file = outputFileStr.c_str();
// OffsetTableConfig config1(OffsetTableConfig::Preset::OffsetOnly);
// encode_tiled_pyramid(input_file, output_file,
//                      5, 256, 256, TilingScheme::TILI,
//                      ExtendedCompressionType::DEFLATE, 100, 8, 255,config1);

// Example 20: Create tiled pyramid with UNCI tiling scheme and DEFLATE compression but with smaller tile size (256x256)
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_5levels_unci_256_deflate.heif").string();
// output_file = outputFileStr.c_str();
// encode_tiled_pyramid(input_file, output_file,
//                     5, 256, 256, TilingScheme::UNCI,
//                     ExtendedCompressionType::DEFLATE, 100, 8, 255);

// Example 21: Create tiled pyramid with GRID tiling scheme and DEFLATE compression but with smaller tile size (256x256)
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_tiled_pyramid_5levels_grid_256_deflate.heif").string();
// output_file = outputFileStr.c_str();
// encode_tiled_pyramid(input_file, output_file,
//                     5, 256, 256, TilingScheme::Grid,
//                     ExtendedCompressionType::DEFLATE);


// Example 13: Create a tiled image with TILI tiling scheme and DEFLATE compression (lossless, should work in all implementations that support TILI)
// Test with DEFLATE (should work)
// std::cout << "\n   Testing Grid tiling with DEFLATE (lossless)..." << std::endl;
// outputFileStr = (fs::current_path() / "benchmark_output" / "output_grid_512_deflate.heif").string();
// output_file = outputFileStr.c_str();
// encode_tiled(input_file, output_file, 512, 512,
//            TilingScheme::Grid, ExtendedCompressionType::DEFLATE);

std::cout << "\nReady to encode! Uncomment examples above and set your input file path." << std::endl;
}
}

## GeoHEIF Quick Start Examples

In [ ]:
// Example geoheif-1: Create a tiled pyramid GeoHEIF with TILI tiling scheme and DEFLATE compression but with smaller tile size (256x256) 
{
    const char* geotiff_file = "srcdata/ACT2017.tiff";
    const char* output_dir = "benchmark_output";
    std::string output = std::string(output_dir) + "/geoheif_tiled_pyramid_5levels_tili_256_deflate.heif";
    
    encode_tiled_pyramid_geoheif(geotiff_file, output.c_str(),
                                5, 256, 256, TilingScheme::TILI,
                                ExtendedCompressionType::DEFLATE, 100, 8, 255);
 
}

// Example geoheif-2: Create a tiled pyramid GeoHEIF with GRID tiling scheme and DEFLATE compression but with smaller tile size (256x256) 
{
    const char* geotiff_file = "srcdata/ACT2017.tiff";
    const char* output_dir = "benchmark_output";
    std::string output = std::string(output_dir) + "/geoheif_tiled_pyramid_5levels_grid_256_deflate.heif";
    
    encode_tiled_pyramid_geoheif(geotiff_file, output.c_str(),
                                5, 256, 256, TilingScheme::Grid,
                                ExtendedCompressionType::DEFLATE);
 
}


// Example geoheif-3: Create a tiled pyramid GeoHEIF with UNCI tiling scheme and DEFLATE compression but with smaller tile size (256x256) 
{
    const char* geotiff_file = "srcdata/ACT2017.tiff";
    const char* output_dir = "benchmark_output";
    std::string output = std::string(output_dir) + "/geoheif_tiled_pyramid_5levels_unci_256_deflate.heif";
    
    encode_tiled_pyramid_geoheif(geotiff_file, output.c_str(),
                                5, 256, 256, TilingScheme::UNCI,
                                ExtendedCompressionType::DEFLATE, 100, 8, 255);
 
}



## Summary

### Key Features:

1. **Protection against redefinition**: All code blocks wrapped with `#ifndef` guards

2. **Comprehensive compression support**:
   - Video codecs: HEVC, AV1, VVC, AVC
   - Image codecs: JPEG, JPEG2000, HTJ2K
   - Lossless: DEFLATE, ZLIB, Brotli, ZSTD, LZW, LERC, PackBits
   - GDAL integration for COG/GeoTIFF compression compatibility

3. **Detailed file structure analysis**:
   - Complete box hierarchy visualization
   - Offset and size information for each box
   - Box descriptions and purpose
   - Summary statistics showing space usage
   - Percentage breakdown of file components

4. **Flexible encoding options**:
   - Non-tiled images
   - Tiled images (grid layout)
   - Pyramid/multi-resolution images
   - All with configurable compression

5. **GDAL integration**:
   - Full GeoTIFF support
   - Metadata preservation
   - Projection information
   - Support for various data types

### Usage Notes:
- Cells can be re-executed safely without redefinition errors
- Each encoding automatically analyzes the output file structure
- Compression algorithms list shows what's supported by libheif and GDAL
- File analyzer provides detailed breakdown of mdat, meta, and all other boxes